# 🏭 FC Mold G5 - Scalable Multi-Strand Analysis Pipeline
## TATA IJmuiden CC23 Continuous Casting

### 🎯 Project Objectives
* **Identify stable casting sequences** across multiple strands
* **Remove sensor artifacts** through intelligent filtering
* **Relate mold level stability** to process parameters
* **Enable scalable analysis** for CC23 strands (23_5 and 23_6)

### 📊 Data Sources
* **dtExpert**: 1Hz frequency - EMBR currents & casting parameters
* **boExpert**: 2Hz frequency - FBG temperature & casting parameters
* **Metadata**: Casting quality records with temporal boundaries

### 🔧 Architecture
This notebook implements a **modular, class-based pipeline** that:
* Supports **multi-strand analysis** with parameterized configuration
* Uses **reusable components** (DataLoader, SequenceAnalyzer, Visualizer)
* Generates **strand-specific outputs** with consistent naming
* Enables **parallel processing** and easy extension to new strands

---
**Last Updated**: 2026-02-09

## How to Use This Notebook

This notebook contains **two execution paths**. Choose one based on your goal:

---

### Path A: Class-Based Pipeline (Recommended for Production)
**Cells 1-18** — Run these in sequence for a complete, reproducible analysis.

| Step | Cell | What it does |
| --- | --- | --- |
| 1 | Cells 2-3 | Load configuration (thresholds, strand paths) |
| 2 | Cells 5-6 | Utility functions and StrandDataLoader |
| 3 | Cells 8-10 | SequenceAnalyzer, disturbance detection |
| 4 | Cells 12-14 | VisualizationFactory, ResultsExporter |
| 5 | Cells 16-17 | StrandAnalysisPipeline — single or multi-strand |

**Output:** `all_results` dict with `df_seq`, `df_raw`, normal/abnormal groups per strand.

---

### Path B: Step-by-Step Exploration (For Understanding / Debugging)
**Cells 30+** — Manual, exploratory workflow for Strand 23-6.  
**Note:** Path B imports from `src/` modules (same code as Path A). Changes in `src/` propagate to both paths.

| Section | Cells | Topic |
| --- | --- | --- |
| Imports | 32-33 | Shared modules from `src/` + visualization libs |
| Data loading | 34-58 | Load files, join, inspect schemas |
| Data quality | 59-82 | Temporal coverage, gap detection, metadata matching |
| Visualization | 83-103 | Distributions, time series, scatter density |
| Sequence analysis | 104-125 | Identify stable sequences, compute statistics |
| Disturbance detection | 126-157 | Classify disturbances, final correlation plots |

---

### Key Variables (After Running)
* `df_23_6` / `df_23_5` — Pandas DataFrames with cleaned sensor + metadata
* `normal_groups_ON_str23_6` — List of index arrays for stable sequences
* `df_seq_normal_ON_str23_6` — Per-sequence statistics (ML sigma, speed, width, etc.)

### Important Context
* **Steel grade classification** uses 19 TATA casting groups from `CastingGroups_ABB_April2026.xlsx` (not the old first-character heuristic)
* **Mold level stability threshold**: sigma < ±1 mm
* **Data frequency**: boExpert=2Hz (aggregated to 1Hz), dtExpert=1Hz
* **Time coverage**: April-May 2025

In [0]:
from dataclasses import dataclass
from typing import List, Dict, Optional
from datetime import datetime

@dataclass
class AnalysisConfig:
    """Global analysis parameters for mold level stability detection"""
    
    # Sequence identification parameters
    window_size: int = 300  # samples (~6 min at 1Hz)
    vc_threshold: float = 0.1  # m/min - casting speed stability threshold
    curr_threshold: float = 50.0  # A - EMBR current stability threshold
    max_gap_seconds: int = 5  # seconds - max gap before new segment
    
    # Data filtering thresholds
    min_casting_speed: float = 0.5  # m/min
    sen_depth_min: float = 0.1  # m
    sen_depth_max: float = 0.26  # m
    
    # Disturbance detection parameters
    excursion_threshold_mm: float = 8.0  # mm deviation from baseline
    excursion_min_duration_s: float = 5.0  # seconds
    slow_drift_min_run_s: float = 60.0  # seconds
    slow_drift_min_amplitude_mm: float = 10.0  # mm
    transient_bump_k_amp: float = 8.0  # multiplier for noise sigma
    transient_bump_min_mm: float = 6.0  # mm absolute minimum
    high_var_ptp_threshold_mm: float = 10.0  # peak-to-peak range
    high_var_band_mm: float = 4.0  # ±mm around baseline
    high_var_fraction_threshold: float = 0.1  # 10% outside band
    
    # Mold level stability threshold
    ml_stability_threshold_mm: float = 2.0  # σ threshold for "stable"
    
    # Sampling for visualization
    viz_sample_fraction: float = 0.25  # 25% sample for large datasets
    
    # Output settings
    timestamp: str = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    def __repr__(self):
        return f"AnalysisConfig(window={self.window_size}, Vc_thr={self.vc_threshold}, ML_thr={self.ml_stability_threshold_mm}mm)"

# Initialize global config
CONFIG = AnalysisConfig()
print(f"✅ Analysis Configuration Loaded: {CONFIG}")

In [0]:
@dataclass
class StrandConfig:
    """Configuration for a specific casting strand"""
    
    strand_id: str  # e.g., "23_5" or "23_6"
    strand_name: str  # e.g., "Strand 23-5"
    data_path: str  # DBFS path to strand data
    
    # EMBR current columns (strand-specific naming)
    embr_current_cols: List[str]
    
    # Casting speed column name
    vc_column: str = "castingSpeed"
    
    # Output directory
    output_base: str = "/dbfs/FileStore/TATAIjmulden_FCMoldG5"
    
    @property
    def output_dir(self) -> str:
        """Generate strand-specific output directory"""
        return f"{self.output_base}/strand_{self.strand_id}"
    
    @property
    def figures_dir(self) -> str:
        return f"{self.output_dir}/figures"
    
    @property
    def reports_dir(self) -> str:
        return f"{self.output_dir}/reports"
    
    @property
    def data_dir(self) -> str:
        return f"{self.output_dir}/processed_data"
    
    def get_output_filename(self, base_name: str, extension: str = "html") -> str:
        """Generate strand-specific filename with timestamp"""
        return f"{base_name}_strand{self.strand_id}_{CONFIG.timestamp}.{extension}"
    
    def __repr__(self):
        return f"StrandConfig({self.strand_name}, path={self.data_path})"

# Define configurations for both strands
STRAND_CONFIGS = {
    "23_6": StrandConfig(
        strand_id="23_6",
        strand_name="Strand 23-6",
        data_path="dbfs:/FileStore/TATA_IJmuiden_CC23/data/strand_6",
        embr_current_cols=[
            "EMBR Current AC Left Master",
            "EMBR Current DC Left Master",
            "EMBR Current DC Left Bottom"
        ]
    ),
    "23_5": StrandConfig(
        strand_id="23_5",
        strand_name="Strand 23-5",
        data_path="dbfs:/FileStore/TATA_IJmuiden_CC23/data/strand_5",
        embr_current_cols=[
            "EMBR Current AC Left Master",
            "EMBR Current DC Left Master",
            "EMBR Current DC Left Bottom"
        ]
    )
}

# Metadata path (shared across strands)
METADATA_PATH = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv"

print("✅ Strand Configurations Loaded:")
for strand_id, config in STRAND_CONFIGS.items():
    print(f"   • {config}")

## 📦 Data Loading & Preprocessing Pipeline

This module provides a **reusable, strand-agnostic data loading pipeline** that:
* Loads boExpert and dtExpert files from DBFS
* Handles timestamp normalization across different formats
* Performs unit conversions (m/s → m/min, m → mm, etc.)
* Joins sensor data with metadata using range-based temporal alignment
* Applies data quality filters
* Returns clean, analysis-ready Pandas DataFrames

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType, LongType, IntegerType, DoubleType, FloatType, NumericType
from typing import List, Tuple
import os

def add_plain_timestamp(df):
    """
    Ensure df has a 'plainTimeStamp' column of TimestampType.
    Handles multiple timestamp formats from boExpert/dtExpert files.
    """
    schema = df.schema
    cols = df.columns

    # Case 1: plainTimeStamp already exists
    if "plainTimeStamp" in cols:
        dt = schema["plainTimeStamp"].dataType
        if isinstance(dt, StringType):
            df = df.withColumn("plainTimeStamp", F.to_timestamp("plainTimeStamp"))
        return df

    # Case 2: dt_timestamp_utc exists (CSV case)
    if "dt_timestamp_utc" in cols:
        dt = schema["dt_timestamp_utc"].dataType
        if isinstance(dt, StringType):
            df = df.withColumn("plainTimeStamp", F.to_timestamp("dt_timestamp_utc"))
        elif isinstance(dt, TimestampType):
            df = df.withColumn("plainTimeStamp", F.col("dt_timestamp_utc"))
        elif isinstance(dt, (LongType, IntegerType, DoubleType, FloatType)):
            col = F.col("dt_timestamp_utc")
            df = df.withColumn(
                "plainTimeStamp",
                F.from_unixtime(
                    F.when(col > 1e12, col / 1000).otherwise(col)
                ).cast("timestamp")
            )
        return df

    # Case 3: TIMESTAMP exists (parquet case)
    if "TIMESTAMP" in cols:
        dt = schema["TIMESTAMP"].dataType
        if isinstance(dt, TimestampType):
            df = df.withColumn("plainTimeStamp", F.col("TIMESTAMP"))
        elif isinstance(dt, (LongType, IntegerType, DoubleType, FloatType)):
            col = F.col("TIMESTAMP")
            df = df.withColumn(
                "plainTimeStamp",
                F.from_unixtime(
                    F.when(col > 1e12, col / 1000).otherwise(col)
                ).cast("timestamp")
            )
        return df

    return df

def get_expert_files(folder_path: str) -> Tuple[List[str], List[str]]:
    """
    Recursively find boExpert and dtExpert files in DBFS folder.
    
    Returns:
        (bo_expert_files, dt_expert_files) - lists of full DBFS paths
    """
    bo_expert_files = []
    dt_expert_files = []

    def walk(path: str):
        for fi in dbutils.fs.ls(path):
            if fi.path.endswith('/'):
                walk(fi.path)
            else:
                name = fi.name
                if 'boExpert' in name:
                    bo_expert_files.append(fi.path)
                elif 'dtExpert' in name:
                    dt_expert_files.append(fi.path)

    walk(folder_path)
    return bo_expert_files, dt_expert_files

def load_expert_files(file_paths: List[str]):
    """
    Load parquet/CSV files and add plainTimeStamp column.
    """
    if not file_paths:
        return None
        
    parquet_files = [f for f in file_paths if f.lower().endswith(".parquet")]
    csv_files = [f for f in file_paths if f.lower().endswith(".csv")]

    if parquet_files:
        df = spark.read.parquet(*parquet_files)
    elif csv_files:
        df = spark.read.option("header", True).option("inferSchema", True).csv(csv_files)
    else:
        return None

    df = add_plain_timestamp(df)
    return df

def convert_units(df):
    """
    Convert units:
      - castingSpeed: m/s → m/min (×60)
      - Mold Level: m → mm (×1000)
      - Argon flows: m³/s → L/min (×60000)
    """
    conversions = {
        "castingSpeed": 60,
        "Mold Level": 1000,
        "Mold Level Sensor Left": 1000,
        "Mold Level Sensor Right": 1000,
        "Argon Flow SEN": 60000,
        "Argon Flow Stopper": 60000
    }

    df_conv = df
    for col, factor in conversions.items():
        if col in df.columns:
            df_conv = df_conv.withColumn(col, F.col(col) * factor)

    return df_conv

print("✅ Utility functions loaded")

In [0]:
import pandas as pd
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql.functions import broadcast

class StrandDataLoader:
    """
    Encapsulates all data loading and preprocessing for a specific strand.
    
    Usage:
        loader = StrandDataLoader(STRAND_CONFIGS["23_6"], CONFIG)
        df_pandas = loader.load_and_process()
    """
    
    def __init__(self, strand_config: StrandConfig, analysis_config: AnalysisConfig):
        self.strand_config = strand_config
        self.analysis_config = analysis_config
        self.logger_prefix = f"[{strand_config.strand_name}]"
        
        # Create output directories using dbutils for DBFS
        for dir_path in [strand_config.figures_dir, strand_config.reports_dir, strand_config.data_dir]:
            try:
                dbutils.fs.mkdirs(dir_path)
            except Exception as e:
                # Directory might already exist, that's okay
                pass
    
    def log(self, message: str):
        """Print with strand-specific prefix"""
        print(f"{self.logger_prefix} {message}")
    
    def load_expert_data(self) -> Tuple[SparkDataFrame, SparkDataFrame]:
        """
        Load boExpert and dtExpert files for this strand.
        
        Returns:
            (df_boExpert, df_dtExpert) - Spark DataFrames
        """
        self.log("📂 Loading expert files...")
        
        bo_files, dt_files = get_expert_files(self.strand_config.data_path)
        self.log(f"   Found {len(bo_files)} boExpert files, {len(dt_files)} dtExpert files")
        
        df_boExpert = load_expert_files(bo_files)
        df_dtExpert = load_expert_files(dt_files)
        
        self.log(f"   boExpert: {df_boExpert.count():,} rows")
        self.log(f"   dtExpert: {df_dtExpert.count():,} rows")
        
        return df_boExpert, df_dtExpert
    
    def aggregate_boexpert(self, df_boExpert: SparkDataFrame) -> SparkDataFrame:
        """
        Aggregate boExpert data by timestamp (handles 2Hz → 1Hz downsampling).
        """
        self.log("🔄 Aggregating boExpert data by timestamp...")
        
        key_col = "plainTimeStamp"
        
        # Identify numeric vs non-numeric columns
        numeric_cols = [
            f.name for f in df_boExpert.schema.fields
            if isinstance(f.dataType, NumericType) and f.name != key_col
        ]
        non_numeric_cols = [
            f.name for f in df_boExpert.schema.fields
            if not isinstance(f.dataType, NumericType) and f.name != key_col
        ]
        
        # Build aggregation expressions
        agg_exprs = [F.avg(c).alias(c) for c in numeric_cols]
        agg_exprs += [F.first(c).alias(c) for c in non_numeric_cols]
        
        df_agg = df_boExpert.groupBy(key_col).agg(*agg_exprs)
        
        self.log(f"   Aggregated to {df_agg.count():,} rows")
        return df_agg
    
    def join_expert_data(self, df_boExpert_agg: SparkDataFrame, df_dtExpert: SparkDataFrame) -> SparkDataFrame:
        """
        Inner join boExpert and dtExpert on plainTimeStamp.
        """
        self.log("🔗 Joining boExpert and dtExpert...")
        
        df_joined = df_boExpert_agg.alias("bo").join(
            df_dtExpert.alias("dt"),
            on="plainTimeStamp",
            how="inner"
        )
        
        df_joined = df_joined.cache()
        row_count = df_joined.count()
        
        self.log(f"   Joined: {row_count:,} rows")
        return df_joined
    
    def join_metadata(self, df_joined: SparkDataFrame) -> SparkDataFrame:
        """
        Range join with metadata to add Quality casting labels.
        FIXED: Filters by Strand ID to prevent duplicates from multi-strand metadata.
        """
        self.log("📊 Joining with metadata...")
        
        # Load metadata with semicolon delimiter
        df_metadata = spark.read.csv(
            METADATA_PATH,
            header=True,
            inferSchema=True,
            sep=";"
        )
        
        # Cast plainTimeStamp to PlainTimeStamp (capital P) to match original
        df_joined = df_joined.withColumn(
            "PlainTimeStamp",
            F.col("plainTimeStamp").cast("timestamp")
        )
        
        # Cast metadata timestamps and FILTER BY STRAND ID
        strand_number = int(self.strand_config.strand_id.split("_")[1])
        
        df_metadata = df_metadata \
            .withColumn("Datetime start first heat", F.col("Datetime start first heat").cast("timestamp")) \
            .withColumn("Datetime start last heat", F.col("Datetime start last heat").cast("timestamp")) \
            .filter(F.col("Strand ID") == strand_number)  # CRITICAL: Prevent duplicates!
        
        self.log(f"   Metadata filtered to Strand ID = {strand_number}: {df_metadata.count()} casting periods")
        
        # Range join condition
        join_condition = (
            (df_joined["PlainTimeStamp"] >= df_metadata["Datetime start first heat"]) &
            (df_joined["PlainTimeStamp"] <= df_metadata["Datetime start last heat"])
        )
        
        # Left join with metadata
        df_with_metadata = df_joined.join(
            df_metadata.select(
                "Datetime start first heat",
                "Datetime start last heat",
                "Quality casting"
            ),
            on=join_condition,
            how="left"
        )
        
        row_count_after = df_with_metadata.count()
        matched_count = df_with_metadata.filter(F.col("Quality casting").isNotNull()).count()
        match_pct = 100 * matched_count / row_count_after if row_count_after > 0 else 0
        
        self.log(f"   After join: {row_count_after:,} rows (no duplicates!)")
        self.log(f"   Matched {matched_count:,}/{row_count_after:,} rows ({match_pct:.1f}%) to casting periods")
        
        return df_with_metadata
    
    def apply_filters(self, df: SparkDataFrame) -> SparkDataFrame:
        """
        Apply data quality filters.
        """
        self.log("⚙️ Applying data quality filters...")
        
        df_filtered = (
            df
            .filter(F.col("castingSpeed") >= self.analysis_config.min_casting_speed)
            .filter(F.col("SENImmersionDepth").between(
                self.analysis_config.sen_depth_min,
                self.analysis_config.sen_depth_max
            ))
        )
        
        before_count = df.count()
        after_count = df_filtered.count()
        filtered_pct = 100 * (before_count - after_count) / before_count if before_count > 0 else 0
        
        self.log(f"   Filtered: {before_count:,} → {after_count:,} rows ({filtered_pct:.1f}% removed)")
        
        return df_filtered
    
    def to_pandas(self, df_spark: SparkDataFrame) -> pd.DataFrame:
        """
        Convert to Pandas and sort by timestamp.
        Uses PlainTimeStamp (capital P) after metadata join.
        """
        self.log("🐼 Converting to Pandas...")
        
        # Determine which timestamp column exists
        time_col = "PlainTimeStamp" if "PlainTimeStamp" in df_spark.columns else "plainTimeStamp"
        
        # Select relevant columns
        cols_for_analysis = [
            time_col,
            "castingSpeed",
            "moldWidth",
            "SENImmersionDepth",
            "Mold Level",
            "Mold Level Sensor Left",
            "Mold Level Sensor Right",
            "Argon Flow SEN",
            "Argon Flow Stopper",
            "Quality casting"
        ] + self.strand_config.embr_current_cols
        
        # Filter to existing columns
        cols_available = [c for c in cols_for_analysis if c in df_spark.columns]
        
        df_pandas = (
            df_spark
            .select(*cols_available)
            .dropna(subset=["castingSpeed", "moldWidth", "SENImmersionDepth"])
            .toPandas()
            .sort_values(time_col)
            .reset_index(drop=True)
        )
        
        # Rename to lowercase for consistency in analysis
        if time_col == "PlainTimeStamp":
            df_pandas = df_pandas.rename(columns={"PlainTimeStamp": "plainTimeStamp"})
        
        self.log(f"   Pandas DataFrame: {df_pandas.shape[0]:,} rows, {df_pandas.shape[1]} columns")
        
        # Report Quality casting distribution
        null_count = df_pandas['Quality casting'].isna().sum()
        non_null_count = len(df_pandas) - null_count
        null_pct = 100 * null_count / len(df_pandas)
        non_null_pct = 100 * non_null_count / len(df_pandas)
        
        self.log(f"   Quality casting: {non_null_count:,} non-NULL ({non_null_pct:.1f}%), {null_count:,} NULL ({null_pct:.1f}%)")
        
        return df_pandas
    
    def load_and_process(self) -> pd.DataFrame:
        """
        Main pipeline: load, join, filter, and return analysis-ready DataFrame.
        Order: expert join → metadata join → unit conversion → filtering
        """
        self.log("🚀 Starting data loading pipeline...")
        
        # Load expert files
        df_boExpert, df_dtExpert = self.load_expert_data()
        
        # Aggregate boExpert
        df_boExpert_agg = self.aggregate_boexpert(df_boExpert)
        
        # Join expert data
        df_joined = self.join_expert_data(df_boExpert_agg, df_dtExpert)
        
        # Join metadata (with strand filtering to prevent duplicates)
        df_with_metadata = self.join_metadata(df_joined)
        
        # Convert units
        self.log("🔧 Converting units...")
        df_with_metadata = convert_units(df_with_metadata)
        
        # Apply filters
        df_filtered = self.apply_filters(df_with_metadata)
        
        # Convert to Pandas
        df_pandas = self.to_pandas(df_filtered)
        
        self.log("✅ Data loading pipeline complete!")
        
        return df_pandas

print("✅ StrandDataLoader class defined")

## 🔍 Sequence Analysis & Disturbance Detection

This module provides **intelligent sequence identification and disturbance classification** for mold level stability analysis:

### 🎯 Core Capabilities
* **Sliding window analysis** (300-sample/6-minute windows)
* **Temporal segmentation** (handles data gaps >5 seconds)
* **Multi-criteria stability detection** (casting speed + EMBR currents)
* **Comprehensive disturbance detection**:
  * Excursion events (large deviations)
  * Slow drift (gradual trends)
  * Transient bumps (temporary spikes)
  * High variability (excessive oscillations)
* **Statistical aggregation** per sequence
* **Disturbance classification** with multi-label support

In [0]:
import numpy as np
import pandas as pd
from scipy.ndimage import median_filter
from typing import List, Tuple

def custom_rounding(series, round_up=True):
    """
    Round values with custom logic to maintain consistency.
    """
    rounded_values = []
    for i in range(len(series)):
        if i > 0 and abs(series[i] - series[i-1]) <= 0.01:
            rounded_value = rounded_values[-1]
        else:
            if round_up:
                rounded_value = round(series[i] * 100 + 0.5) / 100
            else:
                rounded_value = round(series[i] * 100 - 0.5) / 100
        rounded_values.append(rounded_value)
    return rounded_values

def segment_by_time_gaps(df, tcol="plainTimeStamp", max_gap_seconds=5):
    """
    Split time-series into continuous segments based on time gaps.
    
    Returns:
        DataFrame with added 'segment_id' column
    """
    df = df.copy()
    df[tcol] = pd.to_datetime(df[tcol])
    df = df.sort_values(tcol)
    
    time_diffs = df[tcol].diff().dt.total_seconds().fillna(0)
    is_new_segment = (time_diffs > max_gap_seconds).astype(int)
    segment_ids = is_new_segment.cumsum()
    
    df["segment_id"] = segment_ids
    return df

def identify_sequences(df, Vc_column, window_size, Vc_threshold, Curr_columns=None, Curr_threshold=None):
    """
    Sliding window classifier for NORMAL/ABNORMAL sequences.
    
    KEY BEHAVIOR (matches original):
    - NORMAL window: skip forward by window_size (non-overlapping)
    - ABNORMAL window: skip forward by 1 (overlapping search)
    
    This creates fewer sequences by merging stable periods.
    
    Returns:
        (normal_groups, abnormal_groups) - lists of index lists
    """
    normal_groups = []
    abnormal_groups = []
    
    i = 0
    n = len(df)
    
    while i <= n - window_size:
        window_df = df.iloc[i:i + window_size]
        window_idx = window_df.index
        
        # Check casting speed stability
        Vc_window = window_df[Vc_column]
        cond_vc = np.all(np.abs(Vc_window - Vc_window.iloc[0]) <= Vc_threshold)
        
        # Check current stability (if specified)
        cond_curr = True
        if cond_vc and Curr_columns is not None and Curr_threshold is not None:
            for col in Curr_columns:
                if col not in window_df.columns:
                    cond_curr = False
                    break
                series = window_df[col]
                if np.any(np.abs(series.diff().dropna()) >= Curr_threshold):
                    cond_curr = False
                    break
        
        is_normal = cond_vc and cond_curr
        
        if is_normal:
            normal_groups.append(window_idx.tolist())
            # Skip past this whole stable window (non-overlapping)
            i += window_size
        else:
            abnormal_groups.append(window_idx.tolist())
            # Move forward by ONE sample (overlapping search)
            i += 1
    
    return normal_groups, abnormal_groups

def identify_sequences_segmented(df, Vc_column, window_size, Vc_threshold, 
                                 Curr_columns=None, Curr_threshold=None,
                                 tcol="plainTimeStamp", max_gap_seconds=5, 
                                 min_segment_len=None):
    """
    Apply identify_sequences within continuous time segments.
    Prevents windows from crossing large time gaps.
    
    Returns:
        (normal_groups, abnormal_groups) - global index lists
    """
    if min_segment_len is None:
        min_segment_len = window_size
    
    # Segment by time gaps
    df_seg = segment_by_time_gaps(df, tcol=tcol, max_gap_seconds=max_gap_seconds)
    
    normal_groups_all = []
    abnormal_groups_all = []
    
    for seg_id, seg_df in df_seg.groupby("segment_id"):
        if len(seg_df) < min_segment_len:
            continue
        
        # Run sequence identification on segment
        n_g, a_g = identify_sequences(
            seg_df,
            Vc_column=Vc_column,
            window_size=window_size,
            Vc_threshold=Vc_threshold,
            Curr_columns=Curr_columns,
            Curr_threshold=Curr_threshold
        )
        
        normal_groups_all.extend(n_g)
        abnormal_groups_all.extend(a_g)
    
    return normal_groups_all, abnormal_groups_all

print("✅ Core sequence functions loaded")

In [0]:
def detect_excursion_event_robust(signal, threshold_mm=8.0, min_duration_s=5.0, sampling_hz=1.0):
    """
    Detect excursion events: large deviations >threshold_mm for >min_duration_s.
    """
    baseline = np.median(signal)
    deviation = np.abs(signal - baseline)
    
    min_samples = int(min_duration_s * sampling_hz)
    
    above_threshold = (deviation > threshold_mm).astype(int)
    
    # Find runs of consecutive True values
    diff = np.diff(np.concatenate(([0], above_threshold, [0])))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    
    for start, end in zip(starts, ends):
        if (end - start) >= min_samples:
            return True
    
    return False

def detect_slow_drift_event(signal, min_run_s=60.0, min_amplitude_mm=10.0, sampling_hz=1.0):
    """
    Detect slow drift: sustained monotonic trends over min_run_s with >min_amplitude_mm.
    """
    min_samples = int(min_run_s * sampling_hz)
    
    if len(signal) < min_samples:
        return False
    
    # Check for monotonic increasing
    diffs = np.diff(signal)
    increasing_runs = []
    current_run = 0
    
    for d in diffs:
        if d > 0:
            current_run += 1
        else:
            if current_run >= min_samples:
                increasing_runs.append(current_run)
            current_run = 0
    
    if current_run >= min_samples:
        increasing_runs.append(current_run)
    
    # Check for monotonic decreasing
    decreasing_runs = []
    current_run = 0
    
    for d in diffs:
        if d < 0:
            current_run += 1
        else:
            if current_run >= min_samples:
                decreasing_runs.append(current_run)
            current_run = 0
    
    if current_run >= min_samples:
        decreasing_runs.append(current_run)
    
    # Check amplitude
    for run_len in increasing_runs + decreasing_runs:
        if run_len >= min_samples:
            # Check if amplitude exceeds threshold
            amplitude = np.ptp(signal)
            if amplitude >= min_amplitude_mm:
                return True
    
    return False

def detect_transient_bump_dynamic(signal, k_amp=8.0, min_amp_mm=6.0, 
                                  min_duration_s=5.0, max_duration_s=180.0,
                                  return_band_sigma=2.5, sampling_hz=1.0):
    """
    Detect transient bumps using dynamic threshold based on noise estimation.
    """
    # Baseline using rolling median
    window_size = int(20 * sampling_hz)
    baseline = median_filter(signal, size=window_size, mode='nearest')
    
    # Noise estimation using MAD
    residuals = signal - baseline
    mad = np.median(np.abs(residuals - np.median(residuals)))
    sigma = 1.4826 * mad
    
    # Dynamic threshold
    threshold = max(k_amp * sigma, min_amp_mm)
    
    # Detect excursions
    deviation = np.abs(signal - baseline)
    above_threshold = (deviation > threshold).astype(int)
    
    # Find runs
    diff = np.diff(np.concatenate(([0], above_threshold, [0])))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    
    min_samples = int(min_duration_s * sampling_hz)
    max_samples = int(max_duration_s * sampling_hz)
    
    for start, end in zip(starts, ends):
        duration = end - start
        if min_samples <= duration <= max_samples:
            # Check if signal returns to baseline
            if end < len(signal):
                post_event = signal[end:min(end + int(30 * sampling_hz), len(signal))]
                if len(post_event) > 0:
                    post_baseline = np.median(post_event)
                    if np.abs(post_baseline - np.median(baseline)) < return_band_sigma * sigma:
                        return True
    
    return False

def detect_high_variability_event(signal, ptp_threshold_mm=10.0, band_mm=4.0, fraction_threshold=0.1):
    """
    Detect high variability: excessive oscillations.
    """
    # Peak-to-peak range
    ptp = np.ptp(signal)
    if ptp > ptp_threshold_mm:
        return True
    
    # Fraction outside band
    baseline = np.median(signal)
    outside_band = np.abs(signal - baseline) > band_mm
    fraction_outside = np.mean(outside_band)
    
    if fraction_outside > fraction_threshold:
        return True
    
    return False

print("✅ Disturbance detection functions loaded")

In [0]:
class SequenceAnalyzer:
    """
    Encapsulates sequence identification, disturbance detection, and statistical analysis.
    
    Usage:
        analyzer = SequenceAnalyzer(strand_config, analysis_config)
        df_seq, normal_groups, abnormal_groups = analyzer.analyze(df_pandas)
    """
    
    def __init__(self, strand_config: StrandConfig, analysis_config: AnalysisConfig):
        self.strand_config = strand_config
        self.config = analysis_config
        self.logger_prefix = f"[{strand_config.strand_name}]"
    
    def log(self, message: str):
        print(f"{self.logger_prefix} {message}")
    
    def filter_fc_mold_active(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Filter to periods where FC Mold is active (all EMBR currents non-zero).
        """
        self.log("⚡ Filtering for FC Mold active periods...")
        
        mask = pd.Series(True, index=df.index)
        for col in self.strand_config.embr_current_cols:
            if col in df.columns:
                mask &= (df[col] != 0)
        
        df_filtered = df[mask].reset_index(drop=True)
        
        self.log(f"   FC Mold active: {len(df_filtered):,}/{len(df):,} rows ({100*len(df_filtered)/len(df):.1f}%)")
        
        return df_filtered
    
    def identify_sequences(self, df: pd.DataFrame) -> Tuple[List[List[int]], List[List[int]]]:
        """
        Identify normal and abnormal sequences using sliding window analysis.
        """
        self.log("🔍 Identifying sequences...")
        
        # Apply custom rounding to casting speed
        df = df.copy()
        df[self.strand_config.vc_column] = custom_rounding(df[self.strand_config.vc_column], round_up=True)
        
        # Run segmented sequence identification
        normal_groups, abnormal_groups = identify_sequences_segmented(
            df,
            Vc_column=self.strand_config.vc_column,
            window_size=self.config.window_size,
            Vc_threshold=self.config.vc_threshold,
            Curr_columns=self.strand_config.embr_current_cols,
            Curr_threshold=self.config.curr_threshold,
            tcol="plainTimeStamp",
            max_gap_seconds=self.config.max_gap_seconds,
            min_segment_len=self.config.window_size
        )
        
        self.log(f"   Found {len(normal_groups)} normal sequences, {len(abnormal_groups)} abnormal sequences")
        
        return normal_groups, abnormal_groups
    
    def generate_statistics_with_disturbance(self, df: pd.DataFrame, sequences: List[List[int]]) -> pd.DataFrame:
        """
        Generate comprehensive statistics for each sequence including disturbance detection.
        """
        self.log("📊 Generating sequence statistics with disturbance detection...")
        
        results = []
        
        for seq_idx, group in enumerate(sequences):
            if max(group) >= len(df):
                continue
            
            temp_df = df.iloc[group].copy()
            
            # Basic metadata
            seq_name = f"seq_{seq_idx:04d}"
            seq_start = temp_df["plainTimeStamp"].min()
            seq_end = temp_df["plainTimeStamp"].max()
            seq_duration_min = (seq_end - seq_start).total_seconds() / 60
            
            # Quality casting (take most common value, or first non-null)
            quality_casting = None
            if "Quality casting" in temp_df.columns:
                quality_values = temp_df["Quality casting"].dropna()
                if len(quality_values) > 0:
                    quality_casting = quality_values.mode()[0] if len(quality_values.mode()) > 0 else quality_values.iloc[0]
            
            # Mold level signal for disturbance detection
            ml_signal = temp_df["Mold Level"].values
            
            # Disturbance detection
            has_excursion = detect_excursion_event_robust(
                ml_signal,
                threshold_mm=self.config.excursion_threshold_mm,
                min_duration_s=self.config.excursion_min_duration_s
            )
            
            has_slow_drift = detect_slow_drift_event(
                ml_signal,
                min_run_s=self.config.slow_drift_min_run_s,
                min_amplitude_mm=self.config.slow_drift_min_amplitude_mm
            )
            
            has_transient_bump = detect_transient_bump_dynamic(
                ml_signal,
                k_amp=self.config.transient_bump_k_amp,
                min_amp_mm=self.config.transient_bump_min_mm
            )
            
            has_high_variability = detect_high_variability_event(
                ml_signal,
                ptp_threshold_mm=self.config.high_var_ptp_threshold_mm,
                band_mm=self.config.high_var_band_mm,
                fraction_threshold=self.config.high_var_fraction_threshold
            )
            
            has_disturbance = any([has_excursion, has_slow_drift, has_transient_bump, has_high_variability])
            
            # Statistical aggregations
            stats = {
                "Seq_Name": seq_name,
                "Seq_time_Start": seq_start,
                "Seq_time_End": seq_end,
                "Seq_duration_min": seq_duration_min,
                "Seq_num_samples": len(temp_df),
                
                # Casting parameters
                "CASTING_SPEED_avg [m/min]": temp_df["castingSpeed"].mean(),
                "CASTING_SPEED_std [m/min]": temp_df["castingSpeed"].std(),
                "MOLD_WIDTH_avg [m]": temp_df["moldWidth"].mean(),
                "MOLD_WIDTH_std [m]": temp_df["moldWidth"].std(),
                "SEN_avg [mm]": temp_df["SENImmersionDepth"].mean() * 1000,
                "SEN_std [mm]": temp_df["SENImmersionDepth"].std() * 1000,
                
                # Mold level
                "MOLD_LEVEL_avg [mm]": temp_df["Mold Level"].mean(),
                "MOLD_LEVEL_std [mm]": temp_df["Mold Level"].std(),
                "MOLD_LEVEL_min [mm]": temp_df["Mold Level"].min(),
                "MOLD_LEVEL_max [mm]": temp_df["Mold Level"].max(),
                "ptp_mm": temp_df["Mold Level"].max() - temp_df["Mold Level"].min(),
                
                # Argon flow
                "ArFlow_avg [NL/min]": (temp_df["Argon Flow SEN"].mean() + temp_df["Argon Flow Stopper"].mean()) if "Argon Flow SEN" in temp_df.columns else np.nan,
                
                # Quality casting (most common value in sequence)
                "Quality casting": quality_casting,
                
                # Disturbance flags
                "has_excursion_event": has_excursion,
                "has_slow_drift": has_slow_drift,
                "has_transient_bump": has_transient_bump,
                "has_high_variability": has_high_variability,
                "has_disturbance": has_disturbance,
            }
            
            # Add EMBR current statistics
            for col in self.strand_config.embr_current_cols:
                if col in temp_df.columns:
                    col_clean = col.replace(" ", "_").replace("EMBR_Current_", "")
                    stats[f"{col_clean}_avg [A]"] = temp_df[col].mean()
                    stats[f"{col_clean}_std [A]"] = temp_df[col].std()
            
            results.append(stats)
        
        df_seq = pd.DataFrame(results)
        
        self.log(f"   Generated statistics for {len(df_seq)} sequences")
        
        return df_seq
    
    def classify_disturbance(self, df_seq: pd.DataFrame) -> pd.DataFrame:
        """
        Classify disturbance type based on flags.
        """
        self.log("🏷️ Classifying disturbance types...")
        
        def classify_row(row):
            flags = []
            if row.get("has_excursion_event", False):
                flags.append("Excursion")
            if row.get("has_high_variability", False):
                flags.append("High variability")
            if row.get("has_transient_bump", False):
                flags.append("Transient bump")
            if row.get("has_slow_drift", False):
                flags.append("Slow drift")
            return "Normal" if not flags else " + ".join(flags)
        
        df_seq["disturbance_type"] = df_seq.apply(classify_row, axis=1)
        
        # Count distribution
        dist_counts = df_seq["disturbance_type"].value_counts()
        self.log("   Disturbance distribution:")
        for dtype, count in dist_counts.items():
            self.log(f"      {dtype}: {count} sequences")
        
        return df_seq
    
    def filter_normal_process(self, df_seq: pd.DataFrame) -> pd.DataFrame:
        """
        Filter to sequences with normal process (disturbance_type == 'Normal').
        """
        df_normal = df_seq[df_seq["disturbance_type"] == "Normal"].copy()
        self.log(f"✅ Normal process sequences: {len(df_normal)}/{len(df_seq)}")
        return df_normal
    
    def analyze(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, List[List[int]], List[List[int]]]:
        """
        Main analysis pipeline.
        
        Returns:
            (df_fc_mold, df_seq, normal_groups, abnormal_groups)
        """
        self.log("🚀 Starting sequence analysis...")
        
        # Filter for FC Mold active
        df_fc_mold = self.filter_fc_mold_active(df)
        
        # Identify sequences
        normal_groups, abnormal_groups = self.identify_sequences(df_fc_mold)
        
        # Generate statistics
        df_seq = self.generate_statistics_with_disturbance(df_fc_mold, normal_groups)
        
        # Classify disturbances
        df_seq = self.classify_disturbance(df_seq)
        
        self.log("✅ Sequence analysis complete!")
        
        return df_fc_mold, df_seq, normal_groups, abnormal_groups

print("✅ SequenceAnalyzer class defined")

## 🎨 Visualization Factory

This module provides **strand-specific visualization generation** with:
* Automatic filename generation with strand ID and timestamp
* Consistent styling and color schemes
* Interactive Plotly plots with hover data
* Matplotlib plots for publication-quality figures
* Automatic saving to strand-specific directories

In [0]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import numpy as np

class VisualizationFactory:
    """
    Generates strand-specific visualizations with automatic file naming.
    
    Usage:
        viz = VisualizationFactory(strand_config, analysis_config)
        viz.plot_disturbance_examples(df_fc_mold, df_seq, normal_groups)
    """
    
    def __init__(self, strand_config: StrandConfig, analysis_config: AnalysisConfig):
        self.strand_config = strand_config
        self.config = analysis_config
        self.logger_prefix = f"[{strand_config.strand_name}]"
        
        # Custom colormap for mold level sigma
        self.ml_cmap = LinearSegmentedColormap.from_list(
            "ml_sigma_cmap",
            [
                (0.0, "#440053"),   # dark purple
                (0.3, "#404388"),
                (0.6, "#2a788e"),
                (0.8, "#21a784"),
                (0.95, "#78d151"),
                (1.0, "#fde624"),   # yellow
            ],
            N=256,
        )
    
    def log(self, message: str):
        print(f"{self.logger_prefix} {message}")
    
    def save_figure(self, fig, base_name: str, format: str = "html"):
        """
        Save figure with strand-specific filename.
        """
        filename = self.strand_config.get_output_filename(base_name, format)
        filepath = f"{self.strand_config.figures_dir}/{filename}"
        
        # Convert DBFS path to local path
        local_filepath = filepath.replace("/dbfs", "")
        
        if format == "html":
            fig.write_html(local_filepath)
        elif format == "png":
            fig.savefig(local_filepath, dpi=300, bbox_inches='tight')
        
        self.log(f"💾 Saved: {filename}")
        return filepath
    
    def plot_disturbance_examples(self, df_fc_mold: pd.DataFrame, df_seq: pd.DataFrame, 
                                   sequences: List[List[int]], 
                                   example_indices: dict = None):
        """
        Plot representative sequences for each disturbance type.
        """
        self.log("📊 Creating disturbance type examples plot...")
        
        # Default examples: first occurrence of each type
        if example_indices is None:
            example_indices = df_seq.groupby("disturbance_type").head(1).reset_index()["index"].to_dict()
        
        types = list(example_indices.keys())
        nrows = len(types)
        
        fig = make_subplots(
            rows=nrows,
            cols=1,
            shared_xaxes=False,
            vertical_spacing=0.08,
            subplot_titles=[
                f"Disturbance type: {t}  |  Sequence index: {example_indices[t]}"
                for t in types
            ],
        )
        
        for r, dtype in enumerate(types, start=1):
            seq_idx = example_indices[dtype]
            
            # Get raw sequence data
            df_seq_raw = df_fc_mold.iloc[sequences[seq_idx]].copy()
            
            mean_value = df_seq_raw["Mold Level"].mean()
            min_value = df_seq_raw["Mold Level"].min()
            max_value = df_seq_raw["Mold Level"].max()
            
            # Mold Level signal
            fig.add_trace(
                go.Scatter(
                    x=df_seq_raw["plainTimeStamp"],
                    y=df_seq_raw["Mold Level"],
                    mode="lines",
                    name=dtype,
                    hovertemplate=(
                        "Time=%{x}<br>"
                        "Mold Level=%{y:.2f} mm<br>"
                        f"Type={dtype}"
                        "<extra></extra>"
                    ),
                ),
                row=r,
                col=1,
            )
            
            # Reference lines
            fig.add_hline(y=mean_value, line_dash="dash", line_color="green", row=r, col=1)
            fig.add_hline(y=min_value, line_dash="dot", line_color="blue", row=r, col=1)
            fig.add_hline(y=max_value, line_dash="dot", line_color="red", row=r, col=1)
            
            fig.update_yaxes(title_text="Mold Level [mm]", row=r, col=1)
        
        fig.update_xaxes(title_text="Time", row=nrows, col=1)
        
        fig.update_layout(
            height=320 * nrows,
            width=1200,
            title=f"{self.strand_config.strand_name} - Representative sequences per disturbance type",
            showlegend=False,
        )
        
        self.save_figure(fig, "disturbance_type_examples", "html")
        fig.show()
    
    def plot_correlation_scatter(self, df_seq: pd.DataFrame, filter_normal_only: bool = False):
        """
        Plot Mold Width vs Casting Speed colored by disturbance type.
        """
        self.log("📊 Creating correlation scatter plot...")
        
        dfp = df_seq.copy()
        if filter_normal_only:
            dfp = dfp[dfp["disturbance_type"] == "Normal"]
        
        # Data vectors
        x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
        y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
        ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()
        dtype = dfp["disturbance_type"].astype(str).to_numpy()
        
        # Filter finite
        mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
        x, y, ml_std, dtype = x[mask], y[mask], ml_std[mask], dtype[mask]
        
        above_thr = ml_std > self.config.ml_stability_threshold_mm
        
        # Stable order for legend
        order = [
            "Normal",
            "High variability",
            "Transient bump",
            "Slow drift",
            "Excursion",
            "Excursion + High variability",
            "Excursion + Transient bump + High variability",
        ]
        present = [c for c in order if c in set(dtype)]
        present += [c for c in np.unique(dtype) if c not in present]
        
        # Color mapping
        colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(present)}
        point_colors = np.array([color_map[c] for c in dtype])
        
        # Plot
        fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)
        
        # Base scatter
        ax.scatter(x, y, c=point_colors, s=40, edgecolor="none", alpha=0.8)
        
        # Highlight σ > threshold
        ax.scatter(
            x[above_thr],
            y[above_thr],
            c=point_colors[above_thr],
            s=70,
            edgecolor="black",
            linewidth=0.9,
            alpha=0.95,
        )
        
        # Legend
        handles = [
            plt.Line2D(
                [0], [0],
                marker="o",
                color="none",
                markerfacecolor=color_map[cat],
                markersize=8,
                label=cat,
            )
            for cat in present
        ]
        ax.legend(handles=handles, title="Disturbance type", loc="best", frameon=True)
        
        ax.set_xlabel("Mold Width avg [m]")
        ax.set_ylabel("Casting Speed avg [m/min]")
        title_suffix = " (Normal only)" if filter_normal_only else ""
        ax.set_title(
            f"{self.strand_config.strand_name} - Mold Width vs Casting Speed{title_suffix}\n"
            f"(black outline: MOLD_LEVEL_std > {self.config.ml_stability_threshold_mm} mm)"
        )
        
        base_name = "correlation_scatter_normal" if filter_normal_only else "correlation_scatter_all"
        self.save_figure(fig, base_name, "png")
        plt.show()
    
    def plot_correlation_heatmap(self, df_seq: pd.DataFrame):
        """
        Plot Normal sequences colored by Mold Level σ.
        """
        self.log("📊 Creating correlation heatmap (Normal only)...")
        
        dfp = df_seq[df_seq["disturbance_type"] == "Normal"].copy()
        
        x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
        y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
        ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()
        
        mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
        x, y, ml_std = x[mask], y[mask], ml_std[mask]
        
        above_thr = ml_std > self.config.ml_stability_threshold_mm
        ml_std_max = np.nanmax(ml_std)
        
        # Threshold-aware normalization
        norm = TwoSlopeNorm(
            vmin=0, vcenter=self.config.ml_stability_threshold_mm, vmax=ml_std_max
        )
        
        fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)
        
        # Base scatter
        sc = ax.scatter(
            x, y, c=ml_std, cmap=self.ml_cmap, norm=norm, s=40, edgecolor="none", alpha=0.85
        )
        
        # Highlight σ > threshold
        ax.scatter(
            x[above_thr],
            y[above_thr],
            c=ml_std[above_thr],
            cmap=self.ml_cmap,
            norm=norm,
            s=70,
            edgecolor="black",
            linewidth=0.9,
            alpha=0.95,
        )
        
        # Colorbar
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label("MOLD_LEVEL_std [mm]")
        cbar.ax.axhline(self.config.ml_stability_threshold_mm, color="black", linewidth=1)
        cbar.ax.text(
            1.05,
            self.config.ml_stability_threshold_mm,
            f"σ = {self.config.ml_stability_threshold_mm} mm",
            transform=cbar.ax.get_yaxis_transform(),
            va="center",
        )
        
        ax.set_xlabel("Mold Width avg [m]")
        ax.set_ylabel("Casting Speed avg [m/min]")
        ax.set_title(
            f"{self.strand_config.strand_name} - Normal sequences only\n"
            f"Colored by Mold Level σ (black outline: σ > {self.config.ml_stability_threshold_mm} mm)"
        )
        
        self.save_figure(fig, "correlation_heatmap_normal", "png")
        plt.show()
    
    def plot_parameter_correlations(self, df_seq: pd.DataFrame):
        """
        Plot 2x2 subplot grid of parameter correlations with Mold Level σ.
        Layout: Vc, Mold Width, Argon Flow, Quality Casting
        """
        self.log("📊 Creating parameter correlation plots...")
        
        df = df_seq[df_seq["disturbance_type"] == "Normal"].copy()
        
        fig = make_subplots(
            rows=2,
            cols=2,
            shared_xaxes=False,
            shared_yaxes=False,
            vertical_spacing=0.12,
            horizontal_spacing=0.10,
            subplot_titles=[
                "Vc vs Mold Level σ (Normal only)",
                "Mold Width vs Mold Level σ (Normal only)",
                "Argon Flow vs Mold Level σ (Normal only)",
                "Quality Casting vs Mold Level σ (Normal only)",
            ],
        )
        
        # Panel definitions: (row, col, x_column, x_label, is_categorical)
        panels = [
            (1, 1, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]", False),
            (1, 2, "MOLD_WIDTH_avg [m]", "Mold Width Avg [m]", False),
            (2, 1, "ArFlow_avg [NL/min]", "Ar Flow Avg [NL/min]", False),
            (2, 2, "Quality_casting", "Quality Casting", True),  # Categorical
        ]
        
        # Add Quality_casting column if not present (from original data)
        if "Quality_casting" not in df.columns and "Quality casting" in df.columns:
            df["Quality_casting"] = df["Quality casting"]
        
        for r, c, xcol, xlabel, is_categorical in panels:
            if xcol not in df.columns:
                # Skip if column doesn't exist
                continue
            
            if is_categorical:
                # For Quality casting - use as-is (categorical)
                x_values = df[xcol].astype(str)
                x_plot = x_values
            else:
                # For numeric - round for grouping
                x_round_name = f"{xcol}__rounded"
                df[x_round_name] = df[xcol].round(1)
                x_plot = df[x_round_name]
            
            # Box trace
            box = go.Box(
                x=x_plot,
                y=df["MOLD_LEVEL_std [mm]"],
                boxmean=False,
                marker=dict(color="#bdb76b"),
                line=dict(color="#bdb76b"),
                opacity=0.45,
                showlegend=False,
            )
            
            # Scatter trace
            scatter = go.Scatter(
                x=x_plot,
                y=df["MOLD_LEVEL_std [mm]"],
                mode="markers",
                marker=dict(color="#bdb76b", size=7, opacity=0.8),
                showlegend=False,
            )
            
            fig.add_trace(box, row=r, col=c)
            fig.add_trace(scatter, row=r, col=c)
            
            # Mean line (only for numeric)
            if not is_categorical:
                mean_df = (
                    df.groupby(x_round_name if not is_categorical else xcol, as_index=False)["MOLD_LEVEL_std [mm]"]
                    .mean()
                    .sort_values(x_round_name if not is_categorical else xcol)
                )
                mean_line = go.Scatter(
                    x=mean_df[x_round_name if not is_categorical else xcol],
                    y=mean_df["MOLD_LEVEL_std [mm]"],
                    mode="lines+markers",
                    line=dict(color="black", width=2),
                    marker=dict(size=6),
                    showlegend=False,
                )
                fig.add_trace(mean_line, row=r, col=c)
            
            x_label_text = f"{xlabel} (rounded)" if not is_categorical else xlabel
            fig.update_xaxes(title_text=x_label_text, row=r, col=c)
            fig.update_yaxes(title_text="Mold Level σ [mm]", row=r, col=c)
        
        fig.update_layout(
            height=900,
            width=1200,
            title=f"{self.strand_config.strand_name} - Normal process: parameter correlations",
            boxmode="overlay",
            showlegend=False,
        )
        
        self.save_figure(fig, "parameter_correlations", "html")
        fig.show()

print("✅ VisualizationFactory class defined")

## 📤 Results Export & Reporting

This module handles **structured output management** for analysis results:
* Save DataFrames as CSV/Parquet with strand-specific naming
* Generate summary reports comparing strands
* Export metadata and configuration for reproducibility
* Create consolidated multi-strand reports

In [0]:
class ResultsExporter:
    """
    Handles saving and exporting analysis results.
    
    Usage:
        exporter = ResultsExporter(strand_config)
        exporter.save_dataframe(df_seq, "sequence_statistics")
    """
    
    def __init__(self, strand_config: StrandConfig):
        self.strand_config = strand_config
        self.logger_prefix = f"[{strand_config.strand_name}]"
        
        # Create output directories using dbutils
        for dir_path in [strand_config.figures_dir, strand_config.reports_dir, strand_config.data_dir]:
            try:
                dbutils.fs.mkdirs(dir_path)
            except Exception as e:
                # Directory might already exist, that's okay
                pass
    
    def log(self, message: str):
        print(f"{self.logger_prefix} {message}")
    
    def save_dataframe(self, df: pd.DataFrame, base_name: str, format: str = "csv"):
        """
        Save DataFrame with strand-specific filename.
        """
        filename = self.strand_config.get_output_filename(base_name, format)
        filepath = f"{self.strand_config.data_dir}/{filename}"
        
        # Convert DBFS path to local path for pandas
        local_filepath = filepath.replace("/dbfs", "")
        
        if format == "csv":
            df.to_csv(local_filepath, index=False)
        elif format == "parquet":
            df.to_parquet(local_filepath, index=False)
        
        self.log(f"💾 Saved DataFrame: {filename} ({len(df)} rows)")
        return filepath
    
    def save_summary_report(self, df_seq: pd.DataFrame):
        """
        Generate and save summary statistics report.
        """
        self.log("📊 Generating summary report...")
        
        report = []
        report.append(f"# {self.strand_config.strand_name} - Analysis Summary Report")
        report.append(f"Generated: {CONFIG.timestamp}")
        report.append("\n" + "="*70 + "\n")
        
        # Overall statistics
        report.append("## Overall Statistics")
        report.append(f"Total sequences analyzed: {len(df_seq)}")
        report.append(f"\nDisturbance type distribution:")
        for dtype, count in df_seq["disturbance_type"].value_counts().items():
            pct = 100 * count / len(df_seq)
            report.append(f"  - {dtype}: {count} ({pct:.1f}%)")
        
        # Mold level stability
        report.append(f"\n## Mold Level Stability")
        stable_count = (df_seq["MOLD_LEVEL_std [mm]"] <= CONFIG.ml_stability_threshold_mm).sum()
        stable_pct = 100 * stable_count / len(df_seq)
        report.append(f"Sequences with σ ≤ {CONFIG.ml_stability_threshold_mm} mm: {stable_count}/{len(df_seq)} ({stable_pct:.1f}%)")
        report.append(f"Mean mold level σ: {df_seq['MOLD_LEVEL_std [mm]'].mean():.2f} mm")
        report.append(f"Median mold level σ: {df_seq['MOLD_LEVEL_std [mm]'].median():.2f} mm")
        
        # Process parameters
        report.append(f"\n## Process Parameters (Mean ± Std)")
        report.append(f"Casting Speed: {df_seq['CASTING_SPEED_avg [m/min]'].mean():.2f} ± {df_seq['CASTING_SPEED_avg [m/min]'].std():.2f} m/min")
        report.append(f"Mold Width: {df_seq['MOLD_WIDTH_avg [m]'].mean():.3f} ± {df_seq['MOLD_WIDTH_avg [m]'].std():.3f} m")
        report.append(f"SEN Depth: {df_seq['SEN_avg [mm]'].mean():.1f} ± {df_seq['SEN_avg [mm]'].std():.1f} mm")
        
        report_text = "\n".join(report)
        
        # Save report
        filename = self.strand_config.get_output_filename("summary_report", "txt")
        filepath = f"{self.strand_config.reports_dir}/{filename}"
        local_filepath = filepath.replace("/dbfs", "")
        
        with open(local_filepath, "w") as f:
            f.write(report_text)
        
        self.log(f"💾 Saved summary report: {filename}")
        
        # Also print to console
        print("\n" + report_text + "\n")
        
        return filepath

print("✅ ResultsExporter class defined")

## 🚀 Main Orchestration Pipeline

This section provides the **main execution logic** that:
* Orchestrates the entire analysis workflow
* Supports single-strand or multi-strand execution
* Includes error handling and progress tracking
* Generates comprehensive outputs for each strand

In [0]:
from typing import Dict, Any
import traceback

class StrandAnalysisPipeline:
    """
    Complete end-to-end analysis pipeline for a single strand.
    
    Usage:
        pipeline = StrandAnalysisPipeline(STRAND_CONFIGS["23_6"], CONFIG)
        results = pipeline.run()
    """
    
    def __init__(self, strand_config: StrandConfig, analysis_config: AnalysisConfig):
        self.strand_config = strand_config
        self.config = analysis_config
        self.logger_prefix = f"[{strand_config.strand_name}]"
        
        # Initialize components
        self.loader = StrandDataLoader(strand_config, analysis_config)
        self.analyzer = SequenceAnalyzer(strand_config, analysis_config)
        self.visualizer = VisualizationFactory(strand_config, analysis_config)
        self.exporter = ResultsExporter(strand_config)
    
    def log(self, message: str):
        print(f"\n{self.logger_prefix} {message}")
    
    def run(self, generate_visualizations: bool = True) -> Dict[str, Any]:
        """
        Execute complete analysis pipeline.
        
        Returns:
            Dictionary with analysis results and metadata
        """
        self.log("🚀 " + "="*60)
        self.log(f"STARTING ANALYSIS PIPELINE FOR {self.strand_config.strand_name}")
        self.log("="*60)
        
        results = {
            "strand_id": self.strand_config.strand_id,
            "strand_name": self.strand_config.strand_name,
            "success": False,
            "error": None,
        }
        
        try:
            # Step 1: Load and preprocess data
            self.log("📊 STEP 1: Data Loading & Preprocessing")
            df_pandas = self.loader.load_and_process()
            results["df_raw"] = df_pandas
            results["raw_row_count"] = len(df_pandas)
            
            # Step 2: Sequence analysis
            self.log("🔍 STEP 2: Sequence Analysis & Disturbance Detection")
            df_fc_mold, df_seq, normal_groups, abnormal_groups = self.analyzer.analyze(df_pandas)
            results["df_fc_mold"] = df_fc_mold
            results["df_seq"] = df_seq
            results["normal_groups"] = normal_groups
            results["abnormal_groups"] = abnormal_groups
            results["sequence_count"] = len(df_seq)
            
            # Step 3: Export results
            self.log("💾 STEP 3: Exporting Results")
            self.exporter.save_dataframe(df_seq, "sequence_statistics", "csv")
            self.exporter.save_dataframe(df_seq, "sequence_statistics", "parquet")
            self.exporter.save_summary_report(df_seq)
            
            # Step 4: Generate visualizations
            if generate_visualizations:
                self.log("🎨 STEP 4: Generating Visualizations")
                
                # Disturbance examples
                example_indices = {
                    "Normal": 127 if 127 < len(normal_groups) else 0,
                    "High variability": df_seq[df_seq["disturbance_type"] == "High variability"].index[0] if len(df_seq[df_seq["disturbance_type"] == "High variability"]) > 0 else None,
                    "Excursion": df_seq[df_seq["disturbance_type"].str.contains("Excursion", na=False)].index[0] if len(df_seq[df_seq["disturbance_type"].str.contains("Excursion", na=False)]) > 0 else None,
                }
                example_indices = {k: v for k, v in example_indices.items() if v is not None}
                
                if example_indices:
                    self.visualizer.plot_disturbance_examples(df_fc_mold, df_seq, normal_groups, example_indices)
                
                # Correlation plots
                self.visualizer.plot_correlation_scatter(df_seq, filter_normal_only=False)
                self.visualizer.plot_correlation_scatter(df_seq, filter_normal_only=True)
                self.visualizer.plot_correlation_heatmap(df_seq)
                self.visualizer.plot_parameter_correlations(df_seq)
            
            results["success"] = True
            
            self.log("✅ " + "="*60)
            self.log(f"ANALYSIS COMPLETE FOR {self.strand_config.strand_name}")
            self.log("="*60)
            
        except Exception as e:
            results["success"] = False
            results["error"] = str(e)
            results["traceback"] = traceback.format_exc()
            
            self.log(f"❌ ERROR: {str(e)}")
            print(traceback.format_exc())
        
        return results

print("✅ StrandAnalysisPipeline class defined")

In [0]:
# Execute analysis for a single strand
# Change strand_id to "23_5" or "23_6" as needed

STRAND_TO_ANALYZE = "23_6"  # 👈 Change this to switch strands

print(f"\n{'='*70}")
print(f"EXECUTING ANALYSIS FOR STRAND: {STRAND_TO_ANALYZE}")
print(f"{'='*70}\n")

pipeline = StrandAnalysisPipeline(
    strand_config=STRAND_CONFIGS[STRAND_TO_ANALYZE],
    analysis_config=CONFIG
)

results = pipeline.run(generate_visualizations=True)

if results["success"]:
    print(f"\n✅ Analysis completed successfully for {results['strand_name']}")
    print(f"   - Raw data: {results['raw_row_count']:,} rows")
    print(f"   - Sequences identified: {results['sequence_count']}")
    print(f"   - Results saved to: {STRAND_CONFIGS[STRAND_TO_ANALYZE].output_dir}")
else:
    print(f"\n❌ Analysis failed for {results['strand_name']}")
    print(f"   Error: {results['error']}")

In [0]:
# Execute analysis for BOTH strands in sequence
# Uncomment to run multi-strand analysis


print(f"\n{'='*70}")
print(f"EXECUTING MULTI-STRAND ANALYSIS")
print(f"{'='*70}\n")

all_results = {}

for strand_id, strand_config in STRAND_CONFIGS.items():
    print(f"\n\n{'#'*70}")
    print(f"# Processing: {strand_config.strand_name}")
    print(f"{'#'*70}\n")
    
    pipeline = StrandAnalysisPipeline(
        strand_config=strand_config,
        analysis_config=CONFIG
    )
    
    results = pipeline.run(generate_visualizations=True)
    all_results[strand_id] = results

# Summary
print(f"\n\n{'='*70}")
print("MULTI-STRAND ANALYSIS SUMMARY")
print(f"{'='*70}\n")

for strand_id, results in all_results.items():
    status = "✅ SUCCESS" if results["success"] else "❌ FAILED"
    print(f"{status} - {results['strand_name']}")
    if results["success"]:
        print(f"   Sequences: {results['sequence_count']}")
        print(f"   Output: {STRAND_CONFIGS[strand_id].output_dir}")
    else:
        print(f"   Error: {results['error']}")
    print()


print("ℹ️ Uncomment the code above to run multi-strand analysis")

# 📊 Dashboard Section

**Two working approaches:**
1. **Enhanced Standalone HTML Dashboard** — self-contained HTML file with Plotly charts (no server needed)
2. **Databricks AI/BI (Lakeview)** — native dashboard querying Delta tables (recommended for sharing)

**Workflow:** Run the pipeline (cells above) → Generate HTML dashboard OR create Delta tables → Build Lakeview dashboardED:** Use the Databricks AI/BI Dashboard approach (Delta tables + Lakeview) for production.

---

### Original description:
Interactive Dashboard for CC23 Mold Level Analysis

## Overview
This interactive dashboard provides comprehensive visualization and exploration capabilities for continuous casting mold level data:

### 🎯 Key Features
* **Multi-strand support**: Toggle between Strand 23-5 and 23-6
* **Time series explorer**: Interactive plots with zoom, pan, and hover details
* **Sequence analysis**: Drill down into individual sequences and disturbance patterns
* **Parameter correlations**: Explore relationships between process parameters and mold level stability
* **Quality casting insights**: Filter and analyze by casting quality grades
* **Real-time filtering**: Dynamic filtering by date range, parameters, and disturbance types

### 🛠️ Technology Stack
* **Plotly Dash**: Interactive web-based dashboard framework
* **Plotly**: High-performance interactive visualizations
* **Pandas**: Data manipulation and analysis
* **Bootstrap**: Responsive UI components

In [0]:
def create_enhanced_standalone_dashboard(all_results, output_path):
    """
    Create an enhanced standalone HTML dashboard with sequence drill-down.
    Uses Plotly's built-in interactivity (no server needed).
    """
    
    print("\n" + "="*70)
    print("🎨 GENERATING ENHANCED STANDALONE DASHBOARD")
    print("="*70)
    
    strands_available = [sid for sid in ['23_5', '23_6'] 
                        if sid in all_results and all_results[sid]['success']]
    
    if len(strands_available) == 0:
        print("\n⚠️ No data available. Run analysis pipeline first.")
        return None
    
    # We'll create multiple HTML files - one master dashboard + individual sequence viewers
    html_files = {}
    
    for strand_id in strands_available:
        result = all_results[strand_id]
        df_seq = result['df_seq']
        df_fc_mold = result['df_fc_mold']
        normal_groups = result['normal_groups']
        strand_name = STRAND_CONFIGS[strand_id].strand_name
        
        print(f"\n📊 Generating dashboard for {strand_name}...")
        
        # ============================================================
        # 1. CREATE INTERACTIVE SEQUENCE SELECTOR
        # ============================================================
        
        # Create a figure with dropdown to select sequences
        fig_seq_explorer = go.Figure()
        
        # Add traces for each sequence (initially all hidden except first)
        for seq_idx in range(min(len(normal_groups), 100)):  # Limit to first 100 sequences
            group = normal_groups[seq_idx]
            df_seq_data = df_fc_mold.iloc[group]
            
            seq_info = df_seq.iloc[seq_idx]
            seq_name = seq_info['Seq_Name']
            disturbance = seq_info['disturbance_type']
            ml_sigma = seq_info['MOLD_LEVEL_std [mm]']
            vc_avg = seq_info['CASTING_SPEED_avg [m/min]']
            quality = seq_info.get('Quality casting', 'N/A')
            
            fig_seq_explorer.add_trace(
                go.Scatter(
                    x=df_seq_data['plainTimeStamp'],
                    y=df_seq_data['Mold Level'],
                    mode='lines',
                    name=seq_name,
                    visible=(seq_idx == 0),  # Only first sequence visible initially
                    line=dict(width=2),
                    hovertemplate='Time: %{x}<br>Mold Level: %{y:.2f} mm<extra></extra>'
                )
            )
        
        # Create dropdown menu to select sequences
        dropdown_buttons = []
        for seq_idx in range(min(len(normal_groups), 100)):
            seq_info = df_seq.iloc[seq_idx]
            seq_name = seq_info['Seq_Name']
            disturbance = seq_info['disturbance_type']
            ml_sigma = seq_info['MOLD_LEVEL_std [mm]']
            
            # Create visibility array (only this sequence visible)
            visible = [i == seq_idx for i in range(min(len(normal_groups), 100))]
            
            dropdown_buttons.append(
                dict(
                    label=f"{seq_name} | {disturbance} | σ={ml_sigma:.2f}mm",
                    method="update",
                    args=[{"visible": visible},
                          {"title": f"{strand_name} - {seq_name}<br>Disturbance: {disturbance} | ML σ: {ml_sigma:.2f} mm"}]
                )
            )
        
        fig_seq_explorer.update_layout(
            updatemenus=[
                dict(
                    buttons=dropdown_buttons,
                    direction="down",
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.01,
                    xanchor="left",
                    y=1.15,
                    yanchor="top",
                    bgcolor="#f0f0f0",
                    bordercolor="#666",
                    borderwidth=1
                )
            ],
            title=f"{strand_name} - Interactive Sequence Explorer<br><sub>Select sequence from dropdown above</sub>",
            xaxis_title="Time",
            yaxis_title="Mold Level [mm]",
            height=600,
            hovermode='x'
        )
        
        # ============================================================
        # 2. CREATE PARAMETER CORRELATION WITH SEQUENCE HIGHLIGHTING
        # ============================================================
        
        fig_corr_interactive = go.Figure()
        
        # Add scatter for each disturbance type
        for dtype in df_seq['disturbance_type'].unique():
            df_type = df_seq[df_seq['disturbance_type'] == dtype]
            
            fig_corr_interactive.add_trace(
                go.Scatter(
                    x=df_type['MOLD_WIDTH_avg [m]'],
                    y=df_type['CASTING_SPEED_avg [m/min]'],
                    mode='markers',
                    name=dtype,
                    marker=dict(
                        size=8,
                        opacity=0.7,
                        line=dict(width=0.5, color='white')
                    ),
                    customdata=df_type[['Seq_Name', 'MOLD_LEVEL_std [mm]', 'Quality casting']].values,
                    hovertemplate=(
                        '<b>%{customdata[0]}</b><br>'
                        'Width: %{x:.3f} m<br>'
                        'Speed: %{y:.2f} m/min<br>'
                        'ML σ: %{customdata[1]:.2f} mm<br>'
                        'Quality: %{customdata[2]}<br>'
                        f'Type: {dtype}<extra></extra>'
                    )
                )
            )
        
        # Add buttons to filter by disturbance type
        filter_buttons = [
            dict(
                label="All",
                method="update",
                args=[{"visible": [True] * len(df_seq['disturbance_type'].unique())}]
            )
        ]
        
        for i, dtype in enumerate(df_seq['disturbance_type'].unique()):
            visible = [j == i for j in range(len(df_seq['disturbance_type'].unique()))]
            filter_buttons.append(
                dict(
                    label=dtype,
                    method="update",
                    args=[{"visible": visible}]
                )
            )
        
        fig_corr_interactive.update_layout(
            updatemenus=[
                dict(
                    buttons=filter_buttons,
                    direction="down",
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.01,
                    xanchor="left",
                    y=1.15,
                    yanchor="top",
                    bgcolor="#f0f0f0",
                    bordercolor="#666",
                    borderwidth=1
                )
            ],
            title=f"{strand_name} - Mold Width vs Casting Speed<br><sub>Filter by disturbance type using dropdown</sub>",
            xaxis_title="Mold Width [m]",
            yaxis_title="Casting Speed [m/min]",
            height=600,
            hovermode='closest',
            legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02)
        )
        
        # ============================================================
        # 3. CREATE QUALITY CASTING ANALYSIS
        # ============================================================
        
        df_with_quality = df_seq.dropna(subset=['Quality casting'])
        
        if len(df_with_quality) > 0:
            fig_quality = go.Figure()
            
            # Box plot for each quality grade
            for quality in sorted(df_with_quality['Quality casting'].unique()):
                df_q = df_with_quality[df_with_quality['Quality casting'] == quality]
                
                fig_quality.add_trace(
                    go.Box(
                        y=df_q['MOLD_LEVEL_std [mm]'],
                        name=quality,
                        boxmean='sd',
                        customdata=df_q[['Seq_Name', 'CASTING_SPEED_avg [m/min]']].values,
                        hovertemplate=(
                            '<b>%{customdata[0]}</b><br>'
                            'Quality: ' + quality + '<br>'
                            'ML σ: %{y:.2f} mm<br>'
                            'Vc: %{customdata[1]:.2f} m/min<extra></extra>'
                        )
                    )
                )
            
            fig_quality.update_layout(
                title=f"{strand_name} - Mold Level Stability by Quality Casting Grade",
                xaxis_title="Quality Casting Grade",
                yaxis_title="Mold Level σ [mm]",
                height=500,
                showlegend=False
            )
        else:
            fig_quality = None
        
        # ============================================================
        # 4. SAVE AS STANDALONE HTML
        # ============================================================
        
        # Create comprehensive HTML with all interactive plots
        html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{strand_name} - Interactive Analysis Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 0;
            padding: 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        }}
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            background: white;
            border-radius: 15px;
            padding: 30px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.2);
        }}
        .header {{
            text-align: center;
            margin-bottom: 30px;
            padding-bottom: 20px;
            border-bottom: 3px solid #667eea;
        }}
        .header h1 {{
            color: #667eea;
            margin: 0;
            font-size: 36px;
        }}
        .header p {{
            color: #666;
            margin: 10px 0 0 0;
            font-size: 18px;
        }}
        .metrics {{
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 20px;
            margin-bottom: 30px;
        }}
        .metric-card {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 25px;
            border-radius: 10px;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }}
        .metric-value {{
            font-size: 32px;
            font-weight: bold;
            margin-bottom: 5px;
        }}
        .metric-label {{
            font-size: 14px;
            opacity: 0.9;
        }}
        .plot-section {{
            margin-bottom: 40px;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 10px;
        }}
        .plot-section h2 {{
            color: #667eea;
            margin-top: 0;
            font-size: 24px;
            border-bottom: 2px solid #667eea;
            padding-bottom: 10px;
        }}
        .instructions {{
            background: #e7f3ff;
            border-left: 4px solid #2196F3;
            padding: 15px;
            margin-bottom: 20px;
            border-radius: 5px;
        }}
        .instructions h3 {{
            margin-top: 0;
            color: #2196F3;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🏭 {strand_name} - Interactive Analysis Dashboard</h1>
            <p>TATA IJmuiden Continuous Casting | Mold Level Stability Analysis</p>
        </div>
        """
        
        # Add metrics
        total_seq = len(df_seq)
        stable_pct = 100 * (df_seq['MOLD_LEVEL_std [mm]'] <= 2.0).sum() / total_seq
        mean_sigma = df_seq['MOLD_LEVEL_std [mm]'].mean()
        normal_pct = 100 * (df_seq['disturbance_type'] == 'Normal').sum() / total_seq
        
        html_content += f"""
        <div class="metrics">
            <div class="metric-card">
                <div class="metric-value">{total_seq:,}</div>
                <div class="metric-label">Total Sequences</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{stable_pct:.1f}%</div>
                <div class="metric-label">Stable (σ ≤ 2mm)</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{mean_sigma:.2f} mm</div>
                <div class="metric-label">Mean ML σ</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{normal_pct:.1f}%</div>
                <div class="metric-label">Normal Sequences</div>
            </div>
        </div>
        
        <div class="instructions">
            <h3>🎯 How to Use This Dashboard</h3>
            <ul>
                <li><strong>Sequence Explorer:</strong> Use the dropdown to select and view individual sequences</li>
                <li><strong>Parameter Correlations:</strong> Click the filter buttons to show/hide disturbance types</li>
                <li><strong>Interactive Features:</strong> Zoom (drag), Pan (shift+drag), Reset (double-click)</li>
                <li><strong>Download:</strong> Click camera icon on any plot to save as PNG</li>
            </ul>
        </div>
        """
        
        # Add sequence explorer
        html_content += """
        <div class="plot-section">
            <h2>🔍 Interactive Sequence Explorer</h2>
            <p><strong>Select a sequence from the dropdown above the plot to view its time series data.</strong></p>
        """
        html_content += fig_seq_explorer.to_html(full_html=False, include_plotlyjs=False, div_id=f'seq-explorer-{strand_id}')
        html_content += "</div>"
        
        # Add correlation plot
        html_content += """
        <div class="plot-section">
            <h2>📊 Parameter Correlations (Interactive Filtering)</h2>
            <p><strong>Use the dropdown above the plot to filter by disturbance type.</strong></p>
        """
        html_content += fig_corr_interactive.to_html(full_html=False, include_plotlyjs=False, div_id=f'corr-{strand_id}')
        html_content += "</div>"
        
        # Add quality casting analysis if available
        if fig_quality:
            html_content += """
            <div class="plot-section">
                <h2>⭐ Quality Casting Analysis</h2>
                <p><strong>Mold level stability distribution by steel grade.</strong></p>
            """
            html_content += fig_quality.to_html(full_html=False, include_plotlyjs=False, div_id=f'quality-{strand_id}')
            html_content += "</div>"
        
        # Add disturbance distribution
        dist_counts = df_seq['disturbance_type'].value_counts()
        fig_dist = go.Figure(data=[go.Bar(
            x=dist_counts.index,
            y=dist_counts.values,
            marker=dict(
                color=['#28a745', '#ffc107', '#fd7e14', '#e83e8c', '#dc3545'][:len(dist_counts)],
                line=dict(color='white', width=2)
            ),
            text=dist_counts.values,
            textposition='auto',
            hovertemplate='%{x}<br>Count: %{y}<br>Percentage: %{y:.1%}<extra></extra>'
        )])
        
        fig_dist.update_layout(
            title=f"{strand_name} - Disturbance Type Distribution",
            xaxis_title="Disturbance Type",
            yaxis_title="Count",
            height=400
        )
        fig_dist.update_xaxes(tickangle=-45)
        
        html_content += """
        <div class="plot-section">
            <h2>📈 Disturbance Type Distribution</h2>
        """
        html_content += fig_dist.to_html(full_html=False, include_plotlyjs=False, div_id=f'dist-{strand_id}')
        html_content += "</div>"
        
        # Close HTML
        html_content += """
        <div style="text-align: center; margin-top: 40px; padding-top: 20px; border-top: 2px solid #ddd; color: #666;">
            <p>Generated: """ + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + """ | Powered by Plotly</p>
        </div>
    </div>
    
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
</body>
</html>
        """
        
        # Save to file
        filename = f"cc23_interactive_dashboard_{strand_id}.html"
        filepath = f"/Workspace/Users/dieudonne.nkulikiyimfura@se.abb.com/TATAIjmulden_FCMoldG5/{filename}"
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        html_files[strand_id] = filepath
        
        print(f"   ✅ Created: {filename} ({len(html_content)/1024:.1f} KB)")
    
    print(f"\n" + "="*70)
    print("✅ ENHANCED DASHBOARDS CREATED!")
    print("="*70)
    
    for strand_id, filepath in html_files.items():
        print(f"\n📁 {STRAND_CONFIGS[strand_id].strand_name}:")
        print(f"   {filepath}")
    
    print(f"\n💡 Features:")
    print(f"   ✓ Interactive sequence selector (dropdown)")
    print(f"   ✓ Disturbance type filtering")
    print(f"   ✓ Quality casting analysis")
    print(f"   ✓ Full Plotly zoom/pan/hover")
    print(f"   ✓ No server required!")
    
    return html_files

print("✅ Enhanced dashboard generator defined")

In [0]:
# Generate the enhanced standalone dashboard with sequence drill-down

if len(dashboard_results) > 0:
    html_files = create_enhanced_standalone_dashboard(dashboard_results, None)
    
    print("\n" + "="*70)
    print("🎉 SUCCESS! ENHANCED DASHBOARDS READY")
    print("="*70)
    
    print("\n💾 Download Instructions:")
    print("   1. Databricks UI → Workspace (left sidebar)")
    print("   2. Navigate to: TATAIjmulden_FCMoldG5/")
    print("   3. Download the HTML file(s)")
    print("   4. Open in your browser")
    
    print("\n✨ New Features:")
    print("   ✓ Dropdown to select and view ANY sequence (up to 100)")
    print("   ✓ Each sequence shows full time series data")
    print("   ✓ Filter correlations by disturbance type")
    print("   ✓ Quality casting box plots with hover details")
    print("   ✓ All plots fully interactive (zoom, pan, download)")
    print("   ✓ Professional styling with gradient backgrounds")
    
    print("\n💡 No server needed - works offline!")
    
else:
    print("\n⚠️ No data available. Run analysis pipeline first.")

In [0]:
def create_temporal_coverage_bar(df_raw, strand_name):
    """
    Create a temporal coverage bar showing data availability over time.
    Identifies gaps and shows data density.
    """
    
    # Resample to hourly bins to show coverage
    df_temp = df_raw.set_index('plainTimeStamp')
    hourly_counts = df_temp.resample('1H').size()
    
    # Identify gaps (hours with no data)
    has_data = hourly_counts > 0
    
    fig = go.Figure()
    
    # Add bar showing data availability
    fig.add_trace(go.Bar(
        x=hourly_counts.index,
        y=hourly_counts.values,
        marker=dict(
            color=hourly_counts.values,
            colorscale=[
                [0, '#dc3545'],      # Red for no data
                [0.01, '#ffc107'],   # Yellow for sparse
                [0.5, '#28a745'],    # Green for good coverage
                [1, '#007bff']       # Blue for dense
            ],
            showscale=True,
            colorbar=dict(title="Data Points<br>per Hour")
        ),
        hovertemplate='Time: %{x}<br>Data Points: %{y}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f"{strand_name} - Temporal Data Coverage<br><sub>Hourly data point counts | Gaps shown in red</sub>",
        xaxis_title="Time",
        yaxis_title="Data Points per Hour",
        height=300,
        hovermode='x',
        xaxis=dict(
            rangeslider=dict(visible=True),
            type="date"
        )
    )
    
    # Calculate coverage statistics
    total_hours = len(hourly_counts)
    hours_with_data = has_data.sum()
    coverage_pct = 100 * hours_with_data / total_hours
    
    gaps = (~has_data).astype(int)
    gap_starts = gaps.diff().fillna(0) == 1
    num_gaps = gap_starts.sum()
    
    stats_text = f"Coverage: {coverage_pct:.1f}% | {hours_with_data}/{total_hours} hours | {num_gaps} gaps detected"
    
    return fig, stats_text


def create_multi_parameter_plot(df_raw, parameters, time_range=None, strand_name=""):
    """
    Create interactive multi-parameter time series plot.
    
    Args:
        df_raw: Raw data DataFrame
        parameters: List of parameter column names to plot
        time_range: Tuple of (start_time, end_time) or None for all data
        strand_name: Strand identifier for title
    """
    
    # Apply time range filter
    df_plot = df_raw.copy()
    if time_range:
        start_time, end_time = time_range
        mask = (df_plot['plainTimeStamp'] >= start_time) & (df_plot['plainTimeStamp'] <= end_time)
        df_plot = df_plot[mask]
    
    # Sample if too large (prevent crashes)
    if len(df_plot) > 100000:
        df_plot = df_plot.sample(n=100000, random_state=42).sort_values('plainTimeStamp')
        sample_note = f"<br><sub>⚠️ Showing 100k sampled points (original: {len(df_raw):,})</sub>"
    else:
        sample_note = f"<br><sub>Showing {len(df_plot):,} data points</sub>"
    
    # Create subplots (one per parameter)
    n_params = len(parameters)
    fig = make_subplots(
        rows=n_params,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=[p for p in parameters]
    )
    
    # Add trace for each parameter
    colors = ['#007bff', '#28a745', '#dc3545', '#ffc107', '#6f42c1', '#17a2b8', '#fd7e14']
    
    for i, param in enumerate(parameters):
        if param not in df_plot.columns:
            continue
        
        # Determine units
        units = ""
        if 'Speed' in param or 'speed' in param:
            units = "m/min"
        elif 'Width' in param or 'width' in param:
            units = "m"
        elif 'Level' in param:
            units = "mm"
        elif 'Current' in param:
            units = "A"
        elif 'Flow' in param:
            units = "NL/min"
        elif 'Depth' in param:
            units = "m"
        
        fig.add_trace(
            go.Scattergl(
                x=df_plot['plainTimeStamp'],
                y=df_plot[param],
                mode='lines',
                name=param,
                line=dict(color=colors[i % len(colors)], width=1.5),
                hovertemplate=f'Time: %{{x}}<br>{param}: %{{y:.3f}} {units}<extra></extra>'
            ),
            row=i+1, col=1
        )
        
        fig.update_yaxes(title_text=f"{param}<br>[{units}]" if units else param, row=i+1, col=1)
    
    fig.update_xaxes(title_text="Time", row=n_params, col=1)
    
    fig.update_layout(
        title=f"{strand_name} - Multi-Parameter Time Series{sample_note}",
        height=300 * n_params,
        hovermode='x unified',
        showlegend=False,
        xaxis=dict(
            rangeslider=dict(visible=True, thickness=0.05),
            type="date"
        )
    )
    
    return fig


def create_parameter_distribution_grid(df_raw, parameters, strand_name=""):
    """
    Create grid of histograms for parameter distributions.
    """
    
    n_params = len(parameters)
    n_cols = 3
    n_rows = (n_params + n_cols - 1) // n_cols
    
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=parameters
    )
    
    for i, param in enumerate(parameters):
        if param not in df_raw.columns:
            continue
        
        row = i // n_cols + 1
        col = i % n_cols + 1
        
        # Remove NaN values
        data = df_raw[param].dropna()
        
        fig.add_trace(
            go.Histogram(
                x=data,
                nbinsx=50,
                marker=dict(color='#007bff', line=dict(color='white', width=1)),
                hovertemplate='Value: %{x:.3f}<br>Count: %{y}<extra></extra>'
            ),
            row=row, col=col
        )
        
        # Add statistics annotations
        mean_val = data.mean()
        median_val = data.median()
        
        fig.add_vline(
            x=mean_val,
            line_dash="dash",
            line_color="red",
            annotation_text=f"μ={mean_val:.2f}",
            annotation_position="top",
            row=row, col=col
        )
        
        fig.update_xaxes(title_text=param, row=row, col=col)
        fig.update_yaxes(title_text="Count", row=row, col=col)
    
    fig.update_layout(
        title=f"{strand_name} - Parameter Distributions",
        height=400 * n_rows,
        showlegend=False
    )
    
    return fig

print("✅ EDA helper functions defined")

In [0]:
def create_complete_eda_dashboard(all_results):
    """
    Create comprehensive EDA dashboard with:
    - Temporal coverage visualization
    - Interactive time range selection
    - Single/multiple parameter visualization
    - Distribution analysis
    """
    
    print("\n" + "="*70)
    print("🎨 GENERATING COMPLETE EDA DASHBOARD")
    print("="*70)
    
    strands_available = [sid for sid in ['23_5', '23_6'] 
                        if sid in all_results and all_results[sid]['success']]
    
    if len(strands_available) == 0:
        print("\n⚠️ No data available. Run analysis pipeline first.")
        return None
    
    html_files = {}
    
    for strand_id in strands_available:
        result = all_results[strand_id]
        df_raw = result['df_raw']
        df_seq = result['df_seq']
        df_fc_mold = result['df_fc_mold']
        normal_groups = result['normal_groups']
        strand_name = STRAND_CONFIGS[strand_id].strand_name
        
        print(f"\n📊 Generating EDA dashboard for {strand_name}...")
        
        # ============================================================
        # 1. TEMPORAL COVERAGE BAR
        # ============================================================
        print("   📅 Creating temporal coverage visualization...")
        fig_coverage, coverage_stats = create_temporal_coverage_bar(df_raw, strand_name)
        
        # ============================================================
        # 2. INTERACTIVE PARAMETER SELECTOR
        # ============================================================
        print("   📊 Creating multi-parameter explorer...")
        
        # Define available parameters
        available_params = [
            'castingSpeed',
            'moldWidth',
            'SENImmersionDepth',
            'Mold Level',
            'Mold Level Sensor Left',
            'Mold Level Sensor Right',
            'Argon Flow SEN',
            'Argon Flow Stopper',
            'EMBR Current AC Left Master',
            'EMBR Current DC Left Master',
            'EMBR Current DC Left Bottom'
        ]
        
        # Filter to existing columns
        available_params = [p for p in available_params if p in df_raw.columns]
        
        # Create figure with buttons to select parameter combinations
        fig_multi_param = go.Figure()
        
        # Pre-create traces for all parameters (initially hidden)
        for param in available_params:
            fig_multi_param.add_trace(
                go.Scattergl(
                    x=df_raw['plainTimeStamp'],
                    y=df_raw[param],
                    mode='lines',
                    name=param,
                    visible=False,
                    line=dict(width=1.5),
                    hovertemplate=f'Time: %{{x}}<br>{param}: %{{y:.3f}}<extra></extra>'
                )
            )
        
        # Create button menu for parameter selection
        param_buttons = []
        
        # Single parameter buttons
        for i, param in enumerate(available_params):
            visible = [j == i for j in range(len(available_params))]
            param_buttons.append(
                dict(
                    label=param,
                    method="update",
                    args=[{"visible": visible},
                          {"title": f"{strand_name} - {param}<br><sub>Use range slider below to zoom</sub>",
                           "yaxis": {"title": param}}]
                )
            )
        
        # Add separator
        param_buttons.append(
            dict(
                label="─── Multi-Parameter ───",
                method="skip"
            )
        )
        
        # Multi-parameter presets
        presets = [
            {
                'label': 'Casting Parameters',
                'params': ['castingSpeed', 'moldWidth', 'SENImmersionDepth']
            },
            {
                'label': 'Mold Level Sensors',
                'params': ['Mold Level', 'Mold Level Sensor Left', 'Mold Level Sensor Right']
            },
            {
                'label': 'Argon Flow',
                'params': ['Argon Flow SEN', 'Argon Flow Stopper']
            },
            {
                'label': 'EMBR Currents',
                'params': ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
            },
            {
                'label': 'All Parameters',
                'params': available_params
            }
        ]
        
        for preset in presets:
            visible = [param in preset['params'] for param in available_params]
            if any(visible):
                param_buttons.append(
                    dict(
                        label=preset['label'],
                        method="update",
                        args=[{"visible": visible},
                              {"title": f"{strand_name} - {preset['label']}<br><sub>Use range slider below to zoom</sub>"}]
                    )
                )
        
        # Set first parameter visible by default
        fig_multi_param.data[0].visible = True
        
        fig_multi_param.update_layout(
            updatemenus=[
                dict(
                    buttons=param_buttons,
                    direction="down",
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.01,
                    xanchor="left",
                    y=1.20,
                    yanchor="top",
                    bgcolor="#f0f0f0",
                    bordercolor="#666",
                    borderwidth=1
                )
            ],
            title=f"{strand_name} - {available_params[0]}<br><sub>Use range slider below to zoom</sub>",
            xaxis_title="Time",
            yaxis_title=available_params[0],
            height=600,
            hovermode='x',
            xaxis=dict(
                rangeslider=dict(
                    visible=True,
                    thickness=0.08,
                    bgcolor="#f0f0f0",
                    bordercolor="#666",
                    borderwidth=1
                ),
                type="date"
            )
        )
        
        # ============================================================
        # 3. PARAMETER DISTRIBUTIONS
        # ============================================================
        print("   📈 Creating parameter distributions...")
        
        key_params = ['castingSpeed', 'moldWidth', 'SENImmersionDepth', 
                     'Mold Level', 'Argon Flow SEN', 'Argon Flow Stopper']
        key_params = [p for p in key_params if p in df_raw.columns]
        
        fig_distributions = create_parameter_distribution_grid(df_raw, key_params, strand_name)
        
        # ============================================================
        # 4. CORRELATION MATRIX
        # ============================================================
        print("   🔗 Creating correlation matrix...")
        
        # Calculate correlations for key parameters
        corr_params = ['castingSpeed', 'moldWidth', 'SENImmersionDepth', 'Mold Level']
        corr_params = [p for p in corr_params if p in df_raw.columns]
        
        if len(corr_params) > 1:
            corr_matrix = df_raw[corr_params].corr()
            
            fig_corr_matrix = go.Figure(data=go.Heatmap(
                z=corr_matrix.values,
                x=corr_matrix.columns,
                y=corr_matrix.columns,
                colorscale='RdBu',
                zmid=0,
                text=corr_matrix.values,
                texttemplate='%{text:.2f}',
                textfont={"size": 12},
                hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>'
            ))
            
            fig_corr_matrix.update_layout(
                title=f"{strand_name} - Parameter Correlation Matrix",
                height=500,
                xaxis=dict(side='bottom'),
                yaxis=dict(autorange='reversed')
            )
        else:
            fig_corr_matrix = None
        
        # ============================================================
        # 5. DATA QUALITY SUMMARY
        # ============================================================
        print("   ✅ Calculating data quality metrics...")
        
        quality_summary = []
        for param in available_params:
            if param in df_raw.columns:
                total = len(df_raw)
                non_null = df_raw[param].notna().sum()
                null_pct = 100 * (total - non_null) / total
                mean_val = df_raw[param].mean()
                std_val = df_raw[param].std()
                min_val = df_raw[param].min()
                max_val = df_raw[param].max()
                
                quality_summary.append({
                    'Parameter': param,
                    'Missing (%)': f"{null_pct:.1f}%",
                    'Mean': f"{mean_val:.3f}",
                    'Std': f"{std_val:.3f}",
                    'Min': f"{min_val:.3f}",
                    'Max': f"{max_val:.3f}"
                })
        
        df_quality = pd.DataFrame(quality_summary)
        
        # ============================================================
        # 6. BUILD COMPLETE HTML
        # ============================================================
        
        html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{strand_name} - Complete EDA Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 0;
            padding: 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        }}
        .container {{
            max-width: 1600px;
            margin: 0 auto;
            background: white;
            border-radius: 15px;
            padding: 30px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.2);
        }}
        .header {{
            text-align: center;
            margin-bottom: 30px;
            padding-bottom: 20px;
            border-bottom: 3px solid #667eea;
        }}
        .header h1 {{
            color: #667eea;
            margin: 0;
            font-size: 36px;
        }}
        .plot-section {{
            margin-bottom: 40px;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 10px;
        }}
        .plot-section h2 {{
            color: #667eea;
            margin-top: 0;
            font-size: 24px;
            border-bottom: 2px solid #667eea;
            padding-bottom: 10px;
        }}
        .instructions {{
            background: #e7f3ff;
            border-left: 4px solid #2196F3;
            padding: 15px;
            margin-bottom: 20px;
            border-radius: 5px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 15px;
        }}
        th, td {{
            padding: 10px;
            text-align: left;
            border-bottom: 1px solid #ddd;
        }}
        th {{
            background-color: #667eea;
            color: white;
            font-weight: bold;
        }}
        tr:hover {{
            background-color: #f5f5f5;
        }}
        .stat-badge {{
            display: inline-block;
            padding: 5px 10px;
            border-radius: 5px;
            background: #28a745;
            color: white;
            font-weight: bold;
            margin: 5px;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🔬 {strand_name} - Exploratory Data Analysis</h1>
            <p>Comprehensive Parameter Analysis | Time Series EDA</p>
        </div>
        
        <div class="instructions">
            <h3>🎯 How to Use This EDA Dashboard</h3>
            <ul>
                <li><strong>Temporal Coverage:</strong> Shows data availability over time (gaps in red)</li>
                <li><strong>Parameter Explorer:</strong> Select single or multiple parameters from dropdown</li>
                <li><strong>Range Slider:</strong> Drag the slider below plots to zoom into specific time periods</li>
                <li><strong>Interactive:</strong> Hover for details, drag to zoom, double-click to reset</li>
                <li><strong>Distributions:</strong> View histograms with mean/median markers</li>
            </ul>
        </div>
        
        <div class="plot-section">
            <h2>📅 Temporal Data Coverage</h2>
            <p><strong>{coverage_stats}</strong></p>
            <p><em>This bar chart shows hourly data point counts. Red bars indicate gaps in data collection.</em></p>
        """
        
        html_content += fig_coverage.to_html(full_html=False, include_plotlyjs=False, div_id=f'coverage-{strand_id}')
        html_content += "</div>"
        
        # Add multi-parameter explorer
        html_content += """
        <div class="plot-section">
            <h2>📊 Interactive Multi-Parameter Explorer</h2>
            <p><strong>Select parameters from the dropdown menu above the plot.</strong></p>
            <p><em>Use the range slider at the bottom to zoom into specific time periods (prevents crashes with large data).</em></p>
        """
        html_content += fig_multi_param.to_html(full_html=False, include_plotlyjs=False, div_id=f'multi-param-{strand_id}')
        html_content += "</div>"
        
        # Add distributions
        html_content += """
        <div class="plot-section">
            <h2>📈 Parameter Distributions</h2>
            <p><strong>Histograms showing the distribution of key casting parameters.</strong></p>
            <p><em>Red dashed line indicates mean value.</em></p>
        """
        html_content += fig_distributions.to_html(full_html=False, include_plotlyjs=False, div_id=f'dist-{strand_id}')
        html_content += "</div>"
        
        # Add correlation matrix
        if fig_corr_matrix:
            html_content += """
            <div class="plot-section">
                <h2>🔗 Parameter Correlation Matrix</h2>
                <p><strong>Correlation coefficients between key parameters.</strong></p>
                <p><em>Blue = positive correlation, Red = negative correlation</em></p>
            """
            html_content += fig_corr_matrix.to_html(full_html=False, include_plotlyjs=False, div_id=f'corr-matrix-{strand_id}')
            html_content += "</div>"
        
        # Add data quality table
        html_content += """
        <div class="plot-section">
            <h2>✅ Data Quality Summary</h2>
            <table>
                <thead>
                    <tr>
                        <th>Parameter</th>
                        <th>Missing (%)</th>
                        <th>Mean</th>
                        <th>Std Dev</th>
                        <th>Min</th>
                        <th>Max</th>
                    </tr>
                </thead>
                <tbody>
        """
        
        for _, row in df_quality.iterrows():
            html_content += f"""
                    <tr>
                        <td><strong>{row['Parameter']}</strong></td>
                        <td>{row['Missing (%)']}</td>
                        <td>{row['Mean']}</td>
                        <td>{row['Std']}</td>
                        <td>{row['Min']}</td>
                        <td>{row['Max']}</td>
                    </tr>
            """
        
        html_content += """
                </tbody>
            </table>
        </div>
        """
        
        # Add summary statistics
        html_content += f"""
        <div class="plot-section">
            <h2>📊 Dataset Summary</h2>
            <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px;">
                <div class="stat-badge">Total Rows: {len(df_raw):,}</div>
                <div class="stat-badge">Time Span: {(df_raw['plainTimeStamp'].max() - df_raw['plainTimeStamp'].min()).days} days</div>
                <div class="stat-badge">Parameters: {len(available_params)}</div>
                <div class="stat-badge">Sequences: {len(df_seq)}</div>
            </div>
        </div>
        """
        
        # Close HTML
        html_content += """
        <div style="text-align: center; margin-top: 40px; padding-top: 20px; border-top: 2px solid #ddd; color: #666;">
            <p>Generated: """ + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + """ | Powered by Plotly</p>
        </div>
    </div>
    
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
</body>
</html>
        """
        
        # Save to file
        filename = f"cc23_eda_dashboard_{strand_id}.html"
        filepath = f"/Workspace/Users/dieudonne.nkulikiyimfura@se.abb.com/TATAIjmulden_FCMoldG5/{filename}"
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        html_files[strand_id] = filepath
        
        print(f"   ✅ Created: {filename} ({len(html_content)/1024:.1f} KB)")
    
    print(f"\n" + "="*70)
    print("✅ EDA DASHBOARDS CREATED!")
    print("="*70)
    
    for strand_id, filepath in html_files.items():
        print(f"\n📁 {STRAND_CONFIGS[strand_id].strand_name}:")
        print(f"   {filepath}")
    
    print(f"\n💡 Features:")
    print(f"   ✓ Temporal coverage bar (shows data gaps)")
    print(f"   ✓ Interactive parameter selector (single/multiple)")
    print(f"   ✓ Range slider for time selection (prevents crashes)")
    print(f"   ✓ Parameter distributions with statistics")
    print(f"   ✓ Correlation matrix")
    print(f"   ✓ Data quality summary table")
    
    return html_files

print("✅ Complete EDA dashboard generator defined")

In [0]:
# Generate the complete EDA dashboard

if len(dashboard_results) > 0:
    eda_files = create_complete_eda_dashboard(dashboard_results)
    
    print("\n" + "="*70)
    print("🎉 EDA DASHBOARD GENERATION COMPLETE")
    print("="*70)
    
    print("\n📥 Download Instructions:")
    print("   1. Databricks UI → Workspace")
    print("   2. Navigate to: TATAIjmulden_FCMoldG5/")
    print("   3. Download: cc23_eda_dashboard_23_6.html")
    print("   4. Open in browser")
    
    print("\n✨ EDA Dashboard Features:")
    print("\n   📅 TEMPORAL COVERAGE:")
    print("      • Hourly data point counts")
    print("      • Visual identification of gaps (red bars)")
    print("      • Coverage statistics")
    
    print("\n   📊 PARAMETER EXPLORER:")
    print("      • Dropdown menu with all parameters")
    print("      • Single parameter view")
    print("      • Multi-parameter presets:")
    print("        - Casting Parameters (Vc, Width, SEN)")
    print("        - Mold Level Sensors (3 sensors)")
    print("        - Argon Flow (SEN + Stopper)")
    print("        - EMBR Currents (3 currents)")
    print("        - All Parameters")
    
    print("\n   🎚️ RANGE SLIDER:")
    print("      • Drag to select time range")
    print("      • Prevents crashes with large data")
    print("      • Smooth zooming and panning")
    
    print("\n   📈 DISTRIBUTIONS:")
    print("      • Histograms for all key parameters")
    print("      • Mean value markers")
    print("      • Interactive hover details")
    
    print("\n   🔗 CORRELATION MATRIX:")
    print("      • Heatmap showing parameter relationships")
    print("      • Color-coded by correlation strength")
    print("      • Hover for exact correlation values")
    
    print("\n   ✅ DATA QUALITY TABLE:")
    print("      • Missing data percentage")
    print("      • Statistical summary (mean, std, min, max)")
    print("      • Quick data quality assessment")
    
    print("\n" + "="*70)
    print("🎯 RECOMMENDATION")
    print("="*70)
    print("\nYou now have TWO standalone HTML dashboards:")
    print("\n1. 📊 cc23_interactive_dashboard_23_6.html")
    print("   → Sequence analysis, disturbance detection, quality casting")
    print("\n2. 🔬 cc23_eda_dashboard_23_6.html (NEW!)")
    print("   → Parameter EDA, temporal coverage, distributions, correlations")
    print("\nBoth work offline, no server needed! 🎉")
    
else:
    print("\n⚠️ No data available. Run analysis pipeline first.")

## 🎯 **Databricks AI/BI Dashboard (Lakeview) - Native Solution**

### **Why This is Better Than Dash:**

✅ **Native Databricks integration** - No proxy issues
✅ **SQL-based** - Query Delta tables directly
✅ **Interactive filters** - Date range, parameters, disturbance types
✅ **Time series optimized** - Built for large datasets
✅ **Easy sharing** - Organization-wide with permissions
✅ **AI-assisted** - Databricks Assistant helps build visualizations
✅ **Production-ready** - Enterprise-grade reliability

---

### **📋 Implementation Plan:**

1. **Create Delta Tables** (from your analysis results)
2. **Create SQL Datasets** (queries for dashboard)
3. **Build Dashboard** (using Databricks UI)
4. **Add Filters** (date range, strand, disturbance type, parameters)
5. **Add Visualizations** (time series, correlations, distributions)

---

### **🎨 Dashboard Pages:**

* **Page 1: Overview** - KPIs, disturbance distribution, temporal coverage
* **Page 2: Time Series Explorer** - Multi-parameter plots with date range filter
* **Page 3: Sequence Analysis** - Sequence selector, drill-down views
* **Page 4: Parameter Correlations** - Scatter plots, quality casting analysis
* **Page 5: Data Quality** - Missing data, distributions, correlation matrix

In [0]:
# Create Delta tables from analysis results for Databricks Dashboard
# Renames columns to remove spaces (Delta requirement)

def clean_column_names(df_spark):
    """Replace spaces and special characters in column names for Delta compatibility."""
    for col in df_spark.columns:
        new_col = col.replace(" ", "_").replace("[", "").replace("]", "").replace("/", "_")
        if new_col != col:
            df_spark = df_spark.withColumnRenamed(col, new_col)
    return df_spark

if len(dashboard_results) > 0:
    print("\n" + "="*70)
    print("💾 CREATING DELTA TABLES FOR DATABRICKS DASHBOARD")
    print("="*70)
    
    # Use default Hive metastore
    database = "cc23_analysis"
    
    # Create database if it doesn't exist
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {database}")
    print(f"\n✅ Database: {database}")
    
    # Use timestamp in table names
    timestamp_suffix = CONFIG.timestamp
    
    tables_created = []
    
    for strand_id, result in dashboard_results.items():
        if not result.get('success', False):
            continue
        
        strand_name = result['strand_name']
        print(f"\n{'─'*70}")
        print(f"📊 Processing {strand_name}...")
        print(f"{'─'*70}")
        
        # ============================================================
        # TABLE 1: Raw Time Series Data
        # ============================================================
        print("\n1️⃣  Creating raw time series table...")
        
        df_raw = result['df_raw']
        df_raw_spark = spark.createDataFrame(df_raw)
        df_raw_spark = clean_column_names(df_raw_spark)
        
        table_name_raw = f"{database}.cc23_raw_timeseries_strand_{strand_id}_{timestamp_suffix}"
        
        df_raw_spark.write \
            .format("delta") \
            .mode("errorIfExists") \
            .saveAsTable(table_name_raw)
        
        row_count = spark.table(table_name_raw).count()
        print(f"   ✅ {table_name_raw}")
        print(f"      Rows: {row_count:,} | Columns: {len(df_raw_spark.columns)}")
        tables_created.append(table_name_raw)
        
        # ============================================================
        # TABLE 2: Sequence Statistics
        # ============================================================
        print("\n2️⃣  Creating sequence statistics table...")
        
        df_seq = result['df_seq']
        df_seq_spark = spark.createDataFrame(df_seq)
        df_seq_spark = clean_column_names(df_seq_spark)
        
        table_name_seq = f"{database}.cc23_sequence_stats_strand_{strand_id}_{timestamp_suffix}"
        
        df_seq_spark.write \
            .format("delta") \
            .mode("errorIfExists") \
            .saveAsTable(table_name_seq)
        
        row_count = spark.table(table_name_seq).count()
        print(f"   ✅ {table_name_seq}")
        print(f"      Rows: {row_count:,} | Columns: {len(df_seq_spark.columns)}")
        tables_created.append(table_name_seq)
        
        # ============================================================
        # TABLE 3: FC Mold Active Data
        # ============================================================
        print("\n3️⃣  Creating FC Mold active data table...")
        
        df_fc_mold = result['df_fc_mold']
        df_fc_mold_spark = spark.createDataFrame(df_fc_mold)
        df_fc_mold_spark = clean_column_names(df_fc_mold_spark)
        
        table_name_fc = f"{database}.cc23_fc_mold_active_strand_{strand_id}_{timestamp_suffix}"
        
        df_fc_mold_spark.write \
            .format("delta") \
            .mode("errorIfExists") \
            .saveAsTable(table_name_fc)
        
        row_count = spark.table(table_name_fc).count()
        print(f"   ✅ {table_name_fc}")
        print(f"      Rows: {row_count:,} | Columns: {len(df_fc_mold_spark.columns)}")
        tables_created.append(table_name_fc)
    
    print("\n" + "="*70)
    print("✅ ALL DELTA TABLES CREATED SUCCESSFULLY")
    print("="*70)
    
    print("\n📊 Tables Created:")
    for i, table in enumerate(tables_created, 1):
        print(f"   {i}. {table}")
    
    print("\n💡 Column names cleaned (spaces → underscores):")
    print("   'Mold Level' → 'Mold_Level'")
    print("   'Quality casting' → 'Quality_casting'")
    print("   'CASTING_SPEED_avg [m/min]' → 'CASTING_SPEED_avg_m_min'")
    
    print("\n" + "="*70)
    print("🎯 NEXT: CREATE DATABRICKS DASHBOARD")
    print("="*70)
    print("\n📍 Quick Start:")
    print("   1. Databricks UI → Dashboards")
    print("   2. Create Dashboard → 'CC23 Mold Level Analysis'")
    print("   3. Add dataset → Use SQL queries from next cell")
    print("   4. Update table names in queries with those above")
    print("   5. Add visualizations + filters")
    
else:
    print("\n⚠️ No data available. Run analysis pipeline first (cell 17).")

## 📝 **SQL Queries for Dashboard Datasets**

Copy these queries into your Databricks Dashboard datasets:

---

### **Dataset 1: Overview Metrics**

```sql
-- Overview KPIs and summary statistics
SELECT 
  COUNT(*) as total_sequences,
  COUNT(CASE WHEN `MOLD_LEVEL_std [mm]` <= 2.0 THEN 1 END) as stable_sequences,
  ROUND(100.0 * COUNT(CASE WHEN `MOLD_LEVEL_std [mm]` <= 2.0 THEN 1 END) / COUNT(*), 1) as stable_pct,
  ROUND(AVG(`MOLD_LEVEL_std [mm]`), 2) as mean_sigma,
  ROUND(STDDEV(`MOLD_LEVEL_std [mm]`), 2) as std_sigma,
  COUNT(CASE WHEN disturbance_type = 'Normal' THEN 1 END) as normal_sequences,
  ROUND(100.0 * COUNT(CASE WHEN disturbance_type = 'Normal' THEN 1 END) / COUNT(*), 1) as normal_pct,
  ROUND(AVG(`CASTING_SPEED_avg [m/min]`), 2) as avg_casting_speed,
  ROUND(AVG(`MOLD_WIDTH_avg [m]`), 3) as avg_mold_width
FROM main.cc23_analysis.cc23_sequence_stats_strand_23_6
```

---

### **Dataset 2: Disturbance Distribution**

```sql
-- Disturbance type counts for pie/bar charts
SELECT 
  disturbance_type,
  COUNT(*) as count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as percentage
FROM main.cc23_analysis.cc23_sequence_stats_strand_23_6
GROUP BY disturbance_type
ORDER BY count DESC
```

---

### **Dataset 3: Temporal Coverage**

```sql
-- Hourly data point counts (for gap visualization)
SELECT 
  DATE_TRUNC('hour', plainTimeStamp) as hour,
  COUNT(*) as data_points,
  CASE 
    WHEN COUNT(*) = 0 THEN 'No Data'
    WHEN COUNT(*) < 1800 THEN 'Sparse'  -- Less than 30 min of data
    WHEN COUNT(*) < 3000 THEN 'Good'
    ELSE 'Dense'
  END as coverage_level
FROM main.cc23_analysis.cc23_raw_timeseries_strand_23_6
GROUP BY DATE_TRUNC('hour', plainTimeStamp)
ORDER BY hour
```

---

### **Dataset 4: Time Series Data (with Parameters)**

```sql
-- Time series data with parameter filter
-- Add dashboard parameter: {{parameter_name}}
SELECT 
  plainTimeStamp as time,
  castingSpeed,
  moldWidth,
  SENImmersionDepth,
  `Mold Level`,
  `Mold Level Sensor Left`,
  `Mold Level Sensor Right`,
  `Argon Flow SEN`,
  `Argon Flow Stopper`,
  `EMBR Current AC Left Master`,
  `EMBR Current DC Left Master`,
  `EMBR Current DC Left Bottom`,
  `Quality casting`
FROM main.cc23_analysis.cc23_raw_timeseries_strand_23_6
WHERE plainTimeStamp BETWEEN :start_date AND :end_date
ORDER BY plainTimeStamp
LIMIT 50000  -- Prevent crashes with large data
```

---

### **Dataset 5: Sequence Details**

```sql
-- Individual sequence data for drill-down
-- Add dashboard parameter: {{sequence_name}}
SELECT 
  fc.plainTimeStamp as time,
  fc.`Mold Level`,
  fc.castingSpeed,
  fc.moldWidth,
  fc.SENImmersionDepth,
  seq.disturbance_type,
  seq.`MOLD_LEVEL_std [mm]` as sequence_sigma,
  seq.`Quality casting`
FROM main.cc23_analysis.cc23_fc_mold_active_strand_23_6 fc
INNER JOIN main.cc23_analysis.cc23_sequence_stats_strand_23_6 seq
  ON fc.plainTimeStamp BETWEEN seq.Seq_time_Start AND seq.Seq_time_End
WHERE seq.Seq_Name = :sequence_name
ORDER BY fc.plainTimeStamp
```

---

### **Dataset 6: Parameter Correlations**

```sql
-- Sequence-level correlations for scatter plots
SELECT 
  Seq_Name,
  `CASTING_SPEED_avg [m/min]` as casting_speed,
  `MOLD_WIDTH_avg [m]` as mold_width,
  `SEN_avg [mm]` as sen_depth,
  `ArFlow_avg [NL/min]` as argon_flow,
  `MOLD_LEVEL_std [mm]` as mold_level_sigma,
  disturbance_type,
  `Quality casting` as quality_casting,
  Seq_duration_min as duration
FROM main.cc23_analysis.cc23_sequence_stats_strand_23_6
WHERE disturbance_type = :disturbance_filter OR :disturbance_filter = 'All'
```

---

### **Dataset 7: Quality Casting Analysis**

```sql
-- Mold level stability by quality grade
SELECT 
  `Quality casting` as quality_grade,
  `MOLD_LEVEL_std [mm]` as mold_level_sigma,
  `CASTING_SPEED_avg [m/min]` as casting_speed,
  Seq_Name,
  disturbance_type
FROM main.cc23_analysis.cc23_sequence_stats_strand_23_6
WHERE `Quality casting` IS NOT NULL
```

In [0]:
# Create Delta tables from analysis results

if len(dashboard_results) > 0:
    print("\n" + "="*70)
    print("💾 CREATING DELTA TABLES FOR DATABRICKS DASHBOARD")
    print("="*70)
    
    # Define catalog and schema
    catalog = "main"
    schema = "cc23_analysis"
    
    # Create schema if it doesn't exist
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    print(f"\n✅ Schema: {catalog}.{schema}")
    
    tables_created = []
    
    for strand_id, result in dashboard_results.items():
        if not result.get('success', False):
            continue
        
        strand_name = result['strand_name']
        print(f"\n{'─'*70}")
        print(f"📊 Processing {strand_name}...")
        print(f"{'─'*70}")
        
        # ============================================================
        # TABLE 1: Raw Time Series Data
        # ============================================================
        print("\n1️⃣  Creating raw time series table...")
        
        df_raw = result['df_raw']
        df_raw_spark = spark.createDataFrame(df_raw)
        
        table_name_raw = f"{catalog}.{schema}.cc23_raw_timeseries_strand_{strand_id}"
        
        df_raw_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name_raw)
        
        row_count = spark.table(table_name_raw).count()
        print(f"   ✅ {table_name_raw}")
        print(f"      Rows: {row_count:,} | Columns: {len(df_raw.columns)}")
        tables_created.append(table_name_raw)
        
        # ============================================================
        # TABLE 2: Sequence Statistics
        # ============================================================
        print("\n2️⃣  Creating sequence statistics table...")
        
        df_seq = result['df_seq']
        df_seq_spark = spark.createDataFrame(df_seq)
        
        table_name_seq = f"{catalog}.{schema}.cc23_sequence_stats_strand_{strand_id}"
        
        df_seq_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name_seq)
        
        row_count = spark.table(table_name_seq).count()
        print(f"   ✅ {table_name_seq}")
        print(f"      Rows: {row_count:,} | Columns: {len(df_seq.columns)}")
        tables_created.append(table_name_seq)
        
        # ============================================================
        # TABLE 3: FC Mold Active Data
        # ============================================================
        print("\n3️⃣  Creating FC Mold active data table...")
        
        df_fc_mold = result['df_fc_mold']
        df_fc_mold_spark = spark.createDataFrame(df_fc_mold)
        
        table_name_fc = f"{catalog}.{schema}.cc23_fc_mold_active_strand_{strand_id}"
        
        df_fc_mold_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name_fc)
        
        row_count = spark.table(table_name_fc).count()
        print(f"   ✅ {table_name_fc}")
        print(f"      Rows: {row_count:,} | Columns: {len(df_fc_mold.columns)}")
        tables_created.append(table_name_fc)
    
    print("\n" + "="*70)
    print("✅ ALL DELTA TABLES CREATED SUCCESSFULLY")
    print("="*70)
    
    print("\n📊 Tables Created:")
    for i, table in enumerate(tables_created, 1):
        print(f"   {i}. {table}")
    
    print("\n🎯 Next Steps:")
    print("   1. Go to Databricks UI → Dashboards")
    print("   2. Click 'Create Dashboard'")
    print("   3. Add datasets using the SQL queries from the cell above")
    print("   4. Add visualizations and filters")
    print("   5. Publish and share!")
    
else:
    print("\n⚠️ No data available. Run analysis pipeline first (cell 17).")

---
---
# PART 2: Step-by-Step Exploratory Analysis (Strand 23-6)

> **For newcomers:** This section walks through the analysis manually, step by step.  
> If you just want results, use **Path A** (cells 2-18) from the class-based pipeline above.  
> This section exists for **understanding the logic** and **debugging**.

**What follows:**
1. Data loading and schema inspection (cells 58-82)
2. Metadata join and data quality checks (cells 83-91)
3. Unit conversion and temporal gap analysis (cells 92-106)
4. Visualization of parameter distributions and time series (cells 107-127)
5. Sequence identification and stable-period statistics (cells 128-149)
6. Disturbance detection and classification (cells 150-183)

## 2.1 Project Goals and Data Sources

**Goal**
- Identify stable casting sequences
- Remove sensor artifacts
- Relate mold level stability to process parameters

**Data**

CC23 has two strands: 23_5 and 23_6.
- dtExpert , data experts logged with a freq of 1Hz, it includes currents of EMBR parts, and casting parameters
- boExpert, BreakOut experts, logged with a freq of 2Hz, it consists of the raw FBG temperature measurements and some casting parameters

**Last updated**: 2025-06-xx


In [0]:
# Required for scatter density plots (high point counts)
%pip install mpl-scatter-density
%pip install astropy

In [0]:
# ---------------------------------------------------------------------------
# Path B Imports: reuse modules from src/ (single source of truth)
# ---------------------------------------------------------------------------
import sys
import os

# Add project root to Python path so we can import from src/
project_root = "/Workspace/Users/dieudonne.nkulikiyimfura@se.abb.com/TATAIjmulden_FCMoldG5"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- Core data functions (from src/) ---
from src.config import AnalysisConfig, StrandConfig, CONFIG, STRAND_CONFIGS, METADATA_PATH
from src.data_loading import (
    add_plain_timestamp,
    get_expert_files,
    load_expert_files,
    convert_units,
    StrandDataLoader,
)
from src.sequence_analysis import (
    custom_rounding,
    segment_by_time_gaps,
    identify_sequences,
    identify_sequences_segmented,
    SequenceAnalyzer,
)
from src.disturbance_detection import (
    detect_excursion_event_robust,
    detect_slow_drift_event,
    detect_transient_bump_dynamic,
    detect_high_variability_event,
)

# --- Standard scientific libraries ---
import pandas as pd
import numpy as np
import time
from scipy.ndimage import median_filter
from pathlib import Path

# --- Visualization ---
import matplotlib.pyplot as plt
import mpl_scatter_density  # adds projection='scatter_density'
from matplotlib.colors import LinearSegmentedColormap
from astropy.visualization import LogStretch
from astropy.visualization.mpl_normalize import ImageNormalize
import plotly.express as px

# --- Spark (for ad-hoc queries in exploration cells) ---
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType, LongType, IntegerType, DoubleType, FloatType

# Visualization normalization for scatter density plots
norm = ImageNormalize(vmin=0., vmax=1000, stretch=LogStretch())

print("Path B imports loaded from src/ modules (single source of truth)")
print(f"  CONFIG: {CONFIG}")
print(f"  Strands available: {list(STRAND_CONFIGS.keys())}")



In [0]:

%fs ls dbfs:/FileStore/TATA_IJmuiden_CC23/data/


In [0]:
# Set strand path and discover boExpert/dtExpert files
strand_23_6_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/strand_6"

bo_files_23_6, dt_files_23_6 = get_expert_files(strand_23_6_path)


In [0]:
# Quick check: which dtExpert files were found?
dt_files_23_6

## Load boExpert dtExpert files

In [0]:
# Load boExpert and dtExpert into Spark DataFrames and verify timestamps

df_boExpert_23_6 = load_expert_files(bo_files_23_6)
df_dtExpert_23_6 = load_expert_files(dt_files_23_6)

# Sanity checks
from pyspark.sql import functions as F

df_boExpert_23_6.select(
    F.count("*").alias("bo_total"),
    F.count("plainTimeStamp").alias("bo_plainTimeStamp_not_null")
).show()

df_dtExpert_23_6.select(
    F.count("*").alias("dt_total"),
    F.count("plainTimeStamp").alias("dt_plainTimeStamp_not_null")
).show()

df_boExpert_23_6.select("plainTimeStamp").show(5, False)
df_dtExpert_23_6.select("plainTimeStamp").show(5, False)


In [0]:
# Inspect column types (numeric vs string timestamps)
df_boExpert_23_6.printSchema()


In [0]:
df_boExpert_23_6.count()

In [0]:
df_boExpert_23_6.display()

## Join using plainTimeStamp

In [0]:
# Aggregate boExpert from 2Hz to 1Hz: mean for numerics, first for categoricals
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

key_col = "plainTimeStamp"

# Identify numeric vs non-numeric columns (excluding the key)
numeric_cols = [
    f.name for f in df_boExpert_23_6.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != key_col
]

non_numeric_cols = [
    f.name for f in df_boExpert_23_6.schema.fields
    if not isinstance(f.dataType, NumericType) and f.name != key_col
]

print("Numeric columns to average:", numeric_cols)
print("Non-numeric columns (take first):", non_numeric_cols)

# Build aggregation expressions
agg_exprs = (
    [F.avg(c).alias(c) for c in numeric_cols] +
    [F.first(c, ignorenulls=True).alias(c) for c in non_numeric_cols]
)

# Aggregate bo on plainTimeStamp
df_boExpert_23_6_agg = (
    df_boExpert_23_6
    .groupBy(key_col)
    .agg(*agg_exprs)
)

# Sanity check: should now equal bo_distinct
print("df_boExpert_23_6_agg rows:", df_boExpert_23_6_agg.count())


In [0]:
from pyspark.sql import functions as F

# What columns exist and their types?
print(df_boExpert_23_6.dtypes)

# Non-null counts
df_boExpert_23_6.select(
    F.count("*").alias("bo_total"),
    F.count("plainTimeStamp").alias("plainTimeStamp_not_null"),
    F.count(F.when(F.col("plainTimeStamp").isNull(), 1)).alias("plainTimeStamp_nulls")
).show()

# Peek at *all* potential time columns
time_cols = [c for c, t in df_boExpert_23_6.dtypes if 'time' in c.lower() or 'timestamp' in c.lower()]
print("Time-like columns:", time_cols)
df_boExpert_23_6.select(*time_cols).show(20, truncate=False)


In [0]:
df_boExpert_23_6.count()

In [0]:
df_boExpert_23_6_agg.count()

In [0]:
df_dtExpert_23_6.display()

In [0]:
# Inner join on plainTimeStamp - only timestamps present in BOTH sources are kept
df_23_6_spark = (
    df_boExpert_23_6_agg.alias("bo")
    .join(df_dtExpert_23_6.alias("dt"), on=key_col, how="inner")
)

print("joined rows:", df_23_6_spark.count())


df_23_6_spark.cache()
df_23_6_spark.count()


In [0]:
print("bo rows:", df_boExpert_23_6.count())
print("dt rows:", df_dtExpert_23_6.count())
print("joined rows:", df_23_6_spark.count())


In [0]:
from pyspark.sql import functions as F

bo_total = df_boExpert_23_6.count()
dt_total = df_dtExpert_23_6.count()

bo_distinct = df_boExpert_23_6.select("plainTimeStamp").distinct().count()
dt_distinct = df_dtExpert_23_6.select("plainTimeStamp").distinct().count()

print("bo_total     :", bo_total)
print("bo_distinct  :", bo_distinct)
print("bo duplicates:", bo_total - bo_distinct)

print("dt_total     :", dt_total)
print("dt_distinct  :", dt_distinct)
print("dt duplicates:", dt_total - dt_distinct)


In [0]:
df_23_6_spark.count()

In [0]:
# Load casting quality metadata (semicolon-delimited CSV with temporal boundaries)
metadata_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv"
df_metadata_spark = spark.read.csv('dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv', header=True, inferSchema=True, sep=";")
display(df_metadata_spark)


In [0]:
from pyspark.sql import functions as F

#metadata_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv"
#df_metadata_spark = spark.read.csv('dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv', header=True, inferSchema=True, sep=";")
#display(df_metadata_spark)

df_23_6_spark = df_23_6_spark.withColumn(
    "PlainTimeStamp",
    F.col("PlainTimeStamp").cast("timestamp")
)

df_metadata_spark = df_metadata_spark \
    .withColumn("Datetime start first heat", F.col("Datetime start first heat").cast("timestamp")) \
    .withColumn("Datetime start last heat",  F.col("Datetime start last heat").cast("timestamp"))

join_condition = (
    (df_23_6_spark["PlainTimeStamp"] >= df_metadata_spark["Datetime start first heat"]) &
    (df_23_6_spark["PlainTimeStamp"] <= df_metadata_spark["Datetime start last heat"])
)

df_23_6_spark = (
    df_23_6_spark
    .join(
        df_metadata_spark.select(
            "Datetime start first heat",
            "Datetime start last heat",
            "Quality casting"
        ),
        on=join_condition,
        how="left"
    )
)



In [0]:
display(df_23_6_spark)

In [0]:
from pyspark.sql import functions as F

df_metadata_spark.select(
    "Datetime start first heat",
    "Datetime start last heat",
    F.col("Datetime start first heat").cast("timestamp").alias("first_cast"),
    F.col("Datetime start last heat").cast("timestamp").alias("last_cast")
).show(20, truncate=False)


In [0]:
# Strip timezone offset from metadata timestamps before joining
from pyspark.sql import functions as F

def parse_ts_with_tz(colname):
    # example: "2025-05-01 21:56:21+02:00" -> "2025-05-01 21:56:21"
    return F.to_timestamp(
        F.regexp_replace(F.col(colname), r"\+\d{2}:\d{2}$", ""),
        "yyyy-MM-dd HH:mm:ss"
    )

df_metadata_fixed = (
    df_metadata_spark
    .withColumn("ts_start", parse_ts_with_tz("Datetime start first heat"))
    .withColumn("ts_end",   parse_ts_with_tz("Datetime start last heat"))
)


## 2.2 Data Quality and Metadata Matching

Validate that sensor timestamps align with casting metadata records:
* Parse metadata timestamps (handle timezone offsets)
* Range-join sensor data to casting periods (filter by Strand ID to prevent duplicates)
* Investigate NULL `Quality casting` records (expected for inter-cast gaps)
* Visualise temporal coverage and identify significant data gaps

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# 1) Convert PlainTimeStamp (UTC) into local time to match metadata
df_23_6_local = df_23_6_spark.withColumn(
    "PlainTimeStamp_local",
    F.from_utc_timestamp(F.col("PlainTimeStamp"), "Europe/Amsterdam")
)

# Drop "Quality casting" if it exists to avoid duplicate columns in the join
if "Quality casting" in df_23_6_local.columns:
    df_23_6_local = df_23_6_local.drop("Quality casting")

# 2) Parse metadata timestamps and FILTER FOR STRAND 6 ONLY
df_metadata_fixed = (
    df_metadata_spark
    .filter(F.col("Strand ID") == 6)  # ← CRITICAL: Filter for Strand 6 only
    .withColumn("ts_start", F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ts_end",   F.to_timestamp("Datetime start last heat",  "yyyy-MM-dd HH:mm:ss"))
    .filter(F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull())
)

print(f"Metadata records for Strand 6: {df_metadata_fixed.count()}")

# 3) Range join condition in SAME timezone reference (local)
join_condition = (
    (df_23_6_local["PlainTimeStamp_local"] >= df_metadata_fixed["ts_start"]) &
    (df_23_6_local["PlainTimeStamp_local"] <= df_metadata_fixed["ts_end"])
)

# 4) Join (broadcast metadata if small)
df_23_6_joined = (
    df_23_6_local
    .join(
        broadcast(df_metadata_fixed.select("ts_start", "ts_end", "Quality casting", "Casting ID")),
        on=join_condition,
        how="left"
    )
)

# 5) (Optional) drop helper cols
df_23_6_joined = df_23_6_joined.drop("ts_start", "ts_end")

print(f"\nJoin results:")
df_23_6_joined.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("Quality casting").isNotNull().cast("int")).alias("matched_rows"),
    F.countDistinct("Casting ID").alias("unique_castings")
).show()

In [0]:
# Check how many rows matched a casting period
df_23_6_joined.select(
    F.count("*").alias("rows"),
    F.sum(F.col("Quality casting").isNotNull().cast("int")).alias("matched_rows")
).show()

df_23_6_joined.select("PlainTimeStamp", "PlainTimeStamp_local", "Quality casting") \
    .where(F.col("Quality casting").isNotNull()) \
    .show(20, truncate=False)


In [0]:
# What quality labels exist and how frequent are they?
%python
display(
    df_23_6_spark.groupBy(df_23_6_spark["Quality casting"]).count()
)

In [0]:
from pyspark.sql import functions as F

# Get statistics on null vs non-null Quality casting
quality_stats = df_23_6_spark.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("Quality casting").isNull(), 1).otherwise(0)).alias("null_quality"),
    F.sum(F.when(F.col("Quality casting").isNotNull(), 1).otherwise(0)).alias("non_null_quality")
).collect()[0]

print("="*70)
print("QUALITY CASTING DISTRIBUTION ANALYSIS")
print("="*70)
print(f"Total rows: {quality_stats['total_rows']:,}")
print(f"Null Quality casting: {quality_stats['null_quality']:,} ({100*quality_stats['null_quality']/quality_stats['total_rows']:.2f}%)")
print(f"Non-null Quality casting: {quality_stats['non_null_quality']:,} ({100*quality_stats['non_null_quality']/quality_stats['total_rows']:.2f}%)")
print("="*70)

# Get time range for null vs non-null
print("\nTIME RANGE COMPARISON:")
print("-"*70)

# Time range for NULL quality casting
null_time_range = df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    F.min("PlainTimeStamp").alias("first_null"),
    F.max("PlainTimeStamp").alias("last_null")
).collect()[0]

print(f"\nNull Quality casting time range:")
print(f"  First: {null_time_range['first_null']}")
print(f"  Last: {null_time_range['last_null']}")

# Time range for NON-NULL quality casting
non_null_time_range = df_23_6_spark.filter(F.col("Quality casting").isNotNull()).select(
    F.min("PlainTimeStamp").alias("first_non_null"),
    F.max("PlainTimeStamp").alias("last_non_null")
).collect()[0]

print(f"\nNon-null Quality casting time range:")
print(f"  First: {non_null_time_range['first_non_null']}")
print(f"  Last: {non_null_time_range['last_non_null']}")
print("="*70)

In [0]:
from pyspark.sql import functions as F

# Get metadata casting period ranges
metadata_range = df_metadata_spark.select(
    F.min(F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss")).alias("first_casting_start"),
    F.max(F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss")).alias("last_casting_end")
).collect()[0]

print("="*70)
print("METADATA CASTING PERIODS")
print("="*70)
print(f"First casting starts: {metadata_range['first_casting_start']}")
print(f"Last casting ends: {metadata_range['last_casting_end']}")
print("="*70)

# Get overall data time range
data_range = df_23_6_spark.select(
    F.min("PlainTimeStamp").alias("data_start"),
    F.max("PlainTimeStamp").alias("data_end")
).collect()[0]

print("\nOVERALL DATA TIME RANGE")
print("="*70)
print(f"Data starts: {data_range['data_start']}")
print(f"Data ends: {data_range['data_end']}")
print("="*70)

# Check if data extends beyond casting periods
print("\nANALYSIS:")
print("-"*70)
if data_range['data_start'] < metadata_range['first_casting_start']:
    print(f"⚠️  Data starts BEFORE first casting period")
    print(f"   Gap: {(metadata_range['first_casting_start'] - data_range['data_start']).total_seconds() / 3600:.2f} hours")
else:
    print(f"✓ Data starts within or after casting periods")

if data_range['data_end'] > metadata_range['last_casting_end']:
    print(f"⚠️  Data ends AFTER last casting period")
    print(f"   Gap: {(data_range['data_end'] - metadata_range['last_casting_end']).total_seconds() / 3600:.2f} hours")
else:
    print(f"✓ Data ends within or before casting periods")
print("="*70)

In [0]:
from pyspark.sql import functions as F, Window

# Get casting periods sorted by time
df_casting_periods_sorted = df_metadata_spark.select(
    F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss").alias("ts_start"),
    F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss").alias("ts_end"),
    "Casting ID",
    "Strand ID"
).filter(
    F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull()
).orderBy("ts_start")

# Calculate gaps between consecutive casting periods
window_spec = Window.orderBy("ts_start")

df_casting_gaps = df_casting_periods_sorted.withColumn(
    "prev_end",
    F.lag("ts_end").over(window_spec)
).withColumn(
    "gap_hours",
    (F.unix_timestamp("ts_start") - F.unix_timestamp("prev_end")) / 3600
).filter(
    F.col("gap_hours").isNotNull() & (F.col("gap_hours") > 0)
)

print("="*70)
print("GAPS BETWEEN CASTING PERIODS")
print("="*70)
print(f"Number of gaps between castings: {df_casting_gaps.count()}")
print("\nTop 10 largest gaps:")
df_casting_gaps.select(
    "prev_end",
    "ts_start",
    "gap_hours",
    "Casting ID",
    "Strand ID"
).orderBy(F.desc("gap_hours")).show(10, truncate=False)

# Calculate total gap time
total_gap_hours = df_casting_gaps.select(F.sum("gap_hours")).collect()[0][0]
print(f"\nTotal gap time between castings: {total_gap_hours:.2f} hours")
print("="*70)

In [0]:
from pyspark.sql import functions as F

# Sample some null Quality casting records to see their timestamps
print("="*70)
print("SAMPLE RECORDS WITH NULL QUALITY CASTING")
print("="*70)
print("\nFirst 20 records with null Quality casting:")
df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    "PlainTimeStamp",
    "Quality casting"
).orderBy("PlainTimeStamp").show(20, truncate=False)

print("\nLast 20 records with null Quality casting:")
df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    "PlainTimeStamp",
    "Quality casting"
).orderBy(F.desc("PlainTimeStamp")).show(20, truncate=False)
print("="*70)

## Analysis Summary: Null Quality Casting Values

### Key Findings:

**1. Distribution:**
* **45.74%** of measurements (324,079 rows) have **NULL** Quality casting
* **54.26%** of measurements (384,376 rows) have **non-null** Quality casting

**2. Temporal Analysis:**

**Null Quality Casting Time Range:**
* First: `2025-04-14 01:09:46`
* Last: `2025-05-16 05:36:25`
* Spans throughout the entire dataset

**Non-null Quality Casting Time Range:**
* First: `2025-04-13 22:59:26`
* Last: `2025-04-30 10:09:43`
* Limited to specific casting periods

**3. Root Cause:**

⚠️ **YES - Null values are from measurements OUTSIDE casting periods:**

* **Metadata casting periods:** April 13, 21:14:33 → May 5, 05:53:52
* **Actual data range:** April 13, 22:59:26 → **May 16, 05:36:25**
* **Data extends 263.71 hours (11 days) AFTER the last casting period**

**4. Explanation:**

The null `Quality casting` values occur because:

1. **Between casting periods:** 26 gaps totaling 410.73 hours between consecutive castings
   - Largest gap: 78.6 hours (April 18 → April 21)
   - During these gaps, the MEX system continues recording data but no casting is active

2. **After last casting:** 263.71 hours of data after May 5, 05:53:52
   - System continues logging even though no casting metadata exists
   - This is the majority of null values

3. **Before first casting:** Minimal (data starts within casting periods)

### Recommendation:

For your **mold level fluctuation analysis**, you should:
* **Filter out null Quality casting rows** - they represent non-production periods
* **Focus on sequences with valid Quality casting** - these are actual casting operations
* **Use Quality casting as a grouping variable** for sequence-based analysis

In [0]:
import plotly.graph_objects as go
import pandas as pd
from pyspark.sql import functions as F

# Sample data for visualization (to avoid memory issues)
sample_rate = max(1, df_23_6_spark.count() // 10000)

df_sample = df_23_6_spark.sample(fraction=1.0/sample_rate, seed=42).select(
    "PlainTimeStamp",
    F.when(F.col("Quality casting").isNull(), "No Casting (NULL)").otherwise("Active Casting").alias("status"),
    "Quality casting"
).orderBy("PlainTimeStamp").toPandas()

# Create figure
fig = go.Figure()

# Add traces for null and non-null
for status in df_sample['status'].unique():
    df_status = df_sample[df_sample['status'] == status]
    color = 'red' if status == 'No Casting (NULL)' else 'green'
    
    fig.add_trace(go.Scatter(
        x=df_status['PlainTimeStamp'],
        y=[1] * len(df_status),
        mode='markers',
        marker=dict(color=color, size=3, opacity=0.5),
        name=status,
        hovertemplate='%{x}<br>Status: ' + status + '<extra></extra>'
    ))

# Add metadata casting period boundaries
metadata_periods = df_metadata_spark.select(
    F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss").alias("start"),
    F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss").alias("end"),
    "Casting ID"
).filter(F.col("start").isNotNull()).toPandas()

for idx, row in metadata_periods.iterrows():
    fig.add_vrect(
        x0=row['start'], x1=row['end'],
        fillcolor='lightblue', opacity=0.1,
        layer="below", line_width=0
    )

fig.update_layout(
    title=dict(
        text='Temporal Distribution: Null vs Non-Null Quality Casting',
        font=dict(size=16, color='darkblue')
    ),
    xaxis_title='Timeline',
    yaxis_title='Data Presence',
    height=500,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.15,
        xanchor="center",
        x=0.5
    ),
    yaxis=dict(showticklabels=False, range=[0.5, 1.5]),
    hovermode='closest'
)

fig.show()

print("\n" + "="*70)
print("VISUALIZATION EXPLANATION")
print("="*70)
print("• GREEN dots: Measurements during active casting (with Quality casting)")
print("• RED dots: Measurements outside casting periods (NULL Quality casting)")
print("• Light blue bands: Metadata-defined casting periods")
print("\nNotice: Red dots appear in gaps between blue bands and after the last band")
print("="*70)

## 2.3 Unit Conversion and Feature Engineering

Convert raw sensor units to physical quantities:
* castingSpeed: m/s → m/min (×60)
* Mold Level: m → mm (×1000)
* Argon flows: m³/s → L/min (×60000)

Then derive temporal features (time gaps, casting periods).

In [0]:
# Apply unit conversions: m/s->m/min, m->mm, m³/s->L/min
df_23_6_spark_unitConverted = convert_units(df_23_6_spark)
#df_23_6_spark_unitConverted = convert_argon_flow_to_l_per_min(df_23_6_spark_unitConverted)
display(df_23_6_spark_unitConverted)

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Sort by timestamp and calculate time difference between consecutive rows
window_spec = Window.orderBy("PlainTimeStamp")

df_time_gaps = (
    df_23_6_spark_unitConverted
    .select("PlainTimeStamp")
    .withColumn("prev_timestamp", F.lag("PlainTimeStamp").over(window_spec))
    .withColumn(
        "time_diff_seconds",
        F.unix_timestamp("PlainTimeStamp") - F.unix_timestamp("prev_timestamp")
    )
    .filter(F.col("time_diff_seconds").isNotNull())
)

# Show statistics about time gaps
df_time_gaps.select(
    F.min("time_diff_seconds").alias("min_gap_sec"),
    F.max("time_diff_seconds").alias("max_gap_sec"),
    F.avg("time_diff_seconds").alias("avg_gap_sec"),
    F.expr("percentile(time_diff_seconds, 0.5)").alias("median_gap_sec"),
    F.expr("percentile(time_diff_seconds, 0.95)").alias("p95_gap_sec")
).show()

In [0]:
# Define threshold for "significant" gap (e.g., > 5 minutes = 300 seconds)
threshold_seconds = 300

df_significant_gaps = (
    df_time_gaps
    .filter(F.col("time_diff_seconds") > threshold_seconds)
    .withColumn("time_diff_minutes", F.col("time_diff_seconds") / 60)
    .withColumn("time_diff_hours", F.col("time_diff_seconds") / 3600)
    .orderBy(F.desc("time_diff_seconds"))
)

print(f"Found {df_significant_gaps.count()} gaps larger than {threshold_seconds} seconds ({threshold_seconds/60} minutes)")
display(df_significant_gaps.select(
    "prev_timestamp",
    "PlainTimeStamp",
    "time_diff_seconds",
    "time_diff_minutes",
    "time_diff_hours"
).limit(20))

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Convert to pandas for plotting
df_gaps_pd = df_time_gaps.select("PlainTimeStamp", "time_diff_seconds").toPandas()

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Time Series Gap Analysis', fontsize=16, fontweight='bold')

# 1. Histogram of time gaps (log scale)
ax1 = axes[0, 0]
ax1.hist(df_gaps_pd['time_diff_seconds'], bins=100, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Time Gap (seconds)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Time Gaps')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# 2. Time gaps over time
ax2 = axes[0, 1]
ax2.scatter(df_gaps_pd['PlainTimeStamp'], df_gaps_pd['time_diff_seconds'], 
            alpha=0.5, s=1)
ax2.set_xlabel('Timestamp')
ax2.set_ylabel('Time Gap (seconds)')
ax2.set_title('Time Gaps Over Time')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Cumulative distribution of gaps
ax3 = axes[1, 0]
sorted_gaps = np.sort(df_gaps_pd['time_diff_seconds'])
cumulative = np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps) * 100
ax3.plot(sorted_gaps, cumulative, linewidth=2)
ax3.set_xlabel('Time Gap (seconds)')
ax3.set_ylabel('Cumulative Percentage (%)')
ax3.set_title('Cumulative Distribution of Time Gaps')
ax3.set_xscale('log')
ax3.grid(True, alpha=0.3)
ax3.axhline(y=95, color='r', linestyle='--', label='95th percentile')
ax3.legend()

# 4. Box plot of time gaps (filtered to show detail)
ax4 = axes[1, 1]
# Filter out extreme outliers for better visualization
q99 = df_gaps_pd['time_diff_seconds'].quantile(0.99)
filtered_gaps = df_gaps_pd[df_gaps_pd['time_diff_seconds'] <= q99]['time_diff_seconds']
ax4.boxplot(filtered_gaps, vert=True)
ax4.set_ylabel('Time Gap (seconds)')
ax4.set_title(f'Box Plot of Time Gaps (up to 99th percentile: {q99:.1f}s)')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/time_gaps_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal records analyzed: {len(df_gaps_pd):,}")
print(f"Mean gap: {df_gaps_pd['time_diff_seconds'].mean():.2f} seconds")
print(f"Median gap: {df_gaps_pd['time_diff_seconds'].median():.2f} seconds")
print(f"Max gap: {df_gaps_pd['time_diff_seconds'].max():.2f} seconds ({df_gaps_pd['time_diff_seconds'].max()/3600:.2f} hours)")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# Get significant gaps for visualization
df_sig_gaps_pd = df_significant_gaps.select(
    "prev_timestamp", 
    "PlainTimeStamp", 
    "time_diff_hours"
).toPandas()

# Create timeline visualization
fig, ax = plt.subplots(figsize=(16, 6))

# Plot the continuous timeline
if len(df_gaps_pd) > 0:
    # Sample data points for timeline (every 100th point to avoid overcrowding)
    sample_indices = range(0, len(df_gaps_pd), max(1, len(df_gaps_pd) // 1000))
    timeline_sample = df_gaps_pd.iloc[sample_indices]
    
    ax.scatter(timeline_sample['PlainTimeStamp'], 
               [1] * len(timeline_sample), 
               alpha=0.3, s=10, c='blue', label='Data points')

# Mark significant gaps
if len(df_sig_gaps_pd) > 0:
    for idx, row in df_sig_gaps_pd.iterrows():
        ax.axvspan(row['prev_timestamp'], row['PlainTimeStamp'], 
                   alpha=0.3, color='red')
        # Add text label for large gaps
        if row['time_diff_hours'] > 1:
            mid_time = row['prev_timestamp'] + (row['PlainTimeStamp'] - row['prev_timestamp']) / 2
            ax.text(mid_time, 1.1, f"{row['time_diff_hours']:.1f}h", 
                   rotation=90, ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Timestamp', fontsize=12)
ax.set_ylabel('Data Presence', fontsize=12)
ax.set_title(f'Time Series Timeline with Gaps > {threshold_seconds/60} minutes (Red Regions)', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.5)
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45, ha='right')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', alpha=0.3, label='Data points'),
    Patch(facecolor='red', alpha=0.3, label=f'Gaps > {threshold_seconds/60} min')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
fig.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/timeline_with_gaps.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization shows {len(df_sig_gaps_pd)} significant gaps in the timeline")

In [0]:
from pyspark.sql import functions as F

# Parse metadata timestamps and filter valid records
df_casting_periods = (
    df_metadata_spark
    .withColumn("ts_start", F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ts_end", F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss"))
    .filter(F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull())
    .select(
        "Casting ID",
        "Strand ID",
        "ts_start",
        "ts_end",
        "Quality casting",
        "Heat Count"
    )
    .orderBy("ts_start")
)

print(f"Total casting periods: {df_casting_periods.count()}")
display(df_casting_periods.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Join gaps with casting periods to classify them
df_gaps_classified = df_significant_gaps.alias("gaps").join(
    broadcast(df_casting_periods.alias("cast")),
    (
        (F.col("gaps.prev_timestamp") >= F.col("cast.ts_start")) &
        (F.col("gaps.prev_timestamp") <= F.col("cast.ts_end")) &
        (F.col("gaps.PlainTimeStamp") >= F.col("cast.ts_start")) &
        (F.col("gaps.PlainTimeStamp") <= F.col("cast.ts_end"))
    ),
    how="left"
).select(
    F.col("gaps.prev_timestamp"),
    F.col("gaps.PlainTimeStamp"),
    F.col("gaps.time_diff_hours"),
    F.col("cast.Casting ID").alias("casting_id"),
    F.col("cast.Strand ID").alias("strand_id"),
    F.col("cast.Quality casting").alias("quality"),
    F.when(F.col("cast.Casting ID").isNotNull(), "Within Casting")
     .otherwise("Between Castings").alias("gap_type")
)

print("\nGap Classification:")
df_gaps_classified.groupBy("gap_type").agg(
    F.count("*").alias("count"),
    F.sum("time_diff_hours").alias("total_hours"),
    F.avg("time_diff_hours").alias("avg_hours")
).show()

print("\nGaps WITHIN casting periods (potential issues):")
display(
    df_gaps_classified
    .filter(F.col("gap_type") == "Within Casting")
    .orderBy(F.desc("time_diff_hours"))
)

In [0]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from datetime import datetime
import os

# Convert casting periods to pandas
df_casting_pd = df_casting_periods.toPandas()
df_gaps_class_pd = df_gaps_classified.toPandas()

# Filter for Strand 6 only (Strand 5 data not loaded yet)
df_casting_pd = df_casting_pd[df_casting_pd['Strand ID'] == 5].reset_index(drop=True)

# ---- helpers to restrict events to Strand-6 casting windows ----
# Build IntervalIndex of Strand-6 casting periods
casting_intervals = pd.IntervalIndex.from_arrays(
    df_casting_pd["ts_start"],
    df_casting_pd["ts_end"],
    closed="both"
)

def in_any_casting_window(ts_series):
    """Return boolean mask: timestamp is inside any Strand-6 casting interval."""
    # get_indexer gives -1 when not contained in any interval
    return casting_intervals.get_indexer(ts_series) >= 0

def gap_overlaps_any_casting(row):
    """True if [prev_timestamp, PlainTimeStamp] overlaps ANY Strand-6 casting window."""
    a = row["prev_timestamp"]
    b = row["PlainTimeStamp"]
    if pd.isna(a) or pd.isna(b):
        return False
    # overlap test with all intervals: overlap exists if max(starts) < min(ends)
    # here do it simply: does either end fall inside, or does interval cover a casting boundary
    if in_any_casting_window(pd.Series([a]))[0] or in_any_casting_window(pd.Series([b]))[0]:
        return True
    # also catch gaps spanning across a casting window without endpoints inside
    return ((df_casting_pd["ts_start"] <= b) & (df_casting_pd["ts_end"] >= a)).any()




# Ensure timestamp columns are datetime objects (not Timedelta)
for col in ['ts_start', 'ts_end']:
    if col in df_casting_pd.columns:
        df_casting_pd[col] = pd.to_datetime(df_casting_pd[col])

for col in ['prev_timestamp', 'PlainTimeStamp']:
    if col in df_gaps_class_pd.columns:
        df_gaps_class_pd[col] = pd.to_datetime(df_gaps_class_pd[col])

if 'PlainTimeStamp' in df_gaps_pd.columns:
    df_gaps_pd['PlainTimeStamp'] = pd.to_datetime(df_gaps_pd['PlainTimeStamp'])

# Create output directory if it doesn't exist
output_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/'
os.makedirs(output_dir, exist_ok=True)

# ========================================================================
# FIGURE 1: Casting Periods as Horizontal Timeline (Gantt-style)
# ========================================================================

# Prepare data for Gantt chart using plotly express
df_gantt = df_casting_pd.copy()
df_gantt['Casting_Label'] = df_gantt['Casting ID'].apply(lambda x: f"Casting {int(x)}")
df_gantt['Duration_hours'] = (df_gantt['ts_end'] - df_gantt['ts_start']).dt.total_seconds() / 3600

# Create Gantt chart using plotly express timeline
fig1 = px.timeline(
    df_gantt,
    x_start='ts_start',
    x_end='ts_end',
    y='Casting_Label',
    color_discrete_sequence=['steelblue'],
    title='Casting Periods Timeline - Strand 6',
    labels={'Casting_Label': 'Casting ID'},
    hover_data={
        'Quality casting': True,
        'Heat Count': True,
        'ts_start': '|%Y-%m-%d %H:%M',
        'ts_end': '|%Y-%m-%d %H:%M',
        'Duration_hours': ':.1f',
        'Casting_Label': False
    }
)

# Add quality labels alternating above and below bars
for idx, row in df_gantt.iterrows():
    # Calculate middle of the bar for x position
    mid_time = row['ts_start'] + (row['ts_end'] - row['ts_start']) / 2
    
    # Alternate position: even indices above, odd indices below
    if idx % 2 == 0:
        # Position above the bar
        y_offset = 25  # pixels above
        arrow_direction = -15  # arrow points down to bar
    else:
        # Position below the bar
        y_offset = -25  # pixels below
        arrow_direction = 15  # arrow points up to bar
    
    fig1.add_annotation(
        x=mid_time,
        y=row['Casting_Label'],
        text=row['Quality casting'],
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.5,
        arrowcolor='gray',
        ax=0,  # No horizontal offset
        ay=arrow_direction,  # Vertical arrow pointing to bar
        font=dict(size=10, color='darkblue', weight='bold'),
        xanchor='center',
        yanchor='middle',
        yshift=y_offset,  # Shift text up or down
        bgcolor='rgba(255, 255, 255, 0.8)',  # White background for readability
        bordercolor='lightgray',
        borderwidth=1,
        borderpad=3
    )

fig1.update_layout(
    xaxis_title='Date and Time',
    yaxis_title='Casting ID',
    height=max(600, len(df_casting_pd) * 25),
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    xaxis=dict(
        showgrid=True,
        gridcolor='white',
        gridwidth=1,
        type='date'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='white',
        gridwidth=1,
        autorange='reversed'
    ),
    title_font=dict(size=18, color='darkblue'),
    margin=dict(t=100, b=80)  # Add top and bottom margins for labels
)

# Update bar appearance
fig1.update_traces(
    marker=dict(opacity=0.7, line=dict(color='darkblue', width=1))
)

fig1.show()

# Save Figure 1
fig1_path = os.path.join(output_dir, 'casting_periods_timeline_strand6.html')
fig1.write_html(fig1_path)
print(f"\n✓ Figure 1 saved to: {fig1_path}")

print(f"\n{'='*70}")
print(f"FIGURE 1: Casting Periods Overview (Strand 6)")
print(f"{'='*70}")
print(f"Total casting periods: {len(df_casting_pd)}")
print(f"Date range: {df_casting_pd['ts_start'].min().strftime('%Y-%m-%d')} to {df_casting_pd['ts_end'].max().strftime('%Y-%m-%d')}")
print(f"Total casting time: {(df_casting_pd['ts_end'] - df_casting_pd['ts_start']).sum().total_seconds() / 3600:.1f} hours")


# df_gaps_pd must exist; if not, skip
if "PlainTimeStamp" in df_gaps_pd.columns:
    df_gaps_pd = df_gaps_pd.copy()
    df_gaps_pd["PlainTimeStamp"] = pd.to_datetime(df_gaps_pd["PlainTimeStamp"], errors="coerce")

    # keep only points inside Strand-6 casting windows
    df_gaps_pd = df_gaps_pd[in_any_casting_window(df_gaps_pd["PlainTimeStamp"])].reset_index(drop=True)

df_gaps_class_pd = df_gaps_class_pd.copy()

# Ensure datetimes (again safe)
df_gaps_class_pd["prev_timestamp"] = pd.to_datetime(df_gaps_class_pd["prev_timestamp"], errors="coerce")
df_gaps_class_pd["PlainTimeStamp"] = pd.to_datetime(df_gaps_class_pd["PlainTimeStamp"], errors="coerce")

# Keep only gaps that overlap Strand-6 casting windows
df_gaps_class_pd = df_gaps_class_pd[
    df_gaps_class_pd.apply(gap_overlaps_any_casting, axis=1)
].reset_index(drop=True)

# OPTIONAL: recompute gaps BETWEEN Strand-6 castings from df_casting_pd
df_casting_sorted = df_casting_pd.sort_values("ts_start").reset_index(drop=True)

between_rows = []
for i in range(1, len(df_casting_sorted)):
    prev_end = df_casting_sorted.loc[i-1, "ts_end"]
    cur_start = df_casting_sorted.loc[i, "ts_start"]
    if cur_start > prev_end:
        between_rows.append({
            "prev_timestamp": prev_end,
            "PlainTimeStamp": cur_start,
            "time_diff_hours": (cur_start - prev_end).total_seconds() / 3600.0,
            "gap_type": "Between Castings",
            "casting_id": None
        })

df_between_strand6 = pd.DataFrame(between_rows)



# ========================================================================
# FIGURE 2: Data Points and Gaps Timeline
# ========================================================================
fig2 = go.Figure()

# Add casting period background bands (without annotations to avoid timestamp arithmetic issues)
for idx, row in df_casting_pd.iterrows():
    fig2.add_vrect(
        x0=row['ts_start'], x1=row['ts_end'],
        fillcolor='lightblue', opacity=0.25,
        layer="below", line_width=0
    )

# Add gaps between castings (orange) and within castings (red)
if len(df_gaps_class_pd) > 0:
    #gaps_between = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Between Castings']
    gaps_between = df_between_strand6
    gaps_within = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Within Casting']
    
    # Add gap rectangles for gaps between castings
    for idx, row in gaps_between.iterrows():
        fig2.add_trace(
            go.Scatter(
                x=[row['prev_timestamp'], row['PlainTimeStamp'], row['PlainTimeStamp'], row['prev_timestamp'], row['prev_timestamp']],
                y=[0.5, 0.5, 1.5, 1.5, 0.5],
                fill='toself',
                fillcolor='orange',
                opacity=0.4,
                line=dict(width=0),
                mode='lines',
                showlegend=False,
                hovertemplate=f"<b>Gap Between Castings</b><br>" +
                             f"Duration: {row['time_diff_hours']:.1f}h<br>" +
                             f"From: {row['prev_timestamp'].strftime('%Y-%m-%d %H:%M')}<br>" +
                             f"To: {row['PlainTimeStamp'].strftime('%Y-%m-%d %H:%M')}<extra></extra>"
            )
        )
    
    # Add gap rectangles for gaps within castings
    for idx, row in gaps_within.iterrows():
        fig2.add_trace(
            go.Scatter(
                x=[row['prev_timestamp'], row['PlainTimeStamp'], row['PlainTimeStamp'], row['prev_timestamp'], row['prev_timestamp']],
                y=[0.5, 0.5, 1.5, 1.5, 0.5],
                fill='toself',
                fillcolor='red',
                opacity=0.6,
                line=dict(width=0),
                mode='lines',
                showlegend=False,
                hovertemplate=f"<b>⚠️ Gap WITHIN Casting (Data Loss!)</b><br>" +
                             f"Duration: {row['time_diff_hours']:.1f}h<br>" +
                             f"Casting: {int(row['casting_id']) if pd.notna(row['casting_id']) else 'Unknown'}<br>" +
                             f"From: {row['prev_timestamp'].strftime('%Y-%m-%d %H:%M')}<br>" +
                             f"To: {row['PlainTimeStamp'].strftime('%Y-%m-%d %H:%M')}<extra></extra>"
            )
        )

# Add data points (sampled)
if len(df_gaps_pd) > 0:
    sample_indices = range(0, len(df_gaps_pd), max(1, len(df_gaps_pd) // 3000))
    timeline_sample = df_gaps_pd.iloc[sample_indices]
    
    fig2.add_trace(
        go.Scatter(
            x=timeline_sample['PlainTimeStamp'],
            y=[1] * len(timeline_sample),
            mode='markers',
            marker=dict(color='green', size=2, opacity=0.3),
            name='Data points',
            hovertemplate='Data point at: %{x}<extra></extra>'
        )
    )

# Add legend items
fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='lightblue', opacity=0.25, line=dict(color='steelblue', width=1)),
        name='Casting period',
        showlegend=True
    )
)

fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='red', opacity=0.6),
        name='⚠️ Gap WITHIN casting (data loss)',
        showlegend=True
    )
)

fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='orange', opacity=0.4),
        name='Gap BETWEEN castings (expected)',
        showlegend=True
    )
)

fig2.update_layout(
    title=dict(
        text='Data Collection Timeline with Gaps - Strand 6',
        font=dict(size=18, color='darkblue')
    ),
    xaxis_title='Date and Time',
    yaxis_title='Data Presence',
    height=600,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.12,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="black",
        borderwidth=1
    ),
    hovermode='closest',
    yaxis=dict(
        showticklabels=False,
        range=[0.4, 1.6],
        showgrid=False
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=0.5,
        type='date'
    ),
    plot_bgcolor='white'
)

fig2.show()

# Save Figure 2
fig2_path = os.path.join(output_dir, 'data_quality_timeline_strand6.html')
fig2.write_html(fig2_path)
print(f"\n✓ Figure 2 saved to: {fig2_path}")

print(f"\n{'='*70}")
print(f"FIGURE 2: Data Quality Analysis (Strand 6)")
print(f"{'='*70}")
if len(df_gaps_class_pd) > 0:
    gaps_within = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Within Casting']
    gaps_between = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Between Castings']
    print(f"✓ Data points analyzed: {len(df_gaps_pd):,}")
    print(f"⚠️  Gaps WITHIN castings: {len(gaps_within)} (RED - data collection issues)")
    print(f"✓ Gaps BETWEEN castings: {len(gaps_between)} (ORANGE - expected downtime)")
    if len(gaps_within) > 0:
        total_loss_hours = gaps_within['time_diff_hours'].sum()
        print(f"⚠️  Total data loss during casting: {total_loss_hours:.1f} hours")
else:
    print(f"✓ No significant gaps found")
print(f"{'='*70}\n")

print(f"\n{'='*70}")
print(f"SAVED FILES")
print(f"{'='*70}")
print(f"Figure 1: {fig1_path}")
print(f"Figure 2: {fig2_path}")
print(f"\nAccess via: https://<databricks-instance>/files/TATAIjmulden_FCMoldG5/figures/")
print(f"{'='*70}\n")

In [0]:
display(dbutils.fs.ls('dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/'))

In [0]:
from pyspark.sql import functions as F

# Find data points that fall outside any casting period
df_data_with_casting = (
    df_23_6_spark_unitConverted
    .select("PlainTimeStamp")
    .alias("data")
    .join(
        broadcast(df_casting_periods.alias("cast")),
        (
            (F.col("data.PlainTimeStamp") >= F.col("cast.ts_start")) &
            (F.col("data.PlainTimeStamp") <= F.col("cast.ts_end"))
        ),
        how="left"
    )
)

# Count matched vs unmatched
matching_stats = df_data_with_casting.select(
    F.count("*").alias("total_data_points"),
    F.sum(F.when(F.col("Casting ID").isNotNull(), 1).otherwise(0)).alias("matched_to_casting"),
    F.sum(F.when(F.col("Casting ID").isNull(), 1).otherwise(0)).alias("unmatched_orphan_data")
).collect()[0]

print("\n" + "="*60)
print("DATA-METADATA MISMATCH ANALYSIS")
print("="*60)
print(f"\nTotal data points: {matching_stats['total_data_points']:,}")
print(f"Matched to casting periods: {matching_stats['matched_to_casting']:,} ({100*matching_stats['matched_to_casting']/matching_stats['total_data_points']:.2f}%)")
print(f"Orphan data (outside any casting): {matching_stats['unmatched_orphan_data']:,} ({100*matching_stats['unmatched_orphan_data']/matching_stats['total_data_points']:.2f}%)")

# Show time range of orphan data
orphan_data = df_data_with_casting.filter(F.col("Casting ID").isNull())
if orphan_data.count() > 0:
    orphan_range = orphan_data.select(
        F.min("PlainTimeStamp").alias("first_orphan"),
        F.max("PlainTimeStamp").alias("last_orphan")
    ).collect()[0]
    
    print(f"\nOrphan data time range:")
    print(f"  First: {orphan_range['first_orphan']}")
    print(f"  Last: {orphan_range['last_orphan']}")

# Check for casting periods with no data
print("\n" + "-"*60)
print("Casting periods with data coverage:")
print("-"*60)

casting_coverage = (
    df_casting_periods.alias("cast")
    .join(
        df_23_6_spark_unitConverted.select("PlainTimeStamp").alias("data"),
        (
            (F.col("data.PlainTimeStamp") >= F.col("cast.ts_start")) &
            (F.col("data.PlainTimeStamp") <= F.col("cast.ts_end"))
        ),
        how="left"
    )
    .groupBy("Casting ID", "Strand ID", "ts_start", "ts_end", "Quality casting")
    .agg(
        F.count("PlainTimeStamp").alias("data_point_count"),
        F.min("PlainTimeStamp").alias("first_data"),
        F.max("PlainTimeStamp").alias("last_data")
    )
    .withColumn(
        "coverage_status",
        F.when(F.col("data_point_count") == 0, "NO DATA")
         .when(F.col("first_data") > F.col("ts_start"), "LATE START")
         .when(F.col("last_data") < F.col("ts_end"), "EARLY END")
         .otherwise("FULL COVERAGE")
    )
    .orderBy("ts_start")
)

display(casting_coverage)

# Summary by coverage status
print("\nCoverage Summary:")
casting_coverage.groupBy("coverage_status").count().orderBy("coverage_status").show()

In [0]:
# Create detailed mismatch report for further investigation
mismatch_report = casting_coverage.filter(
    F.col("coverage_status") != "FULL COVERAGE"
).select(
    "Casting ID",
    "Strand ID",
    "ts_start",
    "ts_end",
    "Quality casting",
    "data_point_count",
    "first_data",
    "last_data",
    "coverage_status"
).withColumn(
    "expected_duration_hours",
    (F.unix_timestamp("ts_end") - F.unix_timestamp("ts_start")) / 3600
).withColumn(
    "data_start_delay_hours",
    F.when(
        F.col("first_data").isNotNull(),
        (F.unix_timestamp("first_data") - F.unix_timestamp("ts_start")) / 3600
    ).otherwise(None)
).withColumn(
    "data_end_early_hours",
    F.when(
        F.col("last_data").isNotNull(),
        (F.unix_timestamp("ts_end") - F.unix_timestamp("last_data")) / 3600
    ).otherwise(None)
)

print(f"\nCastings with data mismatches: {mismatch_report.count()}")
display(mismatch_report.orderBy("ts_start"))

# Save report
import os
report_path = "/dbfs/FileStore/TATAIjmulden_FCMoldG5/reports/casting_data_mismatch_report.csv"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
mismatch_report.toPandas().to_csv(report_path, index=False)
print(f"\nMismatch report saved to: {report_path}")

In [0]:
# Remove non-casting data: speed<0.5 m/min or SEN depth outside physical range
df_23_6_spark_clean = (
    df_23_6_spark_unitConverted
        .filter(F.col("castingSpeed") >= 0.5)
        .filter(F.col("SENImmersionDepth").between(0.1, 0.26))
        #.filter(F.col("moldWidth").isNotNull())
)


In [0]:
print("Before:", df_23_6_spark_unitConverted.count())
print("After :", df_23_6_spark_clean.count())


## 2.4 Parameter Distributions and Time Series Visualization

Explore the cleaned data visually:
* Histograms of casting speed, mold width, SEN depth
* Time series of mold level, argon flows, EMBR currents
* Scatter density plots (high-point-count data)
* Quality casting breakdown by time period

In [0]:
import os
fig_dir = "dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/"
os.makedirs(fig_dir, exist_ok=True)

In [0]:
cols_for_plot = [
    "PlainTimeStamp",
    "castingSpeed",
    "moldWidth",
    "SENImmersionDepth",
    'Mold Level', 
    'Mold Level Sensor Left', 
    'Mold Level Sensor Right',
    'Argon Flow SEN', 'Argon Flow Stopper',
    'EMBR Current DC Left Bottom',
    'EMBR Current AC Left Master',
    'EMBR Current DC Left Master',
    'Quality casting'
]

df_23_6_small = (
    df_23_6_spark_clean
       .select(*cols_for_plot)
        .dropna(subset=["castingSpeed", "moldWidth", "SENImmersionDepth"])
        # Optional if datasets are huge:
        # .sample(False, 0.1, seed=42)
        # .limit(500000)
)

df_23_6 = df_23_6_small.toPandas()
df_23_6 = df_23_6.rename(columns={"PlainTimeStamp": "plainTimeStamp"})

In [0]:
display(df_23_6_small)

In [0]:
df_23_6_small.printSchema()

In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# --- Create subplots container ---
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Casting Speed Distribution",
        "SENID Distribution",
        "Mold Width Distribution",
        "Argon Flow SEN+Stopper Distribution"
    ]
)

# --- Subplot 1: Casting Speed ---
h1 = px.histogram(df_23_6, x='castingSpeed')
fig1.add_trace(h1.data[0], row=1, col=1)
fig1.update_xaxes(title_text="Casting Speed [m/min]", row=1, col=1)
fig1.update_yaxes(title_text="Count", row=1, col=1)

# --- Subplot 2: SEN Immersion Depth ---
h2 = px.histogram(df_23_6, x='SENImmersionDepth')#,nbins=165)
fig1.add_trace(h2.data[0], row=1, col=2)
fig1.update_xaxes(title_text="SENImmersionDepth [m]", row=1, col=2)
fig1.update_yaxes(title_text="Count", row=1, col=2)

# --- Subplot 3: Mold Width ---
h3 = px.histogram(df_23_6, x='moldWidth')
fig1.add_trace(h3.data[0], row=2, col=1)
fig1.update_xaxes(title_text="Mold Width [m]", row=2, col=1)
fig1.update_yaxes(title_text="Count", row=2, col=1)

# --- Subplot 4: Argon Flow (SEN + Stopper) ---
argon_total = df_23_6[['Argon Flow SEN', 'Argon Flow Stopper']].sum(axis=1)
h4 = px.histogram(x=argon_total)
fig1.add_trace(h4.data[0], row=2, col=2)
fig1.update_xaxes(title_text="Argon Flow SEN+STOPPER [l/min]", row=2, col=2)
fig1.update_yaxes(title_text="Count", row=2, col=2)

# --- Final Layout ---
fig1.update_layout(
    height=900,
    width=1100,
    title_text="Strand 23-6 — Parameter Distributions",
    showlegend=False
)

fig1.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/cc_parameters_histograms.html")
fig1.show()




In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplot grid (2 rows × 2 columns)
fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Casting Speed Over Time",
        "SEN Immersion Depth Over Time",
        "Mold Width Over Time",
        "Argon Flow (SEN + Stopper) Over Time"
    ]
)

# 1️⃣ Casting Speed
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['castingSpeed'],
        mode='lines'
    ),
    row=1, col=1
)

# 2️⃣ SEN Immersion Depth
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['SENImmersionDepth'],
        mode='lines'
    ),
    row=1, col=2
)

# 3️⃣ Mold Width
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['moldWidth'],
        mode='lines'
    ),
    row=2, col=1
)

# 4️⃣ Argon Flow = SEN + Stopper
argon_total = df_23_6[['Argon Flow SEN', 'Argon Flow Stopper']].sum(axis=1)

fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=argon_total,
        mode='lines'
    ),
    row=2, col=2
)

# Axis labels
fig2.update_xaxes(title_text="Timestamp", row=1, col=1)
fig2.update_xaxes(title_text="Timestamp", row=1, col=2)
fig2.update_xaxes(title_text="Timestamp", row=2, col=1)
fig2.update_xaxes(title_text="Timestamp", row=2, col=2)

fig2.update_yaxes(title_text="Casting Speed [m/min]", row=1, col=1)
fig2.update_yaxes(title_text="SEN Immersion Depth [m]", row=1, col=2)
fig2.update_yaxes(title_text="Mold Width [m]", row=2, col=1)
fig2.update_yaxes(title_text="Argon Flow [l/min]", row=2, col=2)

# Figure layout
fig2.update_layout(
    height=1000,
    width=1200,
    title_text="Strand 23-6 — Time Series of Key Parameters",
    showlegend=False
)
fig2.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/cc_parameters_timeseriesPlots.html")
fig2.show()


In [0]:
df_23_6.shape

In [0]:
from matplotlib.colors import LogNorm# "Viridis-like" colormap with white background

white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
    (0, '#ffffff'),
    (1e-20, '#440053'),
    (0.2, '#404388'),
    (0.4, '#2a788e'),
    (0.6, '#21a784'),
    (0.8, '#78d151'),
    (1, '#fde624'),
], N=256)

def using_mpl_scatter_density(fig, x, y, labels):
    ax = fig.add_subplot(1, 1, 1, projection='scatter_density')
    density = ax.scatter_density(x, y, cmap=white_viridis, norm=LogNorm(vmin=1,vmax=5000), dpi=12)
    fig.colorbar(density, label=labels)

fig3 = plt.figure(tight_layout = True)
using_mpl_scatter_density(fig3, df_23_6.moldWidth, df_23_6.castingSpeed, labels='CC snapshot counts')
plt.xlabel('mold Width, [m]')
plt.ylabel('casting Speed, [m/min]')
fig3.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/width_vs_speed_density_strd_23_6.png")
plt.show()


In [0]:
# Plotly line plot for Mold Level
fig4 = px.line(
    df_23_6,
    y=['Mold Level', 'Mold Level Sensor Left', 'Mold Level Sensor Right'],
    x=  'plainTimeStamp',
    title='Mold Level Over Time'
    )
fig4.update_layout(xaxis_title='plainTimeStamp', yaxis_title='Mold Level [m]')
fig4.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/mold_level_plot.html")
fig4.show()

In [0]:
%fs ls dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/

In [0]:
# after any sampling or filtering
df_23_6_sampled = (
    df_23_6
      .sample(frac=0.25, random_state=42)   # or whatever sampling you used
      .sort_values("plainTimeStamp")        # VERY important
      .reset_index(drop=True)
)
df_23_6_sampled.shape

In [0]:
df_23_6_sampled.head()

In [0]:
df_23_6_sampled.columns

In [0]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.dropna(subset=["plainTimeStamp"]).sort_values("plainTimeStamp").reset_index(drop=True)

# gap-compressed x-axis
df["t_idx"] = np.arange(len(df))

# Argon average (use .sum(axis=1) if you want total instead)
df["Argon Flow AVG"] = df[["Argon Flow SEN", "Argon Flow Stopper"]].mean(axis=1)

# Variables to plot (one subplot each)
series_specs = [
    ("castingSpeed",        "Casting speed"),
    ("moldWidth",           "Mold width"),
    ("SENImmersionDepth",   "SEN immersion depth"),
    ("Mold Level",          "Mold level"),
    ("Argon Flow AVG",      "Argon flow (avg of SEN+Stopper)"),
]

nrows = len(series_specs)

fig = make_subplots(
    rows=nrows, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[title for _, title in series_specs],
)

for r, (col, _) in enumerate(series_specs, start=1):
    fig.add_trace(
        go.Scatter(
            x=df["t_idx"],
            y=df[col],
            mode="lines",
            customdata=df[["plainTimeStamp"]].to_numpy(),
            hovertemplate=(
                "idx=%{x}<br>"
                "time=%{customdata[0]}<br>"
                f"{col}=%{{y}}<extra></extra>"
            ),
            name=col,
            showlegend=False,
        ),
        row=r, col=1
    )
    fig.update_yaxes(title_text=col, row=r, col=1)

# Sparse readable timestamp ticks
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig.update_xaxes(
    title_text="Time (gaps removed; hover shows real timestamp)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
    row=nrows, col=1
)

fig.update_layout(
    title="Strand#23_6 – Casting parameters (sampled and gap-compressed)",
    height=260 * nrows,
    width=1200,
)

# Save & show
fig.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_casting_parameters_subplots.html")
fig.show()


In [0]:
import pandas as pd
import plotly.express as px

df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig = px.line(
    df,
    x="t_idx",
    y=[
        "castingSpeed",
        "moldWidth",
       # "SENImmersionDepth",
       # 'Mold Level'
    ],
    title="Strand#23_6: Casting speed and Mold width (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)
# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_Vc_and_width_parameters.html"
)

fig.show()


In [0]:
df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig7 = px.line(
    df,
    x="t_idx",
    y=[
        "Mold Level", 
        "Mold Level Sensor Left", 
        "Mold Level Sensor Right"
    ],
    title="Strand#23_6: Mold Level (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig7.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig7.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)
fig7.update_layout(yaxis_title="Mold Level [mm]")
fig7.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/Argon_plot.html")
fig7.show()

In [0]:


df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig8 = px.line(
    df,
    x="t_idx",
    y=[
        'Argon Flow SEN', 
        'Argon Flow Stopper'
    ],
    title="Strand#23_6: Argon flow (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig8.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig8.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)

fig8.show()

In [0]:


df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig8 = px.line(
    df,
    x="t_idx",
    y=[
        "EMBR Current DC Left Bottom",
        "EMBR Current AC Left Master",
        "EMBR Current DC Left Master",
    ],
    title="Strand#23_6: FC Mold currents (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig8.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig8.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)

fig8.show()


In [0]:
%python
import plotly.express as px

columns_to_plot = [
    "castingSpeed",
    "moldWidth",
    "SENImmersionDepth",
    "Mold Level",
    "Argon Flow SEN",
    "Argon Flow Stopper",
    "EMBR Current DC Left Bottom",
    "EMBR Current AC Left Master",
    "EMBR Current DC Left Master"
]

# Reshape to long format
df_long = df_23_6.melt(
    id_vars=["plainTimeStamp"],
    value_vars=columns_to_plot,
    var_name="Parameter",
    value_name="Value"
)

fig = px.line(
    df_long,
    x="plainTimeStamp",
    y="Value",
    color="Parameter",
    facet_col="Parameter",
    facet_col_wrap=4,
    labels={"plainTimeStamp": "Time", "Value": "Value"},
    color_discrete_sequence=["blue"],
    facet_col_spacing=0.05,
    facet_row_spacing=0.1,
    category_orders={"plainTimeStamp": df_long["plainTimeStamp"].astype(str).unique().tolist()}
)


fig.update_layout(showlegend=False)

fig.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand1_ALL_signals_SEN.html")
fig.show()

## 2.5 Sequence Identification and Statistics

Identify **stable casting sequences** using a sliding-window approach:
* Window = 300 samples (6 min at 1Hz)
* Stability criteria: casting speed variation < 0.1 m/min, EMBR current variation < 50 A
* Gaps > 5 seconds trigger a new segment

Each sequence is characterised by its mean/std of mold level, speed, width, SEN depth, and argon flows.

In [0]:
# ---------------------------------------------------------------------------
# Sequence functions: imported from src/ in cell 33
#   - custom_rounding, segment_by_time_gaps, identify_sequences,
#     identify_sequences_segmented, SequenceAnalyzer
# Below: ONLY local extras not in src/
# ---------------------------------------------------------------------------

def filter_and_group_sequences(df, status_column, speed_column, status_value=1, diff_threshold=0.1, min_length=300):
    """
    Filters the DataFrame based on a status column and groups sequences of rows
    where the difference between consecutive values in a specified column is
    less than or equal to a threshold, retaining only groups with a minimum length.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        status_column (str): The column to filter by status.
        speed_column (str): The column to group by consecutive differences.
        status_value (int): The value to filter the status column. Default is 1.
        diff_threshold (float): The maximum allowed difference between consecutive values. Default is 0.1.
        min_length (int): The minimum length of a valid group. Default is 300.

    Returns:
        pd.DataFrame: A DataFrame containing valid groups.
    """
    # Step 1: Filter the DataFrame where the status column matches the specified value
    filtered_df = df[df[status_column] == status_value].reset_index(drop=True)

    # Step 2: Identify groups based on the difference in the speed column
    # Create a group identifier where the difference between consecutive rows is > diff_threshold
    filtered_df['group'] = (filtered_df[speed_column].diff().abs() > diff_threshold).cumsum()

    # Step 3: Group by the 'group' column and filter groups with at least the minimum length
    valid_groups = (
        filtered_df.groupby('group')
        .filter(lambda x: len(x) >= min_length)
        .reset_index(drop=True)
    )

    # Drop the temporary 'group' column if not needed
    valid_groups = valid_groups.drop(columns=['group'])

    return valid_groups

# segment_by_time_gaps, identify_sequences, identify_sequences_segmented
# are imported from src.sequence_analysis (cell 33)
print("Sequence functions available from src/:")
print(f"  custom_rounding, segment_by_time_gaps, identify_sequences,")
print(f"  identify_sequences_segmented, SequenceAnalyzer")
print(f"Local extra: filter_and_group_sequences")


In [0]:
df_23_6.head(5)

In [0]:
# Keep only records where FC Mold is active (all 3 EMBR currents non-zero)
df_FCMold_ON_str23_6 = df_23_6[(df_23_6['EMBR Current AC Left Master'] != 0) & (df_23_6['EMBR Current DC Left Master'] != 0) & (df_23_6['EMBR Current DC Left Bottom'] != 0)].reset_index(drop=True)

In [0]:
(df_23_6.shape, df_FCMold_ON_str23_6.shape)

In [0]:
df_FCMold_ON_str23_6['castingSpeed'] =  custom_rounding(df_FCMold_ON_str23_6.castingSpeed, 2)

In [0]:
df_FCMold_ON_str23_6.head(10)

In [0]:
# Configuration for sequence detection
window_size = 300  # 6 min window size
Vc_column_str5 = 'castingSpeed'
Vc_column_str6 = 'castingSpeed'
Vc_threshold = 0.1
Curr_columns_str5 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_columns_str6 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_threshold = 50

In [0]:

Vc_column_str6 = "castingSpeed"

normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = identify_sequences_segmented(
    df_FCMold_ON_str23_6,
    Vc_column=Vc_column_str6,
    window_size=window_size,
    Vc_threshold=Vc_threshold,
    Curr_columns=Curr_columns_str6,
    Curr_threshold=Curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=5,          # treat gaps >5s as new segments
    min_segment_len=window_size # ignore segments shorter than one window
)

len(normal_groups_ON_str23_6), len(abnormal_groups_ON_str23_6)

In [0]:
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import pandas as pd
import math

df = df_FCMold_ON_str23_6
Vc_col = Vc_column_str6
window_size_vis = 300
max_windows = 12
Vc_threshold = Vc_threshold

# ---------- 1) Restrict DF to FIRST 30 MIN ONLY & SORT ----------
if "plainTimeStamp" in df.columns:
    tcol = "plainTimeStamp"
elif "plainTimestamp" in df.columns:
    tcol = "plainTimestamp"
else:
    raise ValueError("Need timestamp column.")

ts_full = df[tcol]
start_ts = ts_full.iloc[0]
end_ts_2h = start_ts + pd.Timedelta(hours=0.5)

mask_2h = (ts_full >= start_ts) & (ts_full <= end_ts_2h)
df_chunk = (
    df.loc[mask_2h]
      .copy()
      .sort_values(tcol)
      .reset_index(drop=True)
)

ts = df_chunk[tcol]
Vc = df_chunk[Vc_col]
y_all = df_chunk["castingSpeed"]

n = len(df_chunk)
idx_all = np.arange(n)

# ---------- 2) Sliding-window search on chunk ----------
windows = []

i = 0
while i <= n - window_size_vis and len(windows) < max_windows:
    window = Vc.iloc[i:i + window_size_vis]
    diffs = np.abs(window - window.iloc[0])

    bad_offsets = np.where(diffs > Vc_threshold)[0]
    if len(bad_offsets) > 0:
        first_bad = bad_offsets[0]
        break_idx = i + first_bad
        cond = False
    else:
        break_idx = None
        cond = True

    start = i
    end = i + window_size_vis
    windows.append((start, end, break_idx, cond))

    if cond:
        i += window_size_vis
    else:
        i = break_idx + 1

n_windows = len(windows)
n_cols = 2
n_rows = max(1, math.ceil(n_windows / n_cols))

# ---------- 3) Plotly figure with 2 columns ----------
titles = [
    f"Window {k} (start={start}, end={end-1}) — {'NORMAL' if cond else 'ABNORMAL'}"
    for k, (start, end, break_idx, cond) in enumerate(windows)
]
# pad titles if last row not full
if len(titles) < n_rows * n_cols:
    titles += [""] * (n_rows * n_cols - len(titles))

fig = sp.make_subplots(
    rows=n_rows,
    cols=n_cols,
    shared_xaxes=True,
    vertical_spacing=0.06,
    horizontal_spacing=0.08,
    subplot_titles=titles,
)

for idx, (start, end, break_idx, cond) in enumerate(windows):
    row = idx // n_cols + 1
    col = idx % n_cols + 1

    first_subplot = (idx == 0)
    normal_subplot = (idx == n_windows - 1)

    gray_y = y_all.copy()

    in_window = (idx_all >= start) & (idx_all < end)
    blue_mask = np.zeros(n, dtype=bool)
    black_mask = np.zeros(n, dtype=bool)
    red_mask = np.zeros(n, dtype=bool)

    if cond:
        blue_mask[in_window] = True
        gray_y[blue_mask] = np.nan
    else:
        if break_idx is not None:
            black_mask[(idx_all >= start) & (idx_all < break_idx)] = True
            red_mask[(idx_all >= break_idx) & (idx_all < end)] = True

        gray_y[in_window] = np.nan

    # gray trace
    fig.add_trace(
        go.Scatter(
            x=ts,
            y=gray_y,
            mode="lines",
            line=dict(color="gray", width=1),
            name="castingSpeed, m/min" if first_subplot else None,
            showlegend=first_subplot,
        ),
        row=row,
        col=col,
    )

    # blue: full normal windows
    if blue_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(blue_mask, np.nan),
                mode="lines",
                line=dict(color="blue", width=3),
                name="Normal window", # if first_subplot else None,
                showlegend=normal_subplot,
            ),
            row=row,
            col=col,
        )

    # black: pre-break stable region in abnormal windows
    if black_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(black_mask, np.nan),
                mode="lines",
                line=dict(color="black", width=2),
                name="normal region" if first_subplot else None,
                showlegend=first_subplot,
            ),
            row=row,
            col=col,
        )

    # red: abnormal region
    if red_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(red_mask, np.nan),
                mode="lines",
                line=dict(color="red", width=3),
                name="abnormal region" if first_subplot else None,
                showlegend=first_subplot,
            ),
            row=row,
            col=col,
        )

# layout
fig.update_layout(
    height=260 * n_rows,
    width=1300,
    title="Data grouping - Sliding Window Visualization",
)

# x-axis label only on bottom row
for c in range(1, n_cols + 1):
    fig.update_xaxes(title_text="Time", row=n_rows, col=c)

fig.show()


In [0]:
# Compute aggregate statistics (mean, std, min, max) per stable sequence
def generate_seq_statistics(df, sequences, seq_condition):
    df_result = pd.DataFrame()
    cnt = 0

    for group in sequences:
        if max(group) >= len(df):
            raise IndexError("positional indexers are out-of-bounds")
        
        temp_df = df.iloc[group]

        temp_df = temp_df.reset_index(drop=True)

        # Populate df_results with Statistics
        df_result.loc[cnt, 'Seq_Name'] = ','.join(['Seq#'+'%d' %(cnt)]) #', '.join(temp_df['filename'].apply(lambda x: os.path.basename(x)).unique())[25:29]+'_%d' %(cnt)]

        timestamps = temp_df['plainTimeStamp'].to_list()

        if timestamps:
            df_result.loc[cnt, 'Seq_time_Start'] = timestamps[0]
            #if len(timestamps) > (result_reduced[0][1] - 1):
            #    df_result.loc[cnt, 'Seq_time_End'] = timestamps[result_reduced[0][1] - 1]
            #else:
            df_result.loc[cnt, 'Seq_time_End'] = timestamps[-1]  # Use the last timestamp if the index is out of range
        else:
            df_result.loc[cnt, 'Seq_time_Start'] = None
            df_result.loc[cnt, 'Seq_time_End'] = None

        df_result.loc[cnt, 'SEN_avg [mm]'] = temp_df['SENImmersionDepth'].mean()
        df_result.loc[cnt, 'SEN_std [mm]'] = temp_df['SENImmersionDepth'].std()

       
        # Compute statistics
        df_result.loc[cnt,'CASTING_SPEED_avg [m/min]'] = temp_df['castingSpeed'].mean()
        df_result.loc[cnt,'CASTING_SPEED_std [m/min]'] = temp_df['castingSpeed'].std()
        df_result.loc[cnt,'MOLD_WIDTH_avg [m]'] = temp_df['moldWidth'].mean()
        df_result.loc[cnt,'MOLD_WIDTH_std [m]'] = temp_df['moldWidth'].std()
        df_result.loc[cnt,'min DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].min()
        df_result.loc[cnt,'mean DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].mean()
        df_result.loc[cnt,'max DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].max()
        df_result.loc[cnt,'min DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].min()
        df_result.loc[cnt,'mean DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].mean()
        df_result.loc[cnt,'max DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].max()
        df_result.loc[cnt,'min AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].min()
        df_result.loc[cnt,'mean AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].mean()
        df_result.loc[cnt,'max AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].max()
        df_result.loc[cnt, 'MOLD_LEVEL_avg [mm]'] = temp_df['Mold Level'].mean()
        df_result.loc[cnt, 'MOLD_LEVEL_std [mm]'] = temp_df['Mold Level'].std()
        df_result.loc[cnt, 'ArFlow_avg [NL/min]'] = (temp_df['Argon Flow SEN'] + temp_df['Argon Flow Stopper']).mean()
        df_result.loc[cnt, 'ArFlow_std [NL/min]'] = (temp_df['Argon Flow SEN'] + temp_df['Argon Flow Stopper']).std()
        #df_result.loc[cnt, 'SEN_avg [m]'] = temp_df['SEN'].mean()
        #df_result.loc[cnt, 'SEN_std [m]'] = temp_df['SEN'].std()

        
        df_result.loc[cnt, 'Seq_Condition'] = seq_condition
        cnt +=1
        
    return df_result



In [0]:
df_seq_normal_ON_str23_6 = generate_seq_statistics(df_FCMold_ON_str23_6, normal_groups_ON_str23_6, 'normal')

In [0]:
df_seq_normal_ON_str23_6.shape

In [0]:
df_seq_normal_ON_str23_6.head()

In [0]:
def plot_MLsigma_per_sequence(df_seq, df_FCstatus, str_MLsignal, str_label):
    # Remove underscores from the 'Seq_Name' column values
    df_seq['Seq_Name'] = df_seq['Seq_Name'].str.replace('_', '', regex=False)

    for seq in df_seq['Seq_Name'].to_list():
        # Extract the sequence info - statistics - 
        seq_info = df_seq[df_seq['Seq_Name'] == seq]
        # Remove underscores from the 'Seq_Name' column values
        #seq_info['Seq_Name'] = seq_info['Seq_Name'].str.replace('_', '', regex=False)

        # Get the start and end time as strings
        start_time = seq_info['Seq_time_Start'].iloc[0]
        end_time = seq_info['Seq_time_End'].iloc[0]

        # Slice the DataFrame based on the 'time' column
        df_temp = df_FCstatus[(df_FCstatus['plainTimeStamp'] >= start_time) & (df_FCstatus['plainTimeStamp'] <= end_time)]

        #df_ON_seq6 = df_FCMold_ON.iloc[normal_groups_ON[21]]
        # Calculate mean, min, and max values
        mean_value = df_temp[str_MLsignal].mean()
        min_value = df_temp[str_MLsignal].min()
        max_value = df_temp[str_MLsignal].max()

        # Create the line plot
        fig11 = px.line(
            df_temp,
            x="plainTimeStamp",
            y=[str_MLsignal],
            title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                str_label, seq, start_time, end_time,
                seq_info['CASTING_SPEED_avg [m/min]'].values[0],
                seq_info['min AC Current, Up [A]'].values[0],
                seq_info['mean DC Current, Up [A]'].values[0],
                seq_info['mean DC Current, Low [A]'].values[0]
            ),
)
        fig11.update_layout(title_font_size=14,  # Set the title font size
            yaxis_title="Mold Level [mm]"  # Add y-axis label
        )
        # Add horizontal lines for mean, min, and max
        fig11.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
        fig11.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
        fig11.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

        # Save the plot as a PNG file
        # Sanitize time strings for filename
        safe_start_time = start_time.strftime('%Y-%m-%d_%H-%M-%S')
        safe_end_time = end_time.strftime('%Y-%m-%d_%H-%M-%S')
       
        #fig2.write_html("../reports/figures/data_grouping/window_size_5min/%s_%s_%s_%s_Vc=%.2f_AC_up=%d_DC_up=%d_DC_low=%d.html" % (
        #        str_label, seq, safe_start_time, safe_end_time,
        #        seq_info['CASTING_SPEED_avg [m/min]'].values[0],
        #        seq_info['min AC Current, Up [A]'].values[0],
        #        seq_info['mean DC Current, Up [A]'].values[0],
        #        seq_info['mean DC Current, Low [A]'].values[0]))#, width=1200, height=600, scale=2)
        # Show the plot
        #fig11.show()

In [0]:
#plot_MLsigma_per_sequence(df_seq_normal_ON_str23_6, 
#                            df_FCMold_ON_str23_6, 
#                            str_MLsignal = "Mold Level", 
#                            str_label = 'Strd23_6')

In [0]:
df_seq_normal_ON_str23_6.head()

In [0]:
df_seq_normal_ON_str23_6.shape[0]

In [0]:
for i in range(5): #df_seq_normal_ON_str23_6.shape[0]):
    print(df_seq_normal_ON_str23_6.iloc[i]['Seq_Name'])
    # Select the 
    sequence = i

    df_ON_seq23_6 = df_FCMold_ON_str23_6.iloc[normal_groups_ON_str23_6[sequence]]
    # Calculate mean, min, and max values
    mean_value = df_ON_seq23_6["Mold Level"].mean()
    min_value = df_ON_seq23_6["Mold Level"].min()
    max_value = df_ON_seq23_6["Mold Level"].max()

    #metadata
    str_label = 'strd23_6'
    seq = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_Name']
    start_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_Start']
    end_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_End']
    Vc_seq = df_seq_normal_ON_str23_6.iloc[sequence]['CASTING_SPEED_avg [m/min]']
    ACup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['min AC Current, Up [A]']
    DCup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Up [A]']
    DClow_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Low [A]']

    # Crete the line plot
    fig10 = px.line(
        df_ON_seq23_6,
        x="plainTimeStamp",
        y=["Mold Level"],
        title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                    str_label, seq, start_time, end_time,
                    Vc_seq,
                    ACup_seq,
                    DCup_seq,
                    DClow_seq
                ),
            )

    # Add horizontal lines for mean, min, and max
    fig10.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
    fig10.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
    fig10.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

    # Save the plot as a PNG file
    #fig2.write_html("../reports/figures/data_grouping/window_size_5min/strand1_seq21_mold_level.html")#, width=1200, height=600, scale=2)
    # Show the plot
    fig10.show()

In [0]:
df_seq_normal_ON_str23_6.columns

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_ON_str23_6

# --- Helper to build each scatter with Seq_Name in hover only ---
def make_scatter(df, x_col, x_label):
    fig_px = px.scatter(
        df,
        x=x_col,
        y="MOLD_LEVEL_std [mm]",
        hover_data=["Seq_Name"],     # show Seq_Name in hover
        labels={
            x_col: x_label,
            "MOLD_LEVEL_std [mm]": "Mold Level, [mm]"
        }
    )

    trace = fig_px.data[0]
    trace.text = None  # <-- remove text labels on points
    trace.hovertemplate = (
        "Seq_Name: %{customdata[0]}<br>"
        f"{x_label}: %{{x}}<br>"
        "Mold Level, [mm]: %{y}<extra></extra>"
    )

    return trace

# --- Create 2x2 subplot layout ---
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Vc vs Mold Level",
        "Mold Width vs Mold Level",
        "SEN vs Mold Level",
        "Argon Flow vs Mold Level"
    ]
)

# 1) Casting speed vs Mold Level
fig.add_trace(
    make_scatter(df, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]"),
    row=1, col=1
)

# 2) Mold width vs Mold Level
fig.add_trace(
    make_scatter(df, "MOLD_WIDTH_avg [m]", "Mold Width Avg [m]"),
    row=1, col=2
)

# 3) SEN vs Mold Level
fig.add_trace(
    make_scatter(df, "SEN_avg [mm]", "SEN Avg [mm]"),
    row=2, col=1
)

# 4) Argon Flow vs Mold Level
fig.add_trace(
    make_scatter(df, "ArFlow_avg [NL/min]", "Ar Flow Avg [NL/min]"),
    row=2, col=2
)

# --- Add horizontal threshold lines ---
for r in [1, 2]:
    for c in [1, 2]:
        fig.add_hline(
            y=2,
            line_dash="dash",
            line_color="green",
            annotation_text="ML fluctuation = 2 mm",
            annotation_position="top left",
            row=r, col=c
        )

# --- Axis titles ---
fig.update_xaxes(title_text="Casting Speed Avg [m/min]", row=1, col=1)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=1, col=1)

fig.update_xaxes(title_text="Mold Width Avg [m]",        row=1, col=2)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=1, col=2)

fig.update_xaxes(title_text="SEN Avg [mm]",              row=2, col=1)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=2, col=1)

fig.update_xaxes(title_text="Ar Flow Avg [NL/min]",      row=2, col=2)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=2, col=2)

# --- Layout ---
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Correlations vs Mold Level",
    showlegend=False
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_cc_parameters_and_ML_correlations.html"
)
fig.show()


In [0]:
sequence = 355

df_ON_seq23_6 = df_FCMold_ON_str23_6.iloc[normal_groups_ON_str23_6[sequence]]
    # Calculate mean, min, and max values
mean_value = df_ON_seq23_6["Mold Level"].mean()
min_value = df_ON_seq23_6["Mold Level"].min()
max_value = df_ON_seq23_6["Mold Level"].max()

    #metadata
str_label = 'strd23_6'
seq = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_Name']
start_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_Start']
end_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_End']
Vc_seq = df_seq_normal_ON_str23_6.iloc[sequence]['CASTING_SPEED_avg [m/min]']
ACup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['min AC Current, Up [A]']
DCup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Up [A]']
DClow_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Low [A]']

    # Crete the line plot
fig11 = px.line(
    df_ON_seq23_6,
    x="plainTimeStamp",
    y=["Mold Level"],
    title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                 str_label, seq, start_time, end_time,
                Vc_seq,
                ACup_seq,
                DCup_seq,
                DClow_seq
            ),
        )

    # Add horizontal lines for mean, min, and max
fig11.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
fig11.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
fig11.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

    # Save the plot as a PNG file
    #fig2.write_html("../reports/figures/data_grouping/window_size_5min/strand1_seq21_mold_level.html")#, width=1200, height=600, scale=2)
    # Show the plot
fig11.show()

In [0]:
df_seq_normal_ON_str23_6.columns

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Colormap where low values are visible (no white-on-white)
ml_cmap = LinearSegmentedColormap.from_list(
    'ml_sigma_cmap',
    [
        (0.0, '#440053'),   # dark purple  (very low σ)
        (0.3, '#404388'),
        (0.6, '#2a788e'),
        (0.8, '#21a784'),
        (0.95, '#78d151'),
        (1.0, '#fde624'),   # yellow       (very high σ)
    ],
    N=256
)

ml_std = df_seq_normal_ON_str23_6['MOLD_LEVEL_std [mm]'].to_numpy()
ml_std_max = np.nanmax(ml_std)

norm = TwoSlopeNorm(vmin=0, vcenter=2, vmax=ml_std_max)

x = df_seq_normal_ON_str23_6['MOLD_WIDTH_avg [m]'].to_numpy()
y = df_seq_normal_ON_str23_6['CASTING_SPEED_avg [m/min]'].to_numpy()

mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x = x[mask]
y = y[mask]
ml_std = ml_std[mask]

above_thr = ml_std > 2.0  # bad region

fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter: all points colored by σ, threshold-aware colormap
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge so they stand out
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=60,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.95,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('MOLD_LEVEL_std [mm]')

ax.set_xlabel('Mold Width avg [m]')
ax.set_ylabel('Casting Speed avg [m/min]')
ax.set_title('Mold Width vs Casting Speed colored by Mold Level σ')

# Optional: visual threshold line in colorbar label
# (the TwoSlopeNorm already makes 2 mm visually special)
plt.show()



In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Colormap where low values are visible (no white-on-white)
ml_cmap = LinearSegmentedColormap.from_list(
    'ml_sigma_cmap',
    [
        (0.0, '#440053'),   # dark purple  (very low σ)
        (0.3, '#404388'),
        (0.6, '#2a788e'),
        (0.8, '#21a784'),
        (0.95, '#78d151'),
        (1.0, '#fde624'),   # yellow       (very high σ)
    ],
    N=256
)

ml_std = df_seq_normal_ON_str23_6['MOLD_LEVEL_std [mm]'].to_numpy()
ml_std_max = np.nanmax(ml_std)

norm = TwoSlopeNorm(vmin=0, vcenter=2, vmax=ml_std_max)

x = df_seq_normal_ON_str23_6['SEN_avg [mm]'].to_numpy()
y = df_seq_normal_ON_str23_6['ArFlow_avg [NL/min]'].to_numpy()

mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x = x[mask]
y = y[mask]
ml_std = ml_std[mask]

above_thr = ml_std > 2.0  # bad region

fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter: all points colored by σ, threshold-aware colormap
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge so they stand out
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=60,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.95,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('MOLD_LEVEL_std [mm]')

ax.set_xlabel('SEN_avg [m]')
ax.set_ylabel('ArFlow_avg [NL/min]')
#ax.set_title('Mold Width vs Casting Speed colored by Mold Level σ')

# Optional: visual threshold line in colorbar label
# (the TwoSlopeNorm already makes 2 mm visually special)
plt.show()



In [0]:
# 3D scatter: AC/DC currents colored by mold level sigma
# Shows whether current settings correlate with stability
ml_colorscale = [
    [0.0,  "#440053"],  # dark purple – very low σ
    [0.3,  "#404388"],
    [0.6,  "#2a788e"],
    [0.8,  "#21a784"],
    [0.95, "#78d151"],
    [1.0,  "#fde624"],  # yellow – very high σ
]


import plotly.graph_objects as go
import numpy as np
import pandas as pd

df = df_seq_normal_ON_str23_6

ml_std = df["MOLD_LEVEL_std [mm]"]
vmin = 0.0
vmax = float(ml_std.max())
threshold = 2.0

# Custom colorscale similar to the Matplotlib one
ml_colorscale = [
    [0.0,  "#440053"],
    [0.3,  "#404388"],
    [0.6,  "#2a788e"],
    [0.8,  "#21a784"],
    [0.95, "#78d151"],
    [1.0,  "#fde624"],
]

# Split data into "normal" (σ ≤ 2 mm) and "abnormal" (σ > 2 mm)
df_normal   = df[ml_std <= threshold]
df_abnormal = df[ml_std >  threshold]

fig = go.Figure()

# ---- NORMAL trace (σ ≤ 2 mm) ----
fig.add_trace(
    go.Scatter3d(
        x=df_normal['max AC Current, Up [A]'],
        y=df_normal['max DC Current, Low [A]'],
        z=df_normal['max DC Current, Up [A]'],
        mode='markers',
        marker=dict(
            size=4,
            color=df_normal["MOLD_LEVEL_std [mm]"],
            colorscale=ml_colorscale,
            cmin=vmin,
            cmax=vmax,
            coloraxis="coloraxis",
            opacity=0.8,
        ),
        name="σ ≤ 2 mm",
        hovertemplate=(
            "AC Up: %{x:.2f} A<br>"
            "DC Low: %{y:.2f} A<br>"
            "DC Up: %{z:.2f} A<br>"
            "ML σ: %{marker.color:.2f} mm<extra>Normal</extra>"
        ),
    )
)

# ---- ABNORMAL trace (σ > 2 mm), highlighted with black edge ----
fig.add_trace(
    go.Scatter3d(
        x=df_abnormal['max AC Current, Up [A]'],
        y=df_abnormal['max DC Current, Low [A]'],
        z=df_abnormal['max DC Current, Up [A]'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_abnormal["MOLD_LEVEL_std [mm]"],
            colorscale=ml_colorscale,
            cmin=vmin,
            cmax=vmax,
            coloraxis="coloraxis",
            opacity=0.95,
            line=dict(color="black", width=1),
        ),
        name="σ > 2 mm",
        hovertemplate=(
            "AC Up: %{x:.2f} A<br>"
            "DC Low: %{y:.2f} A<br>"
            "DC Up: %{z:.2f} A<br>"
            "ML σ: %{marker.color:.2f} mm<extra>Abnormal</extra>"
        ),
    )
)

# ---- Shared coloraxis (single colorbar) ----
fig.update_layout(
    coloraxis=dict(
        colorscale=ml_colorscale,
        cmin=vmin,
        cmax=vmax,
        colorbar=dict(
            title="MOLD_LEVEL σ [mm]",
            thickness=15,
            len=0.8,
            x=1.02,
            xanchor="left",
            y=0.5,
            yanchor="middle",
        ),
    ),
    scene=dict(
        xaxis_title="AC Current, Up (A)",
        yaxis_title="DC Current, Low (A)",
        zaxis_title="DC Current, Up (A)",
        aspectmode="cube",
        xaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        yaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        zaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        camera=dict(eye=dict(x=1.2, y=1.2, z=1.2)),
    ),
    width=800,
    height=600,
    margin=dict(l=20, r=120, b=20, t=50),
    title="Strand #23_6 – FC Mold G5 Currents colored by Mold Level σ",
    title_x=0.5,
    autosize=False,
)

fig.show()


## 2.6 Disturbance Detection and Classification

For each stable sequence, detect and classify mold level disturbances:

| Type | Detection Rule |
| --- | --- |
| Excursion | Deviation > 8 mm from baseline for > 5 seconds |
| Slow drift | Monotonic run > 60 s with amplitude > 10 mm |
| Transient bump | Spike > 8× local noise sigma (min 6 mm) |
| High variability | Peak-to-peak > 10 mm, or >10% of signal outside ±4 mm band |

Sequences with none of the above are labelled **Normal**.

In [0]:
# ---------------------------------------------------------------------------
# Disturbance detection: main detectors imported from src/ in cell 33
#   - detect_excursion_event_robust, detect_slow_drift_event,
#     detect_transient_bump_dynamic, detect_high_variability_event
# Below: local helper wrappers used in exploration (simpler True/False interface)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd


def has_slow_drift(df_seq, value_col="Mold Level_clean", time_col="plainTimeStamp",
                   min_run_s=60.0, min_amp_mm=10.0):
    """Simple True/False wrapper for slow drift detection on a sequence slice."""
    if len(df_seq) < 3:
        return False
    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce").to_numpy()
    mask = np.isfinite(v)
    if mask.sum() < 3:
        return False
    t = t[mask]
    v = v[mask]

    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False

    dv = np.diff(v)
    sign = np.sign(dv)
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]

    runs, start = [], 0
    for i in range(1, len(sign)):
        if sign[i] != sign[start]:
            runs.append((start, i-1))
            start = i
    runs.append((start, len(sign)-1))

    for i0, i1 in runs:
        if sign[i0] == 0:
            continue
        duration_s = (i1 - i0 + 1) * dt
        amp = abs(v[i1+1] - v[i0])
        if duration_s >= min_run_s and amp >= min_amp_mm:
            return True
    return False


def has_excursion_event_robust(df_seq, value_col="Mold Level", time_col="plainTimeStamp",
                               excursion_mm=8.0, min_duration_s=5.0):
    """Simple True/False wrapper for excursion detection on a sequence slice."""
    if len(df_seq) < 3:
        return False
    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce")

    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False

    baseline = v.median()
    exc = (v - baseline).abs() > excursion_mm
    exc = exc.fillna(False).to_numpy()

    max_run, cur = 0, 0
    for b in exc:
        if b:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 0

    return (max_run * dt) >= min_duration_s


def simple_spike_clean(series, median_win=5, jump_mm=5.0):
    """Remove spike artifacts from mold level signal."""
    s = pd.to_numeric(series, errors="coerce")
    med = s.rolling(median_win, center=True).median()
    spike = (med.diff().abs() > jump_mm)
    s2 = s.copy()
    s2[spike] = np.nan
    return s2.interpolate(limit_direction="both")


def _unique_quality(seg, col):
    """Get unique quality casting value for a sequence segment."""
    if col not in seg.columns:
        return np.nan

    vals = (
        seg[col]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    if len(vals) == 0:
        return np.nan
    elif len(vals) == 1:
        return vals[0]
    else:
        return "MULTIPLE: " + " | ".join(sorted(vals))


print("Disturbance detection helpers loaded")
print("  From src/: detect_excursion_event_robust, detect_slow_drift_event,")
print("             detect_transient_bump_dynamic, detect_high_variability_event")
print("  Local:     has_slow_drift, has_excursion_event_robust, simple_spike_clean, _unique_quality")


In [0]:
# 3) Usage
window_size = 300
Vc_threshold = 0.1
Vc_column_str6 = "castingSpeed"
Curr_columns_str6 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_threshold = 50

#df_seq_str23_6, normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = (
#    identify_sequences_segmented_with_disturbance_flags(
#        df_FCMold_ON_str23_6,
#        Vc_column=Vc_column_str6,
#        window_size=window_size,
##        Vc_threshold=Vc_threshold,
#        Curr_columns=Curr_columns_str6,
#        Curr_threshold=Curr_threshold,
#        tcol="plainTimeStamp",
#        max_gap_seconds=5,
#        min_segment_len=window_size,
#        mold_col="Mold Level",
#        excursion_mm=8.0,
#        excursion_min_duration_s=5.0,
#        slow_min_run_s=60.0,
#        slow_min_amp_mm=10.0,
#    )
#)

#df_seq_str23_6.head()


In [0]:
df_FCMold_ON_str23_6.head(5)

In [0]:
# Execute sliding-window sequence identification on FC Mold active data
Vc_column_str6 = "castingSpeed"

normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = identify_sequences_segmented(
    df_FCMold_ON_str23_6,
    Vc_column=Vc_column_str6,
    window_size=window_size,
    Vc_threshold=Vc_threshold,
    Curr_columns=Curr_columns_str6,
    Curr_threshold=Curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=5,          # treat gaps >5s as new segments
    min_segment_len=window_size # ignore segments shorter than one window
)

len(normal_groups_ON_str23_6), len(abnormal_groups_ON_str23_6)

In [0]:
# For each normal sequence: compute stats AND run disturbance detectors
df_seq_normal_ON_str23_6 = generate_seq_statistics_with_disturbance(
    df=df_FCMold_ON_str23_6,
    sequences=normal_groups_ON_str23_6,
    seq_condition="ON",
    excursion_mm=8.0,
    excursion_min_duration_s=5.0,
    slow_min_run_s=60.0,
    slow_min_amp_mm=10.0
)


In [0]:
df_seq_normal_ON_str23_6.head(5)

In [0]:
# Map boolean flags -> human-readable disturbance type label
def classify_disturbance(row):
    flags = []
    if row["has_excursion_event"]:
        flags.append("Excursion")
    if row["has_high_variability"]:
        flags.append("High variability")
    if row.get("has_transient_bump", False):
        flags.append("Transient bump")
    if row.get("has_slow_drift", False):
        flags.append("Slow drift")

    return "Normal" if not flags else " + ".join(flags)

def assign_disturbance_type(row):
    """
    Assign a human-readable disturbance type based on flags.
    Priority is intentional: excursion > slow drift > bump > variability > normal
    """

    flags = []

    if row.get("has_excursion_event", False):
        flags.append("Excursion")

    if row.get("has_slow_drift", False):
        flags.append("Slow drift")

    if row.get("has_transient_bump", False):
        flags.append("Transient bump")

    if row.get("has_high_variability", False):
        flags.append("High variability")

    if not flags:
        return "Normal"

    return " + ".join(flags)


df_seq_normal_ON_str23_6 = df_seq_normal_ON_str23_6.copy()
df_seq_normal_ON_str23_6["disturbance_type"] = (
    df_seq_normal_ON_str23_6.apply(assign_disturbance_type, axis=1)
)
df_seq_normal_ON_str23_6["disturbance_type"].value_counts()


In [0]:
df_seq_normal_ON_str23_6.head(5)

In [0]:
# Ordered category for plotting + count how many sequences of each type
disturbance_order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + Transient bump + High variability",
]
disturbance_cols = [
    "has_disturbance",
    "has_excursion_event",
    "has_slow_drift",
    "has_transient_bump",
    "has_high_variability",
]


df_seq_normal_ON_str23_6["disturbance_type"] = pd.Categorical(
    df_seq_normal_ON_str23_6["disturbance_type"],
    categories=disturbance_order,
    ordered=True
)


df_seq_normal_ON_str23_6[disturbance_cols].sum().to_frame("count")


In [0]:
df_seq = df_seq_normal_ON_str23_6.copy()

# quick overlap table
pd.crosstab(df_seq["has_excursion_event"], df_seq["has_high_variability"], margins=True)


In [0]:
df_seq_normal_ON_str23_6["disturbance_type"].value_counts()


In [0]:
df_seq_normal_ON_str23_6["disturbance_type"].unique()


In [0]:
df_seq = df_seq_normal_ON_str23_6.copy()

def classify_disturbance(row):
    flags = []
    if row.get("has_excursion_event", False):   flags.append("Excursion")
    if row.get("has_slow_drift", False):        flags.append("Slow drift")
    if row.get("has_transient_bump", False):    flags.append("Transient bump")
    if row.get("has_high_variability", False):  flags.append("High variability")
    return "Normal" if not flags else " + ".join(flags)

df_seq["disturbance_type"] = df_seq.apply(classify_disturbance, axis=1)
df_seq["disturbance_type"].value_counts()


In [0]:
# Box plot showing relationship between casting speed and mold level variability
import plotly.express as px

fig = px.box(
    df_seq,
    x="CASTING_SPEED_avg [m/min]",
    y="MOLD_LEVEL_std [mm]",
    color="disturbance_type",
    points="all",
    hover_data=["Seq_Name", "disturbance_type"],
    title="Casting Speed vs Mold Level σ — by disturbance type",
    labels={
        "CASTING_SPEED_avg [m/min]": "Casting Speed Avg [m/min]",
        "MOLD_LEVEL_std [mm]": "Mold Level σ [mm]",
        "disturbance_type": "Disturbance type",
    },
)

# threshold
fig.add_hline(
    y=2.0,
    line_dash="dash",
    line_color="green",
    annotation_text="σ = 2 mm (stability threshold)",
    annotation_position="top right",
)
fig.update_layout(
    boxmode="overlay",
    height=700,
    width=1150,

    legend=dict(
        title="Disturbance type",
        x=0.02,                 # inside, left
        y=0.98,                 # inside, top
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="black",
        borderwidth=1,
        traceorder="reversed"   # reverse legend order
    )
)

fig.update_annotations(visible=False)
# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_casting_speed_and_MLsigma_correlations.html"
)
fig.show()


In [0]:
import pandas as pd
import plotly.express as px

#dfp = df_seq_normal_ON_str23_6.copy()
dfp = df_seq

# 1) Make sure disturbance_type exists (if already created, you can skip this block)
def assign_disturbance_type(row):
    flags = []
    if row.get("has_excursion_event", False):   flags.append("Excursion")
    if row.get("has_slow_drift", False):        flags.append("Slow drift")
    if row.get("has_transient_bump", False):    flags.append("Transient bump")
    if row.get("has_high_variability", False):  flags.append("High variability")
    return "Normal" if not flags else " + ".join(flags)

if "disturbance_type" not in dfp.columns:
    dfp["disturbance_type"] = dfp.apply(assign_disturbance_type, axis=1)

# 2) Round casting speed for grouping (same idea as your example)
dfp["CASTING_SPEED_avg_rounded"] = dfp["CASTING_SPEED_avg [m/min]"].round(1)

# Optional: enforce a clean legend order (adjust to what exists in your data)
disturbance_order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + High variability",
    "Excursion + Transient bump + High variability",
]
present = [c for c in disturbance_order if c in set(dfp["disturbance_type"])]
# add any unexpected categories at the end
present += [c for c in dfp["disturbance_type"].unique() if c not in present]

# 3) Boxplot with points overlay, colored by disturbance type
fig = px.box(
    dfp,
    x="CASTING_SPEED_avg_rounded",
    y="MOLD_LEVEL_std [mm]",
    color="disturbance_type",
    category_orders={"disturbance_type": present},
    points="all",
    hover_data=["Seq_Name", "disturbance_type"],
    labels={
        "CASTING_SPEED_avg_rounded": "Casting Speed Avg [m/min] (rounded)",
        "MOLD_LEVEL_std [mm]": "Mold Level σ [mm]",
        "disturbance_type": "Disturbance type",
    },
    title="Strand#23-6 — Casting Speed vs Mold Level σ (by disturbance type)",
)

# 4) Threshold line (σ = 2 mm)
fig.add_hline(
    y=2.0,
    line_dash="dash",
    line_color="green",
    annotation_text="Stability threshold (σ = 2 mm)",
    annotation_position="top right",
)

# 5) Layout + legend inside the plot
fig.update_layout(
    boxmode="overlay",
    height=700,
    width=1150,
    legend=dict(
        title="Disturbance type",
        x=0.02, y=0.98,
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="black",
        borderwidth=1,
        traceorder="reversed",
    ),
)

# If you previously added annotations elsewhere, hide them
fig.update_annotations(visible=False)

fig.show()


In [0]:
# Keep only sequences with NO disturbance flags (the "clean" dataset)
df_seq_normal_process = df_seq_normal_ON_str23_6[~df_seq_normal_ON_str23_6["has_disturbance"]]


In [0]:
df_seq_normal_process.head(5)

In [0]:
# Save the per-sequence summary for use in the Technical Report notebook
df_seq_normal_process.to_csv("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_seq_normal_summary.csv", index=False)


In [0]:
import numpy as np
import pandas as pd

df_seq = df_seq_normal_ON_str23_6.copy()

def classify_disturbance(row):
    flags = []
    if bool(row.get("has_excursion_event", False)):   flags.append("Excursion")
    if bool(row.get("has_high_variability", False)):  flags.append("High variability")
    if bool(row.get("has_transient_bump", False)):    flags.append("Transient bump")
    if bool(row.get("has_slow_drift", False)):        flags.append("Slow drift")
    return "Normal" if not flags else " + ".join(flags)

df_seq["disturbance_type"] = df_seq.apply(classify_disturbance, axis=1)

# pick one example index per type
# option A (simple): first occurrence
seq_idx_by_type = df_seq.groupby("disturbance_type").head(1).reset_index()["index"].to_dict()

# Better option B: pick the "strongest" example per type (more illustrative)
# Uncomment this if you want strongest by ptp_mm
# seq_idx_by_type = (
#     df_seq.sort_values(["disturbance_type", "ptp_mm"], ascending=[True, False])
#           .groupby("disturbance_type")
#           .head(1)
#           .reset_index()
#           .set_index("disturbance_type")["index"]
#           .to_dict()
# )

seq_idx_by_type


In [0]:
import plotly.graph_objects as go
import plotly.subplots as sp

# keys must be disturbance type NAMES (strings)
# e.g.:
seq_idx_by_type = {
     "Normal": 127,
     "High variability": 950,
     "Excursion": 956,
     "Excursion + Transient bump + High variability": 332
 }
disturbance_label_map = {
    0: "Normal",
    1: "High variability",
    2: "Excursion",
    3: "Excursion + Transient bump + High variability",
}

types = list(seq_idx_by_type.keys())


nrows = len(types)

fig = sp.make_subplots(
    rows=nrows,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=[
    f"Disturbance type: {disturbance_label_map.get(t, str(t))}  |  Sequence index: {seq_idx_by_type[t]}"
    for t in types
  ],

)

for r, dtype in enumerate(types, start=1):
    seq_idx = seq_idx_by_type[dtype]

    # raw sequence data
    df_seq_raw = df_FCMold_ON_str23_6.iloc[
        normal_groups_ON_str23_6[seq_idx]
    ].copy()

    mean_value = df_seq_raw["Mold Level"].mean()
    min_value  = df_seq_raw["Mold Level"].min()
    max_value  = df_seq_raw["Mold Level"].max()

    # Mold Level signal
    fig.add_trace(
        go.Scatter(
            x=df_seq_raw["plainTimeStamp"],
            y=df_seq_raw["Mold Level"],
            mode="lines",
            name=dtype,
            hovertemplate=(
                "Time=%{x}<br>"
                "Mold Level=%{y:.2f} mm<br>"
                f"Type={dtype}"
                "<extra></extra>"
            ),
        ),
        row=r,
        col=1,
    )

    # Mean / Min / Max reference lines
    fig.add_hline(
        y=mean_value,
        line_dash="dash",
        line_color="green",
        row=r,
        col=1,
    )
    fig.add_hline(
        y=min_value,
        line_dash="dot",
        line_color="blue",
        row=r,
        col=1,
    )
    fig.add_hline(
        y=max_value,
        line_dash="dot",
        line_color="red",
        row=r,
        col=1,
    )

    fig.update_yaxes(title_text="Mold Level [mm]", row=r, col=1)

fig.update_xaxes(title_text="Time", row=nrows, col=1)

fig.update_layout(
    height=320 * nrows,
    width=1200,
    title="Representative sequences per disturbance type",
    showlegend=False,
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_seq_groupings_by_disturbance_type.html")

fig.show()


In [0]:
# Width vs Speed scatter with dual encoding: color=sigma, marker=disturbance type
import numpy as np
import matplotlib.pyplot as plt

dfp = df_seq_normal_ON_str23_6.copy()

# --- data vectors ---
x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()

# categorical color = disturbance type (must exist)
dtype = dfp["disturbance_type"].astype(str).to_numpy()

# filter finite
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x, y, ml_std, dtype = x[mask], y[mask], ml_std[mask], dtype[mask]

above_thr = ml_std > 2.0

# --- choose a stable order for legend (optional) ---
order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + High variability",
    "Excursion + Transient bump + High variability",
]
present = [c for c in order if c in set(dtype)]
# add any unexpected categories at the end
present += [c for c in np.unique(dtype) if c not in present]

# --- map categories -> colors using matplotlib default cycle ---
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(present)}
point_colors = np.array([color_map[c] for c in dtype])

# --- plot ---
fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter (all points)
ax.scatter(
    x, y,
    c=point_colors,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge (same colors, just outlined)
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=point_colors[above_thr],
    s=70,
    edgecolor="black",
    linewidth=0.9,
    alpha=0.95,
)

# --- legend (disturbance types) ---
handles = [
    plt.Line2D([0], [0], marker="o", color="none",
               markerfacecolor=color_map[cat],
               markersize=8, label=cat)
    for cat in present
]
ax.legend(
    handles=handles,
    title="Disturbance type",
    loc="best",
    frameon=True,
)

ax.set_xlabel("Mold Width avg [m]")
ax.set_ylabel("Casting Speed avg [m/min]")
ax.set_title("Mold Width vs Casting Speed colored by disturbance type\n(black outline: MOLD_LEVEL_std > 2 mm)")

# Save & show
fig.savefig(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_Vc_MoldWidth_sequences_colored.png")
plt.show()


In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# --- filter to NORMAL sequences only ---
dfp = df_seq_normal_ON_str23_6[
    df_seq_normal_ON_str23_6["disturbance_type"] == "Normal"
].copy()

# --- custom colormap (same as your reference) ---
ml_cmap = LinearSegmentedColormap.from_list(
    "ml_sigma_cmap",
    [
        (0.0, "#440053"),   # dark purple (very low σ)
        (0.3, "#404388"),
        (0.6, "#2a788e"),
        (0.8, "#21a784"),
        (0.95, "#78d151"),
        (1.0, "#fde624"),   # yellow (very high σ)
    ],
    N=256,
)

# --- data vectors ---
x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()

# keep finite only
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x, y, ml_std = x[mask], y[mask], ml_std[mask]

# threshold
above_thr = ml_std > 2.0
ml_std_max = np.nanmax(ml_std)

# threshold-aware normalization
norm = TwoSlopeNorm(vmin=0, vcenter=2.0, vmax=ml_std_max)

# --- plot ---
fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) base scatter (Normal only, colored by σ)
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.85,
)

# 2) highlight σ > 2 mm with black outline
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=70,
    edgecolor="black",
    linewidth=0.9,
    alpha=0.95,
)

# colorbar
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("MOLD_LEVEL_std [mm]")
cbar.ax.axhline(2.0, color="black", linewidth=1)
cbar.ax.text(
    1.05, 2.0, "σ = 2 mm",
    transform=cbar.ax.get_yaxis_transform(),
    va="center"
)

# labels & title
ax.set_xlabel("Mold Width avg [m]")
ax.set_ylabel("Casting Speed avg [m/min]")
ax.set_title("Normal sequences only\nColored by Mold Level σ (black outline: σ > 2 mm)")

plt.show()


In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helpers
# -------------------------
def make_box_points_and_meanline(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace, mean_line_trace)
    """
    x_round_name = f"{x_col}__rounded"
    df = df.copy()
    df[x_round_name] = df[x_col].round(round_to)

    # -------------------------
    # Box trace
    # -------------------------
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        boxmean=False,
        marker=dict(color=color),
        line=dict(color=color),
        opacity=0.45,
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        )
    )

    # -------------------------
    # Scatter trace
    # -------------------------
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.8),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False
    )

    # -------------------------
    # Mean line (per bin)
    # -------------------------
    mean_df = (
        df.groupby(x_round_name, as_index=False)["MOLD_LEVEL_std [mm]"]
        .mean()
        .sort_values(x_round_name)
    )

    mean_line = go.Scatter(
        x=mean_df[x_round_name],
        y=mean_df["MOLD_LEVEL_std [mm]"],
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=6),
        name="Mean trend",
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mean Mold Level σ: %{y:.3f} mm<extra></extra>"
        )
    )

    return box, scatter, mean_line


def make_box_and_points(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace) for a given x_col vs Mold Level std.
    Boxes are grouped by rounded x to create meaningful distributions.
    """
    x_round_name = f"{x_col}__rounded"
    df[x_round_name] = df[x_col].round(round_to)

    # Box trace (distribution per rounded bin)
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        name=None,
        boxmean="sd",
        marker=dict(color=color, opacity=0.85),
        line=dict(color=color, width=1),
        opacity=0.55,               # box opacity
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        )
    )

    # Scatter trace (points on top)
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.85),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False,
        name=None
    )

    return box, scatter

# -------------------------
# Subplot layout (2x2)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Vc vs Mold Level σ (Normal only)",
        "Mold Width vs Mold Level σ (Normal only)",
        "SEN vs Mold Level σ (Normal only)",
        "Argon Flow vs Mold Level σ (Normal only)",
    ]
)

panels = [
    (1, 1, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]"),
    (1, 2, "MOLD_WIDTH_avg [m]",        "Mold Width Avg [m]"),
    (2, 1, "SEN_avg [mm]",              "SEN Avg [mm]"),
    (2, 2, "ArFlow_avg [NL/min]",       "Ar Flow Avg [NL/min]"),
]

for r, c, xcol, xlabel in panels:
    box_tr, sc_tr, mean_tr = make_box_points_and_meanline(
    df, xcol, xlabel, round_to=1, color="#bdb76b"
    )

    fig.add_trace(box_tr, row=r, col=c)
    fig.add_trace(sc_tr, row=r, col=c)
    fig.add_trace(mean_tr, row=r, col=c)


    # axis titles
    fig.update_xaxes(title_text=f"{xlabel} (rounded)", row=r, col=c)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=r, col=c)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Normal process only: distributions (box + points)",
    boxmode="overlay",
    showlegend=False,
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_cc_parameters_and_ML_correlations_NORMAL_only_boxpoints.html"
)
fig.show()


In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

YCOL = "MOLD_LEVEL_std [mm]"

# -------------------------
# Helpers
# -------------------------
def add_box_points_mean(fig, df, x_col, x_label, row, col, bin_size, color="#bdb76b"):
    """
    Box + points + mean line using coarse bins.
    bin_size is in the units of x_col (e.g., 0.1 m/min, 0.05 m).
    """
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    # Bin x into coarse categories (works better than rounding for continuous-ish vars)
    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # 1) Boxplot
    fig.add_trace(
        go.Box(
            x=d[x_bin],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.45,
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                f"Mold Level σ [mm]: %{{y:.3f}}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    # 2) Points
    fig.add_trace(
        go.Scatter(
            x=d[x_bin],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label} (bin): %{{x}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # 3) Mean trend per bin
    mean_df = (
        d.groupby(x_bin, as_index=False)[YCOL]
        .mean()
        .sort_values(x_bin)
    )

    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color="white", width=2),
            marker=dict(size=6, color="white"),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label} (binned, Δ={bin_size})", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_scatter_mean(fig, df, x_col, x_label, row, col, bin_size, point_color="#bdb76b", line_color="white"):
    """
    Scatter + mean trend line only (no box). Uses bins to stabilize the mean line.
    """
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # 1) Scatter (raw x, not binned) for truthful geometry
    fig.add_trace(
        go.Scatter(
            x=d[x_col],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=point_color, size=6, opacity=0.55),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.4g}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # 2) Mean trend (computed on binned x, plotted at binned x)
    mean_df = (
        d.groupby(x_bin, as_index=False)[YCOL]
        .mean()
        .sort_values(x_bin)
    )

    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label}", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


# -------------------------
# Subplots (2×2)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Casting speed: box + points + mean (best)",
        "Mold width: box + points + mean (coarse bins)",
        "SEN: scatter + mean trend (no box)",
        "Argon flow: scatter + mean trend (no box)",
    ]
)

# 1) Casting speed: discrete setpoints → box works well (bin ~ 0.1 m/min)
add_box_points_mean(
    fig, df,
    x_col="CASTING_SPEED_avg [m/min]",
    x_label="Casting Speed Avg [m/min]",
    row=1, col=1,
    bin_size=0.1
)

# 2) Mold width: coarse bins needed (e.g. 0.05 m)
add_box_points_mean(
    fig, df,
    x_col="MOLD_WIDTH_avg [m]",
    x_label="Mold Width Avg [m]",
    row=1, col=2,
    bin_size=0.05
)

# 3) SEN: continuous → no box, mean trend stabilised with small bins (e.g. 0.001 m ~ 1 mm)
# Your SEN column label says [mm] but values look like meters (0.16–0.22).
# If it's meters, 0.001 = 1 mm. If it's truly mm, change bin_size accordingly.
add_scatter_mean(
    fig, df,
    x_col="SEN_avg [mm]",
    x_label="SEN Avg",
    row=2, col=1,
    bin_size=0.001
)

# 4) Argon flow: continuous → no box, mean trend stabilised with bins (e.g. 0.5 NL/min)
add_scatter_mean(
    fig, df,
    x_col="ArFlow_avg [NL/min]",
    x_label="Ar Flow Avg [NL/min]",
    row=2, col=2,
    bin_size=0.5
)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Normal process only: recommended summaries",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_6_recommended_correlations_NORMAL_only.html"
)
fig.show()


In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()   # aggregated df (per-sequence)
YCOL = "MOLD_LEVEL_std [mm]"
QCOL = "Quality casting"

# -------------------------
# Helpers
# -------------------------
def add_box_points_mean(fig, df, x_col, x_label, row, col, bin_size, color="#bdb76b"):
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # Box
    fig.add_trace(
        go.Box(
            x=d[x_bin],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.45,
            showlegend=False,
        ),
        row=row, col=col
    )

    # Points
    fig.add_trace(
        go.Scatter(
            x=d[x_bin],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label} (bin): %{{x}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # Mean line per bin
    mean_df = d.groupby(x_bin, as_index=False)[YCOL].mean().sort_values(x_bin)
    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color="black", width=2),
            marker=dict(size=6, color="black"),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label} (binned, Δ={bin_size})", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_scatter_mean(fig, df, x_col, x_label, row, col, bin_size, point_color="#bdb76b", line_color="black"):
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # Scatter (raw x)
    fig.add_trace(
        go.Scatter(
            x=d[x_col],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=point_color, size=6, opacity=0.55),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.4g}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # Mean trend (computed on binned x)
    mean_df = d.groupby(x_bin, as_index=False)[YCOL].mean().sort_values(x_bin)
    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=x_label, row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_quality_box(fig, df, row, col, color="#bdb76b"):
    d = df[[QCOL, YCOL, "Seq_Name"]].dropna().copy()

    # clean category strings
    d[QCOL] = d[QCOL].astype(str).str.strip()

    # Box
    fig.add_trace(
        go.Box(
            x=d[QCOL],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.50,
            showlegend=False,
        ),
        row=row, col=col
    )

    # Points
    fig.add_trace(
        go.Scatter(
            x=d[QCOL],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                "Quality casting: %{x}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text="Quality casting", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)

# -------------------------
# Layout: 2 rows × 3 cols (5 panels used)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Casting speed",
        "Mold width",
        "Quality casting",
        "SEN",
        "Argon flow",
        ""  # empty cell (row 2 col 3)
    ]
)

# Row 1
add_box_points_mean(fig, df, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]", row=1, col=1, bin_size=0.1)
add_box_points_mean(fig, df, "MOLD_WIDTH_avg [m]",        "Mold Width Avg [m]",        row=1, col=2, bin_size=0.05)
#add_quality_box(fig, df, row=1, col=3)

# Row 2
#add_scatter_mean(fig, df, "SEN_avg [mm]",          "SEN Avg [m]",            row=2, col=1, bin_size=0.001)  # 1 mm bins = 0.001 m
add_quality_box(fig, df, row=2, col=1)
add_scatter_mean(fig, df, "ArFlow_avg [NL/min]",   "Ar Flow Avg [NL/min]",   row=2, col=2, bin_size=0.5)
quality_order = (
    df.groupby("Quality casting")["MOLD_LEVEL_std [mm]"]
      .median()
      .sort_values()
      .index
      .tolist()
)

fig.update_xaxes(
    categoryorder="array",
    categoryarray=quality_order,
    row=1, col=3
)


# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1600,
    title="Strand #23-5 – Normal process only",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_5_recommended_correlations_NORMAL_only_plus_quality.html"
)

fig.show()


In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helpers
# -------------------------
def make_box_and_points(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace) for a given x_col vs Mold Level std.
    Boxes are grouped by rounded x to create meaningful distributions.
    Points are plotted at the same rounded x positions for a clean overlay.
    """
    x_round_name = f"{x_col}__rounded"
    df[x_round_name] = df[x_col].round(round_to)

    # Box trace (distribution per rounded bin)
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        boxmean="sd",
        marker=dict(color=color, opacity=0.85),
        line=dict(color=color, width=1),
        opacity=0.55,
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
    )

    # Scatter trace (points on top)
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.85),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False,
    )

    return box, scatter, x_round_name

# -------------------------
# Subplot layout (1x3)
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=[
        "mean DC Current, Low vs Mold Level σ (Normal only)",
        "mean DC Current, Up vs Mold Level σ (Normal only)",
        "mean AC Current, Up vs Mold Level σ (Normal only)",
    ],
)

panels = [
    (1, 1, "mean DC Current, Low [A]", "Mean DC Current, Low [A]", 0),  # round_to=0A
    (1, 2, "mean DC Current, Up [A]",  "Mean DC Current, Up [A]",  0),
    (1, 3, "mean AC Current, Up [A]",  "Mean AC Current, Up [A]",  0),
]

for r, c, xcol, xlabel, round_to in panels:
    box_tr, sc_tr, x_round_name = make_box_and_points(df, xcol, xlabel, round_to=round_to, color="#bdb76b")
    fig.add_trace(box_tr, row=r, col=c)
    fig.add_trace(sc_tr, row=r, col=c)

    fig.update_xaxes(title_text=f"{xlabel} (rounded)", row=r, col=c)

fig.update_yaxes(title_text="Mold Level σ [mm]", row=1, col=1)

fig.update_layout(
    height=520,
    width=1400,
    title="Strand #23-6 – Normal only: Mold Level σ vs mean currents (box + points)",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_currents_vs_ML_correlations_NORMAL_only_boxpoints.html"
)

fig.show()


In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helper: pure scatter
# -------------------------
def make_scatter(df, x_col, x_label, color="#bdb76b"):
    return go.Scatter(
        x=df[x_col],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(
            size=7,
            color=color,
            opacity=0.75,
        ),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label}: %{{x:.2f}}<br>"
            "Mold Level σ: %{y:.3f} mm"
            "<extra></extra>"
        ),
        showlegend=False,
    )

# -------------------------
# Subplots (1 × 3)
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=[
        "mean DC Current, Low vs Mold Level σ (Normal only)",
        "mean DC Current, Up vs Mold Level σ (Normal only)",
        "mean AC Current, Up vs Mold Level σ (Normal only)",
    ],
)

panels = [
    (1, 1, "mean DC Current, Low [A]", "Mean DC Current, Low [A]"),
    (1, 2, "mean DC Current, Up [A]",  "Mean DC Current, Up [A]"),
    (1, 3, "mean AC Current, Up [A]",  "Mean AC Current, Up [A]"),
]

for r, c, xcol, xlabel in panels:
    fig.add_trace(
        make_scatter(df, xcol, xlabel),
        row=r, col=c
    )
    fig.update_xaxes(title_text=xlabel, row=r, col=c)

fig.update_yaxes(title_text="Mold Level σ [mm]", row=1, col=1)

fig.update_layout(
    height=480,
    width=1400,
    title="Strand #23-6 – Normal process only: Mold Level σ vs mean currents",
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_6_currents_vs_ML_scatter_NORMAL_only.html"
)

fig.show()


In [0]:
# Utility to plot the raw Mold Level time series for any individual sequence
import plotly.express as px

def plot_one_sequence(
    df_raw,
    df_seq,
    seq_idx,
    sequences,
    *,
    tcol="plainTimeStamp",
    value_col="Mold Level",
    title_prefix="",
):
    """
    Plot raw Mold Level time series for ONE sequence.

    Parameters
    ----------
    df_raw : pd.DataFrame
        Raw time-series dataframe (e.g. df_FCMold_ON_str23_6)
    df_seq : pd.DataFrame
        Aggregated per-sequence dataframe
        (output of generate_seq_statistics_with_disturbance)
    seq_idx : int
        Index of the sequence in df_seq / sequences
    sequences : list[list[int]]
        List of index groups returned by identify_sequences_segmented
    """

    # --- safety checks ---
    if seq_idx >= len(sequences):
        raise IndexError("seq_idx out of range")

    idxs = sequences[seq_idx]
    seg = df_raw.iloc[idxs].copy()

    # --- metadata ---
    meta = df_seq.iloc[seq_idx]

    title = (
        f"{title_prefix}"
        f"{meta['Seq_Name']} | "
        f"{meta['Seq_time_Start']} → {meta['Seq_time_End']}<br>"
        f"Vc={meta['CASTING_SPEED_avg [m/min]']:.2f}, "
        f"ML σ={meta['MOLD_LEVEL_std [mm]']:.2f} mm, "
        f"Disturbance={meta['has_disturbance']}"
    )

    # --- plot ---
    fig = px.line(
        seg,
        x=tcol,
        y=value_col,
        title=title,
        labels={
            tcol: "Time",
            value_col: "Mold Level [mm]",
        },
    )

    # reference lines
    fig.add_hline(
        y=seg[value_col].mean(),
        line_dash="dash",
        line_color="green",
        annotation_text="Mean",
        annotation_position="top left",
    )

    fig.add_hline(
        y=seg[value_col].min(),
        line_dash="dot",
        line_color="blue",
        annotation_text="Min",
        annotation_position="bottom left",
    )

    fig.add_hline(
        y=seg[value_col].max(),
        line_dash="dot",
        line_color="red",
        annotation_text="Max",
        annotation_position="top right",
    )

    fig.update_layout(
        height=400,
        width=1100,
        showlegend=False,
    )

    fig.show()


In [0]:
sequence = 127
plot_one_sequence(df_FCMold_ON_str23_6, df_seq, sequence, normal_groups_ON_str23_6)